In [4]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)
from sklearn.preprocessing import StandardScaler

from xgboost import XGBRegressor

from lightgbm import (
    LGBMRegressor,
    early_stopping,
)

In [ ]:
project_path = Path.cwd()

print("현재 작업 경로:", project_path)

data_files = sorted([
    path
    for path in project_path.rglob("*")
    if path.is_file()
    and path.suffix.lower() in {
        ".csv",
        ".parquet",
        ".pkl",
        ".xlsx",
        ".zip",
    }
])

print("데이터 파일 수:", len(data_files))

for path in data_files:
    relative_path = path.relative_to(project_path)
    file_size_mb = path.stat().st_size / (1024 ** 2)

    print(
        f"{relative_path} "
        f"| {file_size_mb:.2f} MB"
    )

현재 작업 경로: /Users/chankyulee/Desktop/DACON_BARAM3
데이터 파일 수: 67
.venv/lib/python3.11/site-packages/joblib/test/data/joblib_0.10.0_pickle_py27_np17.pkl | 0.00 MB
.venv/lib/python3.11/site-packages/joblib/test/data/joblib_0.10.0_pickle_py33_np18.pkl | 0.00 MB
.venv/lib/python3.11/site-packages/joblib/test/data/joblib_0.10.0_pickle_py34_np19.pkl | 0.00 MB
.venv/lib/python3.11/site-packages/joblib/test/data/joblib_0.10.0_pickle_py35_np19.pkl | 0.00 MB
.venv/lib/python3.11/site-packages/joblib/test/data/joblib_0.11.0_pickle_py36_np111.pkl | 0.00 MB
.venv/lib/python3.11/site-packages/joblib/test/data/joblib_0.9.2_pickle_py27_np16.pkl | 0.00 MB
.venv/lib/python3.11/site-packages/joblib/test/data/joblib_0.9.2_pickle_py27_np17.pkl | 0.00 MB
.venv/lib/python3.11/site-packages/joblib/test/data/joblib_0.9.2_pickle_py33_np18.pkl | 0.00 MB
.venv/lib/python3.11/site-packages/joblib/test/data/joblib_0.9.2_pickle_py34_np19.pkl | 0.00 MB
.venv/lib/python3.11/site-packages/joblib/test/data/joblib_0.9.2_pic

In [2]:
zip_files = list(project_path.rglob("*.zip"))

print("ZIP 파일 수:", len(zip_files))

for path in zip_files:
    print(path.relative_to(project_path))

ZIP 파일 수: 0


In [5]:
# 1. 원본 파일 불러오기
ldaps_train_raw = pd.read_csv("train/ldaps_train.csv")
gfs_train_raw = pd.read_csv("train/gfs_train.csv")
labels_raw = pd.read_csv("train/train_labels.csv")

scada_unison_raw = pd.read_csv(
    "train/scada_unison_train.csv"
)

scada_vestas_raw = pd.read_csv(
    "train/scada_vestas_train.csv"
)

ldaps_test_raw = pd.read_csv("test/ldaps_test.csv")
gfs_test_raw = pd.read_csv("test/gfs_test.csv")

sample_submission = pd.read_csv(
    "sample_submission.csv"
)


# 2. 기본 구조 확인 함수
def inspect_dataframe(name, df):
    print(f"\n{name}")
    print("shape:", df.shape)
    print("columns:")
    print(df.columns.tolist())

    print("\n앞부분")
    print(df.head(3))

    print("\n자료형")
    print(df.dtypes)

    print("\n결측치 상위")
    missing = (
        df.isna()
        .sum()
        .sort_values(ascending=False)
    )

    print(missing[missing > 0].head(20))

    print("\n전체 중복 행:", df.duplicated().sum())


# 3. 각 원본 파일 확인
inspect_dataframe(
    "LDAPS Train",
    ldaps_train_raw,
)

inspect_dataframe(
    "GFS Train",
    gfs_train_raw,
)

inspect_dataframe(
    "Train Labels",
    labels_raw,
)

inspect_dataframe(
    "SCADA Unison",
    scada_unison_raw,
)

inspect_dataframe(
    "SCADA Vestas",
    scada_vestas_raw,
)

inspect_dataframe(
    "LDAPS Test",
    ldaps_test_raw,
)

inspect_dataframe(
    "GFS Test",
    gfs_test_raw,
)

inspect_dataframe(
    "Sample Submission",
    sample_submission,
)


# 4. 시간으로 추정되는 컬럼 찾기
raw_datasets = {
    "ldaps_train": ldaps_train_raw,
    "gfs_train": gfs_train_raw,
    "labels": labels_raw,
    "scada_unison": scada_unison_raw,
    "scada_vestas": scada_vestas_raw,
    "ldaps_test": ldaps_test_raw,
    "gfs_test": gfs_test_raw,
    "sample_submission": sample_submission,
}

time_keywords = [
    "time",
    "date",
    "dtm",
    "datetime",
    "fcst",
    "forecast",
]

print("\n시간 관련 컬럼 후보")

for name, df in raw_datasets.items():
    time_candidates = [
        col
        for col in df.columns
        if any(
            keyword in col.lower()
            for keyword in time_keywords
        )
    ]

    print(
        name,
        ":",
        time_candidates,
    )


# 5. Excel 시트 구조 확인
excel_file = pd.ExcelFile("info.xlsx")

print("\ninfo.xlsx 시트 목록")
print(excel_file.sheet_names)

for sheet_name in excel_file.sheet_names:
    sheet_df = pd.read_excel(
        "info.xlsx",
        sheet_name=sheet_name,
    )

    print(
        "\n시트:",
        sheet_name,
        "| shape:",
        sheet_df.shape,
    )

    print(sheet_df.head(5))


LDAPS Train
shape: (420864, 35)
columns:
['forecast_kst_dtm', 'data_available_kst_dtm', 'grid_id', 'latitude', 'longitude', 'heightAboveGround_10_10u', 'heightAboveGround_10_10v', 'heightAboveGround_50_50MUmax', 'heightAboveGround_50_50MUmin', 'heightAboveGround_50_50MVmax', 'heightAboveGround_50_50MVmin', 'heightAboveGround_5_XBLWS', 'heightAboveGround_5_YBLWS', 'heightAboveGround_2_t', 'heightAboveGround_2_dpt', 'heightAboveGround_2_r', 'heightAboveGround_2_q', 'surface_0_sp', 'meanSea_0_prmsl', 'etc_0_blh', 'surface_0_NDNSW', 'surface_0_NDNLW', 'heightAboveGround_2_SWDIR', 'heightAboveGround_2_SWDIF', 'etc_0_hcc', 'etc_0_mcc', 'etc_0_lcc', 'etc_0_VLCDC', 'surface_0_avg_lsprate', 'surface_0_lssrate', 'surface_0_ncpcp', 'surface_0_snol', 'surface_0_SNOM', 'surface_0_lsm', 'surface_0_h']

앞부분
      forecast_kst_dtm data_available_kst_dtm  grid_id  latitude  longitude  \
0  2022-01-01 01:00:00    2021-12-31 13:00:00        1   37.3032   128.9443   
1  2022-01-01 01:00:00    2021-12-31 

In [6]:
# 1. 시간 컬럼 변환
for df in [
    ldaps_train_raw,
    ldaps_test_raw,
    gfs_train_raw,
    gfs_test_raw,
]:
    df["forecast_kst_dtm"] = pd.to_datetime(
        df["forecast_kst_dtm"]
    )

    df["data_available_kst_dtm"] = pd.to_datetime(
        df["data_available_kst_dtm"]
    )


# 2. 예보 리드타임 생성
for df in [
    ldaps_train_raw,
    ldaps_test_raw,
    gfs_train_raw,
    gfs_test_raw,
]:
    df["lead_time_hour"] = (
        df["forecast_kst_dtm"]
        - df["data_available_kst_dtm"]
    ).dt.total_seconds() / 3600


# 3. 시간별 기상 데이터 집계 함수
def aggregate_weather(
    df,
    prefix,
):
    exclude_cols = {
        "forecast_kst_dtm",
        "data_available_kst_dtm",
        "grid_id",
        "latitude",
        "longitude",
    }

    feature_cols = [
        col
        for col in df.columns
        if col not in exclude_cols
    ]

    aggregated = (
        df
        .groupby("forecast_kst_dtm")[feature_cols]
        .agg([
            "mean",
            "std",
            "min",
            "max",
        ])
    )

    aggregated.columns = [
        f"{prefix}_{col}_{stat}"
        for col, stat in aggregated.columns
    ]

    aggregated = (
        aggregated
        .reset_index()
        .sort_values("forecast_kst_dtm")
        .reset_index(drop=True)
    )

    return aggregated


# 4. LDAPS 집계
ldaps_train_hourly = aggregate_weather(
    ldaps_train_raw,
    prefix="ldaps",
)

ldaps_test_hourly = aggregate_weather(
    ldaps_test_raw,
    prefix="ldaps",
)


# 5. GFS 집계
gfs_train_hourly = aggregate_weather(
    gfs_train_raw,
    prefix="gfs",
)

gfs_test_hourly = aggregate_weather(
    gfs_test_raw,
    prefix="gfs",
)


# 6. 집계 결과 확인
print("LDAPS Train:", ldaps_train_hourly.shape)
print("LDAPS Test:", ldaps_test_hourly.shape)
print("GFS Train:", gfs_train_hourly.shape)
print("GFS Test:", gfs_test_hourly.shape)

print("\n시간 범위")
print(
    "LDAPS Train:",
    ldaps_train_hourly["forecast_kst_dtm"].min(),
    "~",
    ldaps_train_hourly["forecast_kst_dtm"].max(),
)

print(
    "LDAPS Test:",
    ldaps_test_hourly["forecast_kst_dtm"].min(),
    "~",
    ldaps_test_hourly["forecast_kst_dtm"].max(),
)

print(
    "GFS Train:",
    gfs_train_hourly["forecast_kst_dtm"].min(),
    "~",
    gfs_train_hourly["forecast_kst_dtm"].max(),
)

print(
    "GFS Test:",
    gfs_test_hourly["forecast_kst_dtm"].min(),
    "~",
    gfs_test_hourly["forecast_kst_dtm"].max(),
)


# 7. 집계 후 결측치 확인
print("\nLDAPS Train 결측치:", ldaps_train_hourly.isna().sum().sum())
print("LDAPS Test 결측치:", ldaps_test_hourly.isna().sum().sum())
print("GFS Train 결측치:", gfs_train_hourly.isna().sum().sum())
print("GFS Test 결측치:", gfs_test_hourly.isna().sum().sum())


# 8. Test LDAPS 결측 보간
ldaps_test_feature_cols = [
    col
    for col in ldaps_test_hourly.columns
    if col != "forecast_kst_dtm"
]

ldaps_test_hourly[ldaps_test_feature_cols] = (
    ldaps_test_hourly[ldaps_test_feature_cols]
    .interpolate(
        method="linear",
        limit_direction="both",
    )
)


# 9. 보간 후 결측치 확인
print(
    "\nLDAPS Test 보간 후 결측치:",
    ldaps_test_hourly.isna().sum().sum()
)


# 10. Train/Test 컬럼 일치 여부
print(
    "\nLDAPS 컬럼 일치:",
    ldaps_train_hourly.columns.tolist()
    == ldaps_test_hourly.columns.tolist()
)

print(
    "GFS 컬럼 일치:",
    gfs_train_hourly.columns.tolist()
    == gfs_test_hourly.columns.tolist()
)

LDAPS Train: (26304, 125)
LDAPS Test: (8760, 125)
GFS Train: (26304, 145)
GFS Test: (8760, 145)

시간 범위
LDAPS Train: 2022-01-01 01:00:00 ~ 2025-01-01 00:00:00
LDAPS Test: 2025-01-01 01:00:00 ~ 2026-01-01 00:00:00
GFS Train: 2022-01-01 01:00:00 ~ 2025-01-01 00:00:00
GFS Test: 2025-01-01 01:00:00 ~ 2026-01-01 00:00:00

LDAPS Train 결측치: 0
LDAPS Test 결측치: 188
GFS Train 결측치: 0
GFS Test 결측치: 0

LDAPS Test 보간 후 결측치: 0

LDAPS 컬럼 일치: True
GFS 컬럼 일치: True


In [7]:
# 1. 보간 전 LDAPS Test 결측 위치 재확인
ldaps_test_check = aggregate_weather(
    ldaps_test_raw,
    prefix="ldaps",
)

missing_by_time = (
    ldaps_test_check
    .set_index("forecast_kst_dtm")
    .isna()
    .sum(axis=1)
)

print("LDAPS Test 결측 발생 시간")
print(missing_by_time[missing_by_time > 0])

print(
    "\n결측 발생 시간 수:",
    (missing_by_time > 0).sum(),
)


# 2. Train 기준 상수 컬럼 탐색
ldaps_constant_cols = [
    col
    for col in ldaps_train_hourly.columns
    if col != "forecast_kst_dtm"
    and ldaps_train_hourly[col].nunique(dropna=False) <= 1
]

gfs_constant_cols = [
    col
    for col in gfs_train_hourly.columns
    if col != "forecast_kst_dtm"
    and gfs_train_hourly[col].nunique(dropna=False) <= 1
]

print("\nLDAPS 상수 컬럼 수:", len(ldaps_constant_cols))
print(ldaps_constant_cols)

print("\nGFS 상수 컬럼 수:", len(gfs_constant_cols))
print(gfs_constant_cols)


# 3. Train/Test에서 동일한 상수 컬럼 제거
ldaps_train_hourly = (
    ldaps_train_hourly
    .drop(columns=ldaps_constant_cols)
)

ldaps_test_hourly = (
    ldaps_test_hourly
    .drop(columns=ldaps_constant_cols)
)

gfs_train_hourly = (
    gfs_train_hourly
    .drop(columns=gfs_constant_cols)
)

gfs_test_hourly = (
    gfs_test_hourly
    .drop(columns=gfs_constant_cols)
)


# 4. 제거 후 구조 확인
print("\n상수 컬럼 제거 후")
print("LDAPS Train:", ldaps_train_hourly.shape)
print("LDAPS Test:", ldaps_test_hourly.shape)
print("GFS Train:", gfs_train_hourly.shape)
print("GFS Test:", gfs_test_hourly.shape)

print(
    "LDAPS 컬럼 일치:",
    ldaps_train_hourly.columns.tolist()
    == ldaps_test_hourly.columns.tolist()
)

print(
    "GFS 컬럼 일치:",
    gfs_train_hourly.columns.tolist()
    == gfs_test_hourly.columns.tolist()
)


# 5. LDAPS와 GFS 병합
weather_train = pd.merge(
    ldaps_train_hourly,
    gfs_train_hourly,
    on="forecast_kst_dtm",
    how="inner",
    validate="one_to_one",
)

weather_test = pd.merge(
    ldaps_test_hourly,
    gfs_test_hourly,
    on="forecast_kst_dtm",
    how="inner",
    validate="one_to_one",
)


# 6. 병합 결과 검증
print("\n기상 데이터 병합 결과")
print("Weather Train:", weather_train.shape)
print("Weather Test:", weather_test.shape)

print(
    "Train 시간 중복:",
    weather_train["forecast_kst_dtm"].duplicated().sum()
)

print(
    "Test 시간 중복:",
    weather_test["forecast_kst_dtm"].duplicated().sum()
)

print(
    "Train 결측치:",
    weather_train.isna().sum().sum()
)

print(
    "Test 결측치:",
    weather_test.isna().sum().sum()
)

print(
    "Train 시간 정렬:",
    weather_train["forecast_kst_dtm"].is_monotonic_increasing
)

print(
    "Test 시간 정렬:",
    weather_test["forecast_kst_dtm"].is_monotonic_increasing
)

LDAPS Test 결측 발생 시간
forecast_kst_dtm
2025-04-08 17:00:00    56
2025-06-18 18:00:00    56
2025-07-18 06:00:00    76
dtype: int64

결측 발생 시간 수: 3

LDAPS 상수 컬럼 수: 9
['ldaps_surface_0_lsm_mean', 'ldaps_surface_0_lsm_std', 'ldaps_surface_0_lsm_min', 'ldaps_surface_0_lsm_max', 'ldaps_surface_0_h_mean', 'ldaps_surface_0_h_std', 'ldaps_surface_0_h_min', 'ldaps_surface_0_h_max', 'ldaps_lead_time_hour_std']

GFS 상수 컬럼 수: 1
['gfs_lead_time_hour_std']

상수 컬럼 제거 후
LDAPS Train: (26304, 116)
LDAPS Test: (8760, 116)
GFS Train: (26304, 144)
GFS Test: (8760, 144)
LDAPS 컬럼 일치: True
GFS 컬럼 일치: True

기상 데이터 병합 결과
Weather Train: (26304, 259)
Weather Test: (8760, 259)
Train 시간 중복: 0
Test 시간 중복: 0
Train 결측치: 0
Test 결측치: 0
Train 시간 정렬: True
Test 시간 정렬: True


In [8]:
# 1. Label과 제출 파일 시간 컬럼 변환
labels = labels_raw.copy()

labels["kst_dtm"] = pd.to_datetime(
    labels["kst_dtm"]
)

labels = labels.rename(
    columns={
        "kst_dtm": "forecast_kst_dtm",
    }
)

sample_submission["forecast_kst_dtm"] = pd.to_datetime(
    sample_submission["forecast_kst_dtm"]
)


# 2. 기상 데이터와 발전량 결합
train = pd.merge(
    weather_train,
    labels,
    on="forecast_kst_dtm",
    how="left",
    validate="one_to_one",
)

test = pd.merge(
    sample_submission[
        [
            "forecast_id",
            "forecast_kst_dtm",
        ]
    ],
    weather_test,
    on="forecast_kst_dtm",
    how="left",
    validate="one_to_one",
)


# 3. 시간 파생변수 생성 함수
def add_time_features(df):
    result = df.copy()

    dt = result["forecast_kst_dtm"]

    result["year"] = dt.dt.year
    result["month"] = dt.dt.month
    result["day"] = dt.dt.day
    result["hour"] = dt.dt.hour
    result["dayofweek"] = dt.dt.dayofweek
    result["dayofyear"] = dt.dt.dayofyear
    result["weekofyear"] = dt.dt.isocalendar().week.astype(int)
    result["quarter"] = dt.dt.quarter

    result["hour_sin"] = np.sin(
        2 * np.pi * result["hour"] / 24
    )
    result["hour_cos"] = np.cos(
        2 * np.pi * result["hour"] / 24
    )

    result["dayofyear_sin"] = np.sin(
        2 * np.pi * result["dayofyear"] / 365.25
    )
    result["dayofyear_cos"] = np.cos(
        2 * np.pi * result["dayofyear"] / 365.25
    )

    result["month_sin"] = np.sin(
        2 * np.pi * result["month"] / 12
    )
    result["month_cos"] = np.cos(
        2 * np.pi * result["month"] / 12
    )

    result["is_weekend"] = (
        result["dayofweek"] >= 5
    ).astype(int)

    return result


# 4. Train/Test에 동일한 시간 변수 추가
train = add_time_features(train)
test = add_time_features(test)


# 5. 모델 입력 컬럼 구성
target_cols = [
    "kpx_group_1",
    "kpx_group_2",
    "kpx_group_3",
]

exclude_cols = [
    "forecast_id",
    "forecast_kst_dtm",
] + target_cols

model_feature_cols = [
    col
    for col in train.columns
    if col not in exclude_cols
]


# 6. Train/Test feature 순서 일치
test = test[
    [
        "forecast_id",
        "forecast_kst_dtm",
    ]
    + model_feature_cols
]


# 7. 그룹별 학습 데이터 생성
train_group_1 = (
    train
    .dropna(subset=["kpx_group_1"])
    .reset_index(drop=True)
)

train_group_2 = (
    train
    .dropna(subset=["kpx_group_2"])
    .reset_index(drop=True)
)

train_group_3 = (
    train
    .dropna(subset=["kpx_group_3"])
    .reset_index(drop=True)
)


# 8. 최종 검증
print("Train:", train.shape)
print("Test:", test.shape)
print("Feature 수:", len(model_feature_cols))

print("\n그룹별 데이터")
print("Group 1:", train_group_1.shape)
print("Group 2:", train_group_2.shape)
print("Group 3:", train_group_3.shape)

print("\nTarget 결측치")
print(train[target_cols].isna().sum())

print("\nFeature 결측치")
print(
    "Train:",
    train[model_feature_cols].isna().sum().sum()
)
print(
    "Test:",
    test[model_feature_cols].isna().sum().sum()
)

print("\n무한값")
print(
    "Train:",
    np.isinf(
        train[model_feature_cols].to_numpy()
    ).sum()
)
print(
    "Test:",
    np.isinf(
        test[model_feature_cols].to_numpy()
    ).sum()
)

print(
    "\nFeature 컬럼 일치:",
    train[model_feature_cols].columns.tolist()
    == test[model_feature_cols].columns.tolist()
)

print(
    "제출 시간 순서 일치:",
    (
        test["forecast_kst_dtm"].values
        == sample_submission["forecast_kst_dtm"].values
    ).all()
)

Train: (26304, 277)
Test: (8760, 275)
Feature 수: 273

그룹별 데이터
Group 1: (26200, 277)
Group 2: (26201, 277)
Group 3: (17538, 277)

Target 결측치
kpx_group_1     104
kpx_group_2     103
kpx_group_3    8766
dtype: int64

Feature 결측치
Train: 0
Test: 0

무한값
Train: 0
Test: 0

Feature 컬럼 일치: True
제출 시간 순서 일치: True


In [9]:
from pathlib import Path

processed_dir = Path("processed")
processed_dir.mkdir(exist_ok=True)

train.to_parquet(
    processed_dir / "train_preprocessed.parquet",
    index=False,
)

test.to_parquet(
    processed_dir / "test_preprocessed.parquet",
    index=False,
)

train_group_1.to_parquet(
    processed_dir / "train_group_1.parquet",
    index=False,
)

train_group_2.to_parquet(
    processed_dir / "train_group_2.parquet",
    index=False,
)

train_group_3.to_parquet(
    processed_dir / "train_group_3.parquet",
    index=False,
)

pd.Series(
    model_feature_cols,
    name="feature",
).to_csv(
    processed_dir / "model_feature_cols.csv",
    index=False,
)

print("전처리 파일 저장 완료")

전처리 파일 저장 완료


In [10]:
# 1. 기존과 동일한 시계열 검증 구간 설정
valid_start = pd.Timestamp("2024-10-01 01:00:00")

def split_group_data(group_df, target_col):
    train_mask = (
        group_df["forecast_kst_dtm"] < valid_start
    )

    valid_mask = (
        group_df["forecast_kst_dtm"] >= valid_start
    )

    X_train = group_df.loc[
        train_mask,
        model_feature_cols,
    ]

    y_train = group_df.loc[
        train_mask,
        target_col,
    ]

    X_valid = group_df.loc[
        valid_mask,
        model_feature_cols,
    ]

    y_valid = group_df.loc[
        valid_mask,
        target_col,
    ]

    return (
        X_train,
        X_valid,
        y_train,
        y_valid,
    )


# 2. 그룹별 학습·검증 데이터 생성
X_train_1, X_valid_1, y_train_1, y_valid_1 = split_group_data(
    train_group_1,
    "kpx_group_1",
)

X_train_2, X_valid_2, y_train_2, y_valid_2 = split_group_data(
    train_group_2,
    "kpx_group_2",
)

X_train_3, X_valid_3, y_train_3, y_valid_3 = split_group_data(
    train_group_3,
    "kpx_group_3",
)


# 3. 데이터 크기 확인
print(
    "Group 1:",
    X_train_1.shape,
    X_valid_1.shape,
)

print(
    "Group 2:",
    X_train_2.shape,
    X_valid_2.shape,
)

print(
    "Group 3:",
    X_train_3.shape,
    X_valid_3.shape,
)


# 4. LightGBM 학습 함수
def train_lgbm_baseline(
    X_train,
    y_train,
    X_valid,
    y_valid,
    capacity,
):
    model = LGBMRegressor(
        objective="regression",
        n_estimators=3000,
        learning_rate=0.03,
        num_leaves=31,
        max_depth=-1,
        min_child_samples=20,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=1.0,
        random_state=42,
        n_jobs=-1,
        verbosity=-1,
    )

    model.fit(
        X_train,
        y_train,
        eval_set=[
            (
                X_valid,
                y_valid,
            )
        ],
        eval_metric="rmse",
        callbacks=[
            early_stopping(
                stopping_rounds=100,
                verbose=False,
            )
        ],
    )

    pred = model.predict(
        X_valid,
        num_iteration=model.best_iteration_,
    )

    pred = np.clip(
        pred,
        0,
        capacity,
    )

    rmse = mean_squared_error(
        y_valid,
        pred,
    ) ** 0.5

    mae = mean_absolute_error(
        y_valid,
        pred,
    )

    r2 = r2_score(
        y_valid,
        pred,
    )

    return (
        model,
        pred,
        rmse,
        mae,
        r2,
    )


# 5. 그룹별 LightGBM 학습
(
    lgbm_new_1,
    pred_lgbm_new_1,
    rmse_lgbm_new_1,
    mae_lgbm_new_1,
    r2_lgbm_new_1,
) = train_lgbm_baseline(
    X_train_1,
    y_train_1,
    X_valid_1,
    y_valid_1,
    capacity=21600,
)

(
    lgbm_new_2,
    pred_lgbm_new_2,
    rmse_lgbm_new_2,
    mae_lgbm_new_2,
    r2_lgbm_new_2,
) = train_lgbm_baseline(
    X_train_2,
    y_train_2,
    X_valid_2,
    y_valid_2,
    capacity=21600,
)

(
    lgbm_new_3,
    pred_lgbm_new_3,
    rmse_lgbm_new_3,
    mae_lgbm_new_3,
    r2_lgbm_new_3,
) = train_lgbm_baseline(
    X_train_3,
    y_train_3,
    X_valid_3,
    y_valid_3,
    capacity=21000,
)


# 6. 결과 정리
new_preprocessing_result = pd.DataFrame({
    "group": [
        "Group 1",
        "Group 2",
        "Group 3",
    ],
    "best_iteration": [
        lgbm_new_1.best_iteration_,
        lgbm_new_2.best_iteration_,
        lgbm_new_3.best_iteration_,
    ],
    "rmse": [
        rmse_lgbm_new_1,
        rmse_lgbm_new_2,
        rmse_lgbm_new_3,
    ],
    "mae": [
        mae_lgbm_new_1,
        mae_lgbm_new_2,
        mae_lgbm_new_3,
    ],
    "r2": [
        r2_lgbm_new_1,
        r2_lgbm_new_2,
        r2_lgbm_new_3,
    ],
})

print(new_preprocessing_result)

print(
    "\n평균 RMSE:",
    new_preprocessing_result["rmse"].mean(),
)

print(
    "평균 MAE:",
    new_preprocessing_result["mae"].mean(),
)

print(
    "평균 R2:",
    new_preprocessing_result["r2"].mean(),
)

Group 1: (23992, 273) (2208, 273)
Group 2: (23993, 273) (2208, 273)
Group 3: (15330, 273) (2208, 273)
     group  best_iteration         rmse          mae        r2
0  Group 1             329  2749.596162  1985.369980  0.853112
1  Group 2             205  3104.474671  2249.767401  0.835293
2  Group 3             152  2865.118130  1934.810953  0.800499

평균 RMSE: 2906.3963211894775
평균 MAE: 2056.6494447222335
평균 R2: 0.8296345882257437


In [11]:
# 기존 전처리 데이터와 feature 목록 불러오기
old_train = pd.read_parquet(
    "train_preprocessed.parquet"
)

old_test = pd.read_parquet(
    "test_preprocessed.parquet"
)

old_feature_cols = pd.read_csv(
    "model_feature_cols.csv"
)["feature"].tolist()

new_feature_cols = model_feature_cols

# 기존에만 있던 feature와 새로 추가된 feature 확인
only_old_features = sorted(
    set(old_feature_cols) - set(new_feature_cols)
)

only_new_features = sorted(
    set(new_feature_cols) - set(old_feature_cols)
)

common_features = sorted(
    set(old_feature_cols) & set(new_feature_cols)
)

print("기존 feature 수:", len(old_feature_cols))
print("새 feature 수:", len(new_feature_cols))
print("공통 feature 수:", len(common_features))

print("\n기존에만 존재하는 feature")
for col in only_old_features:
    print(col)

print("\n새 전처리에만 존재하는 feature")
for col in only_new_features:
    print(col)

print("\n기존 Train shape:", old_train.shape)
print("기존 Test shape:", old_test.shape)

print(
    "기존 feature 결측치:",
    old_train[old_feature_cols].isna().sum().sum(),
)

print(
    "새 feature 결측치:",
    train[new_feature_cols].isna().sum().sum(),
)

기존 feature 수: 289
새 feature 수: 273
공통 feature 수: 265

기존에만 존재하는 feature
gfs_wind_speed_100m_max
gfs_wind_speed_100m_mean
gfs_wind_speed_100m_min
gfs_wind_speed_100m_std
gfs_wind_speed_10m_max
gfs_wind_speed_10m_mean
gfs_wind_speed_10m_min
gfs_wind_speed_10m_std
gfs_wind_speed_80m_max
gfs_wind_speed_80m_mean
gfs_wind_speed_80m_min
gfs_wind_speed_80m_std
ldaps_surface_0_h_max
ldaps_surface_0_h_mean
ldaps_surface_0_h_min
ldaps_surface_0_h_std
ldaps_surface_0_lsm_max
ldaps_surface_0_lsm_mean
ldaps_surface_0_lsm_min
ldaps_surface_0_lsm_std
ldaps_wind_speed_10m_max
ldaps_wind_speed_10m_mean
ldaps_wind_speed_10m_min
ldaps_wind_speed_10m_std

새 전처리에만 존재하는 feature
gfs_lead_time_hour_max
gfs_lead_time_hour_mean
gfs_lead_time_hour_min
is_weekend
ldaps_lead_time_hour_max
ldaps_lead_time_hour_mean
ldaps_lead_time_hour_min
quarter

기존 Train shape: (26304, 293)
기존 Test shape: (8760, 290)
기존 feature 결측치: 0
새 feature 결측치: 0


In [12]:
# 1. LDAPS 10m 풍속 생성
for df in [
    ldaps_train_raw,
    ldaps_test_raw,
]:
    df["wind_speed_10m"] = np.sqrt(
        df["heightAboveGround_10_10u"] ** 2
        + df["heightAboveGround_10_10v"] ** 2
    )


# 2. GFS 10m, 80m, 100m 풍속 생성
for df in [
    gfs_train_raw,
    gfs_test_raw,
]:
    df["wind_speed_10m"] = np.sqrt(
        df["heightAboveGround_10_10u"] ** 2
        + df["heightAboveGround_10_10v"] ** 2
    )

    df["wind_speed_80m"] = np.sqrt(
        df["heightAboveGround_80_u"] ** 2
        + df["heightAboveGround_80_v"] ** 2
    )

    df["wind_speed_100m"] = np.sqrt(
        df["heightAboveGround_100_100u"] ** 2
        + df["heightAboveGround_100_100v"] ** 2
    )


# 3. 리드타임 컬럼 제거
for df in [
    ldaps_train_raw,
    ldaps_test_raw,
    gfs_train_raw,
    gfs_test_raw,
]:
    if "lead_time_hour" in df.columns:
        df.drop(
            columns="lead_time_hour",
            inplace=True,
        )


# 4. 기상 데이터 다시 집계
ldaps_train_hourly = aggregate_weather(
    ldaps_train_raw,
    prefix="ldaps",
)

ldaps_test_hourly = aggregate_weather(
    ldaps_test_raw,
    prefix="ldaps",
)

gfs_train_hourly = aggregate_weather(
    gfs_train_raw,
    prefix="gfs",
)

gfs_test_hourly = aggregate_weather(
    gfs_test_raw,
    prefix="gfs",
)


# 5. LDAPS Test 결측 보간
ldaps_test_feature_cols = [
    col
    for col in ldaps_test_hourly.columns
    if col != "forecast_kst_dtm"
]

ldaps_test_hourly[ldaps_test_feature_cols] = (
    ldaps_test_hourly[ldaps_test_feature_cols]
    .interpolate(
        method="linear",
        limit_direction="both",
    )
)


# 6. 상수 컬럼 제거
ldaps_constant_cols = [
    col
    for col in ldaps_train_hourly.columns
    if col != "forecast_kst_dtm"
    and ldaps_train_hourly[col].nunique(dropna=False) <= 1
]

gfs_constant_cols = [
    col
    for col in gfs_train_hourly.columns
    if col != "forecast_kst_dtm"
    and gfs_train_hourly[col].nunique(dropna=False) <= 1
]

ldaps_train_hourly = ldaps_train_hourly.drop(
    columns=ldaps_constant_cols
)

ldaps_test_hourly = ldaps_test_hourly.drop(
    columns=ldaps_constant_cols
)

gfs_train_hourly = gfs_train_hourly.drop(
    columns=gfs_constant_cols
)

gfs_test_hourly = gfs_test_hourly.drop(
    columns=gfs_constant_cols
)


# 7. 기상 데이터 병합
weather_train = pd.merge(
    ldaps_train_hourly,
    gfs_train_hourly,
    on="forecast_kst_dtm",
    how="inner",
    validate="one_to_one",
)

weather_test = pd.merge(
    ldaps_test_hourly,
    gfs_test_hourly,
    on="forecast_kst_dtm",
    how="inner",
    validate="one_to_one",
)

print("Weather Train:", weather_train.shape)
print("Weather Test:", weather_test.shape)
print("Train 결측치:", weather_train.isna().sum().sum())
print("Test 결측치:", weather_test.isna().sum().sum())

Weather Train: (26304, 269)
Weather Test: (8760, 269)
Train 결측치: 0
Test 결측치: 0


In [13]:
def add_time_features(df):
    result = df.copy()

    dt = result["forecast_kst_dtm"]

    result["year"] = dt.dt.year
    result["month"] = dt.dt.month
    result["day"] = dt.dt.day
    result["hour"] = dt.dt.hour
    result["dayofweek"] = dt.dt.dayofweek
    result["dayofyear"] = dt.dt.dayofyear
    result["weekofyear"] = dt.dt.isocalendar().week.astype(int)

    result["hour_sin"] = np.sin(
        2 * np.pi * result["hour"] / 24
    )
    result["hour_cos"] = np.cos(
        2 * np.pi * result["hour"] / 24
    )

    result["dayofyear_sin"] = np.sin(
        2 * np.pi * result["dayofyear"] / 365.25
    )
    result["dayofyear_cos"] = np.cos(
        2 * np.pi * result["dayofyear"] / 365.25
    )

    result["month_sin"] = np.sin(
        2 * np.pi * result["month"] / 12
    )
    result["month_cos"] = np.cos(
        2 * np.pi * result["month"] / 12
    )

    return result

In [14]:
# 1. Label과 sample submission 시간 컬럼 정리
labels = labels_raw.copy()

labels["kst_dtm"] = pd.to_datetime(
    labels["kst_dtm"]
)

labels = labels.rename(
    columns={
        "kst_dtm": "forecast_kst_dtm",
    }
)

sample_submission["forecast_kst_dtm"] = pd.to_datetime(
    sample_submission["forecast_kst_dtm"]
)


# 2. 기상 데이터와 target 병합
train = pd.merge(
    weather_train,
    labels,
    on="forecast_kst_dtm",
    how="left",
    validate="one_to_one",
)

test = pd.merge(
    sample_submission[
        [
            "forecast_id",
            "forecast_kst_dtm",
        ]
    ],
    weather_test,
    on="forecast_kst_dtm",
    how="left",
    validate="one_to_one",
)


# 3. 시간 파생변수 생성
train = add_time_features(train)
test = add_time_features(test)


# 4. 모델 feature 구성
target_cols = [
    "kpx_group_1",
    "kpx_group_2",
    "kpx_group_3",
]

exclude_cols = [
    "forecast_id",
    "forecast_kst_dtm",
] + target_cols

model_feature_cols = [
    col
    for col in train.columns
    if col not in exclude_cols
]

test = test[
    [
        "forecast_id",
        "forecast_kst_dtm",
    ]
    + model_feature_cols
]


# 5. 그룹별 학습 데이터 생성
train_group_1 = (
    train
    .dropna(subset=["kpx_group_1"])
    .reset_index(drop=True)
)

train_group_2 = (
    train
    .dropna(subset=["kpx_group_2"])
    .reset_index(drop=True)
)

train_group_3 = (
    train
    .dropna(subset=["kpx_group_3"])
    .reset_index(drop=True)
)


# 6. 전처리 결과 검증
print("Train:", train.shape)
print("Test:", test.shape)
print("Feature 수:", len(model_feature_cols))

print("\n그룹별 데이터")
print("Group 1:", train_group_1.shape)
print("Group 2:", train_group_2.shape)
print("Group 3:", train_group_3.shape)

print("\nFeature 결측치")
print(
    "Train:",
    train[model_feature_cols].isna().sum().sum()
)

print(
    "Test:",
    test[model_feature_cols].isna().sum().sum()
)

print("\n무한값")
print(
    "Train:",
    np.isinf(
        train[model_feature_cols].to_numpy()
    ).sum()
)

print(
    "Test:",
    np.isinf(
        test[model_feature_cols].to_numpy()
    ).sum()
)

print(
    "\nFeature 컬럼 일치:",
    train[model_feature_cols].columns.tolist()
    == test[model_feature_cols].columns.tolist()
)


# 7. 기존과 동일한 검증 구간 분리
valid_start = pd.Timestamp(
    "2024-10-01 01:00:00"
)

def split_group_data(
    group_df,
    target_col,
):
    train_mask = (
        group_df["forecast_kst_dtm"]
        < valid_start
    )

    valid_mask = (
        group_df["forecast_kst_dtm"]
        >= valid_start
    )

    X_train = group_df.loc[
        train_mask,
        model_feature_cols,
    ]

    y_train = group_df.loc[
        train_mask,
        target_col,
    ]

    X_valid = group_df.loc[
        valid_mask,
        model_feature_cols,
    ]

    y_valid = group_df.loc[
        valid_mask,
        target_col,
    ]

    return (
        X_train,
        X_valid,
        y_train,
        y_valid,
    )


X_train_1, X_valid_1, y_train_1, y_valid_1 = split_group_data(
    train_group_1,
    "kpx_group_1",
)

X_train_2, X_valid_2, y_train_2, y_valid_2 = split_group_data(
    train_group_2,
    "kpx_group_2",
)

X_train_3, X_valid_3, y_train_3, y_valid_3 = split_group_data(
    train_group_3,
    "kpx_group_3",
)


# 8. LightGBM 재검증
(
    lgbm_new_1,
    pred_lgbm_new_1,
    rmse_lgbm_new_1,
    mae_lgbm_new_1,
    r2_lgbm_new_1,
) = train_lgbm_baseline(
    X_train_1,
    y_train_1,
    X_valid_1,
    y_valid_1,
    capacity=21600,
)

(
    lgbm_new_2,
    pred_lgbm_new_2,
    rmse_lgbm_new_2,
    mae_lgbm_new_2,
    r2_lgbm_new_2,
) = train_lgbm_baseline(
    X_train_2,
    y_train_2,
    X_valid_2,
    y_valid_2,
    capacity=21600,
)

(
    lgbm_new_3,
    pred_lgbm_new_3,
    rmse_lgbm_new_3,
    mae_lgbm_new_3,
    r2_lgbm_new_3,
) = train_lgbm_baseline(
    X_train_3,
    y_train_3,
    X_valid_3,
    y_valid_3,
    capacity=21000,
)


# 9. 기존 성능과 비교
comparison = pd.DataFrame({
    "group": [
        "Group 1",
        "Group 2",
        "Group 3",
    ],
    "old_rmse": [
        2753.374311,
        3065.084193,
        2759.233446,
    ],
    "new_rmse": [
        rmse_lgbm_new_1,
        rmse_lgbm_new_2,
        rmse_lgbm_new_3,
    ],
    "new_mae": [
        mae_lgbm_new_1,
        mae_lgbm_new_2,
        mae_lgbm_new_3,
    ],
    "new_r2": [
        r2_lgbm_new_1,
        r2_lgbm_new_2,
        r2_lgbm_new_3,
    ],
})

comparison["rmse_change"] = (
    comparison["new_rmse"]
    - comparison["old_rmse"]
)

print(comparison)

print(
    "\n기존 평균 RMSE:",
    comparison["old_rmse"].mean()
)

print(
    "새 평균 RMSE:",
    comparison["new_rmse"].mean()
)

Train: (26304, 285)
Test: (8760, 283)
Feature 수: 281

그룹별 데이터
Group 1: (26200, 285)
Group 2: (26201, 285)
Group 3: (17538, 285)

Feature 결측치
Train: 0
Test: 0

무한값
Train: 0
Test: 0

Feature 컬럼 일치: True
     group     old_rmse     new_rmse      new_mae    new_r2  rmse_change
0  Group 1  2753.374311  2734.167113  1967.858327  0.854756   -19.207198
1  Group 2  3065.084193  3073.713333  2250.815936  0.838541     8.629140
2  Group 3  2759.233446  2781.192931  1895.937666  0.812015    21.959485

기존 평균 RMSE: 2859.23065
새 평균 RMSE: 2863.0244589877016


In [15]:
from pathlib import Path

processed_dir = Path("processed_v2")
processed_dir.mkdir(exist_ok=True)

train.to_parquet(
    processed_dir / "train_preprocessed.parquet",
    index=False,
)

test.to_parquet(
    processed_dir / "test_preprocessed.parquet",
    index=False,
)

train_group_1.to_parquet(
    processed_dir / "train_group_1.parquet",
    index=False,
)

train_group_2.to_parquet(
    processed_dir / "train_group_2.parquet",
    index=False,
)

train_group_3.to_parquet(
    processed_dir / "train_group_3.parquet",
    index=False,
)

pd.Series(
    model_feature_cols,
    name="feature",
).to_csv(
    processed_dir / "model_feature_cols.csv",
    index=False,
)

print("processed_v2 저장 완료")

processed_v2 저장 완료


In [16]:
for df in [
    gfs_train_raw,
    gfs_test_raw,
]:
    df["wind_speed_diff_100m_10m"] = (
        df["wind_speed_100m"]
        - df["wind_speed_10m"]
    )

    df["wind_speed_diff_100m_80m"] = (
        df["wind_speed_100m"]
        - df["wind_speed_80m"]
    )

    safe_wind_10m = df["wind_speed_10m"].clip(lower=0.5)
    safe_wind_100m = df["wind_speed_100m"].clip(lower=0.5)

    df["wind_shear_alpha_100m_10m"] = (
        np.log(
            safe_wind_100m
            / safe_wind_10m
        )
        / np.log(100 / 10)
    )

print(
    gfs_train_raw[
        [
            "wind_speed_10m",
            "wind_speed_80m",
            "wind_speed_100m",
            "wind_speed_diff_100m_10m",
            "wind_speed_diff_100m_80m",
            "wind_shear_alpha_100m_10m",
        ]
    ].describe()
)

print(
    "\n무한값:",
    np.isinf(
        gfs_train_raw[
            [
                "wind_speed_diff_100m_10m",
                "wind_speed_diff_100m_80m",
                "wind_shear_alpha_100m_10m",
            ]
        ].to_numpy()
    ).sum()
)

       wind_speed_10m  wind_speed_80m  wind_speed_100m  \
count   236736.000000   236736.000000    236736.000000   
mean         2.607444        3.684370         3.833557   
std          1.764672        2.746077         2.898197   
min          0.004578        0.002918         0.001271   
25%          1.420227        1.827960         1.868903   
50%          2.182720        2.948741         3.043425   
75%          3.293716        4.675821         4.885070   
max         25.761482       32.118738        32.895843   

       wind_speed_diff_100m_10m  wind_speed_diff_100m_80m  \
count             236736.000000             236736.000000   
mean                   1.226113                  0.149187   
std                    1.380168                  0.197461   
min                   -2.941575                 -0.713110   
25%                    0.330777                  0.025787   
50%                    0.837360                  0.097490   
75%                    1.699111                  0

In [17]:
gfs_train_hourly_exp1 = aggregate_weather(
    gfs_train_raw,
    prefix="gfs",
)

gfs_test_hourly_exp1 = aggregate_weather(
    gfs_test_raw,
    prefix="gfs",
)

gfs_constant_cols_exp1 = [
    col
    for col in gfs_train_hourly_exp1.columns
    if col != "forecast_kst_dtm"
    and gfs_train_hourly_exp1[col].nunique(
        dropna=False
    ) <= 1
]

gfs_train_hourly_exp1 = (
    gfs_train_hourly_exp1
    .drop(columns=gfs_constant_cols_exp1)
)

gfs_test_hourly_exp1 = (
    gfs_test_hourly_exp1
    .drop(columns=gfs_constant_cols_exp1)
)

print(
    "GFS Train:",
    gfs_train_hourly_exp1.shape
)

print(
    "GFS Test:",
    gfs_test_hourly_exp1.shape
)

print(
    "Train 결측치:",
    gfs_train_hourly_exp1.isna().sum().sum()
)

print(
    "Test 결측치:",
    gfs_test_hourly_exp1.isna().sum().sum()
)

GFS Train: (26304, 165)
GFS Test: (8760, 165)
Train 결측치: 0
Test 결측치: 0


In [18]:
weather_train_exp1 = pd.merge(
    ldaps_train_hourly,
    gfs_train_hourly_exp1,
    on="forecast_kst_dtm",
    how="inner",
    validate="one_to_one",
)

weather_test_exp1 = pd.merge(
    ldaps_test_hourly,
    gfs_test_hourly_exp1,
    on="forecast_kst_dtm",
    how="inner",
    validate="one_to_one",
)

print(
    "Weather Train:",
    weather_train_exp1.shape
)

print(
    "Weather Test:",
    weather_test_exp1.shape
)

print(
    "Train 결측치:",
    weather_train_exp1.isna().sum().sum()
)

print(
    "Test 결측치:",
    weather_test_exp1.isna().sum().sum()
)

Weather Train: (26304, 281)
Weather Test: (8760, 281)
Train 결측치: 0
Test 결측치: 0


In [19]:
train_exp1 = pd.merge(
    weather_train_exp1,
    labels,
    on="forecast_kst_dtm",
    how="left",
    validate="one_to_one",
)

test_exp1 = pd.merge(
    sample_submission[
        [
            "forecast_id",
            "forecast_kst_dtm",
        ]
    ],
    weather_test_exp1,
    on="forecast_kst_dtm",
    how="left",
    validate="one_to_one",
)

train_exp1 = add_time_features(
    train_exp1
)

test_exp1 = add_time_features(
    test_exp1
)

exclude_cols = [
    "forecast_id",
    "forecast_kst_dtm",
    "kpx_group_1",
    "kpx_group_2",
    "kpx_group_3",
]

feature_cols_exp1 = [
    col
    for col in train_exp1.columns
    if col not in exclude_cols
]

test_exp1 = test_exp1[
    [
        "forecast_id",
        "forecast_kst_dtm",
    ]
    + feature_cols_exp1
]

train_group_1_exp1 = (
    train_exp1
    .dropna(subset=["kpx_group_1"])
    .reset_index(drop=True)
)

train_group_2_exp1 = (
    train_exp1
    .dropna(subset=["kpx_group_2"])
    .reset_index(drop=True)
)

train_group_3_exp1 = (
    train_exp1
    .dropna(subset=["kpx_group_3"])
    .reset_index(drop=True)
)

print(
    "Exp1 feature 수:",
    len(feature_cols_exp1)
)

print(
    "Train 결측치:",
    train_exp1[feature_cols_exp1]
    .isna()
    .sum()
    .sum()
)

print(
    "Test 결측치:",
    test_exp1[feature_cols_exp1]
    .isna()
    .sum()
    .sum()
)

Exp1 feature 수: 293
Train 결측치: 0
Test 결측치: 0


In [20]:
def split_exp1(
    group_df,
    target_col,
):
    train_mask = (
        group_df["forecast_kst_dtm"]
        < valid_start
    )

    valid_mask = (
        group_df["forecast_kst_dtm"]
        >= valid_start
    )

    return (
        group_df.loc[
            train_mask,
            feature_cols_exp1,
        ],
        group_df.loc[
            valid_mask,
            feature_cols_exp1,
        ],
        group_df.loc[
            train_mask,
            target_col,
        ],
        group_df.loc[
            valid_mask,
            target_col,
        ],
    )


X_train_1_exp1, X_valid_1_exp1, y_train_1_exp1, y_valid_1_exp1 = split_exp1(
    train_group_1_exp1,
    "kpx_group_1",
)

X_train_2_exp1, X_valid_2_exp1, y_train_2_exp1, y_valid_2_exp1 = split_exp1(
    train_group_2_exp1,
    "kpx_group_2",
)

X_train_3_exp1, X_valid_3_exp1, y_train_3_exp1, y_valid_3_exp1 = split_exp1(
    train_group_3_exp1,
    "kpx_group_3",
)

(
    lgbm_exp1_1,
    pred_exp1_1,
    rmse_exp1_1,
    mae_exp1_1,
    r2_exp1_1,
) = train_lgbm_baseline(
    X_train_1_exp1,
    y_train_1_exp1,
    X_valid_1_exp1,
    y_valid_1_exp1,
    capacity=21600,
)

(
    lgbm_exp1_2,
    pred_exp1_2,
    rmse_exp1_2,
    mae_exp1_2,
    r2_exp1_2,
) = train_lgbm_baseline(
    X_train_2_exp1,
    y_train_2_exp1,
    X_valid_2_exp1,
    y_valid_2_exp1,
    capacity=21600,
)

(
    lgbm_exp1_3,
    pred_exp1_3,
    rmse_exp1_3,
    mae_exp1_3,
    r2_exp1_3,
) = train_lgbm_baseline(
    X_train_3_exp1,
    y_train_3_exp1,
    X_valid_3_exp1,
    y_valid_3_exp1,
    capacity=21000,
)

exp1_result = pd.DataFrame({
    "group": [
        "Group 1",
        "Group 2",
        "Group 3",
    ],
    "baseline_rmse": [
        2734.167113,
        3073.713333,
        2781.192931,
    ],
    "exp1_rmse": [
        rmse_exp1_1,
        rmse_exp1_2,
        rmse_exp1_3,
    ],
    "exp1_mae": [
        mae_exp1_1,
        mae_exp1_2,
        mae_exp1_3,
    ],
    "exp1_r2": [
        r2_exp1_1,
        r2_exp1_2,
        r2_exp1_3,
    ],
})

exp1_result["rmse_change"] = (
    exp1_result["exp1_rmse"]
    - exp1_result["baseline_rmse"]
)

print(exp1_result)

print(
    "\nBaseline 평균 RMSE:",
    exp1_result["baseline_rmse"].mean()
)

print(
    "Exp1 평균 RMSE:",
    exp1_result["exp1_rmse"].mean()
)

     group  baseline_rmse    exp1_rmse     exp1_mae   exp1_r2  rmse_change
0  Group 1    2734.167113  2728.115640  1960.802542  0.855398    -6.051473
1  Group 2    3073.713333  3008.105780  2172.642569  0.845360   -65.607553
2  Group 3    2781.192931  2786.029266  1898.843229  0.811361     4.836335

Baseline 평균 RMSE: 2863.024459
Exp1 평균 RMSE: 2840.7502285223613


In [21]:
R_DRY_AIR = 287.05

for df in [
    gfs_train_raw,
    gfs_test_raw,
]:
    df["wind_speed_cubed_100m"] = (
        df["wind_speed_100m"] ** 3
    )

    df["air_density_proxy"] = (
        df["surface_0_sp"]
        / (
            R_DRY_AIR
            * df["heightAboveGround_2_2t"]
        )
    )

    df["wind_power_density_100m"] = (
        0.5
        * df["air_density_proxy"]
        * df["wind_speed_cubed_100m"]
    )

print(
    gfs_train_raw[
        [
            "wind_speed_100m",
            "wind_speed_cubed_100m",
            "air_density_proxy",
            "wind_power_density_100m",
        ]
    ].describe()
)

print(
    "\n무한값:",
    np.isinf(
        gfs_train_raw[
            [
                "wind_speed_cubed_100m",
                "air_density_proxy",
                "wind_power_density_100m",
            ]
        ].to_numpy()
    ).sum()
)

       wind_speed_100m  wind_speed_cubed_100m  air_density_proxy  \
count    236736.000000           2.367360e+05      236736.000000   
mean          3.833557           1.983012e+02           1.174120   
std           2.898197           6.617284e+02           0.053697   
min           0.001271           2.051012e-09           1.044308   
25%           1.868903           6.527700e+00           1.131515   
50%           3.043425           2.818954e+01           1.172771   
75%           4.885070           1.165768e+02           1.213776   
max          32.895843           3.559779e+04           1.364631   

       wind_power_density_100m  
count             2.367360e+05  
mean              1.173390e+02  
std               3.910693e+02  
min               1.197252e-09  
25%               3.791047e+00  
50%               1.656916e+01  
75%               6.907906e+01  
max               2.085658e+04  

무한값: 0


In [22]:
gfs_train_hourly_exp2 = aggregate_weather(
    gfs_train_raw,
    prefix="gfs",
)

gfs_test_hourly_exp2 = aggregate_weather(
    gfs_test_raw,
    prefix="gfs",
)

gfs_constant_cols_exp2 = [
    col
    for col in gfs_train_hourly_exp2.columns
    if col != "forecast_kst_dtm"
    and gfs_train_hourly_exp2[col].nunique(dropna=False) <= 1
]

gfs_train_hourly_exp2 = gfs_train_hourly_exp2.drop(
    columns=gfs_constant_cols_exp2
)

gfs_test_hourly_exp2 = gfs_test_hourly_exp2.drop(
    columns=gfs_constant_cols_exp2
)

weather_train_exp2 = pd.merge(
    ldaps_train_hourly,
    gfs_train_hourly_exp2,
    on="forecast_kst_dtm",
    how="inner",
    validate="one_to_one",
)

weather_test_exp2 = pd.merge(
    ldaps_test_hourly,
    gfs_test_hourly_exp2,
    on="forecast_kst_dtm",
    how="inner",
    validate="one_to_one",
)

train_exp2 = pd.merge(
    weather_train_exp2,
    labels,
    on="forecast_kst_dtm",
    how="left",
    validate="one_to_one",
)

test_exp2 = pd.merge(
    sample_submission[
        [
            "forecast_id",
            "forecast_kst_dtm",
        ]
    ],
    weather_test_exp2,
    on="forecast_kst_dtm",
    how="left",
    validate="one_to_one",
)

train_exp2 = add_time_features(train_exp2)
test_exp2 = add_time_features(test_exp2)

exclude_cols = [
    "forecast_id",
    "forecast_kst_dtm",
    "kpx_group_1",
    "kpx_group_2",
    "kpx_group_3",
]

feature_cols_exp2 = [
    col
    for col in train_exp2.columns
    if col not in exclude_cols
]

test_exp2 = test_exp2[
    [
        "forecast_id",
        "forecast_kst_dtm",
    ]
    + feature_cols_exp2
]

train_group_1_exp2 = (
    train_exp2
    .dropna(subset=["kpx_group_1"])
    .reset_index(drop=True)
)

train_group_2_exp2 = (
    train_exp2
    .dropna(subset=["kpx_group_2"])
    .reset_index(drop=True)
)

train_group_3_exp2 = (
    train_exp2
    .dropna(subset=["kpx_group_3"])
    .reset_index(drop=True)
)

print("Exp2 feature 수:", len(feature_cols_exp2))
print(
    "Train 결측치:",
    train_exp2[feature_cols_exp2].isna().sum().sum(),
)
print(
    "Test 결측치:",
    test_exp2[feature_cols_exp2].isna().sum().sum(),
)

Exp2 feature 수: 305
Train 결측치: 0
Test 결측치: 0


In [23]:
def split_exp2(
    group_df,
    target_col,
):
    train_mask = (
        group_df["forecast_kst_dtm"]
        < valid_start
    )

    valid_mask = (
        group_df["forecast_kst_dtm"]
        >= valid_start
    )

    return (
        group_df.loc[
            train_mask,
            feature_cols_exp2,
        ],
        group_df.loc[
            valid_mask,
            feature_cols_exp2,
        ],
        group_df.loc[
            train_mask,
            target_col,
        ],
        group_df.loc[
            valid_mask,
            target_col,
        ],
    )


X_train_1_exp2, X_valid_1_exp2, y_train_1_exp2, y_valid_1_exp2 = split_exp2(
    train_group_1_exp2,
    "kpx_group_1",
)

X_train_2_exp2, X_valid_2_exp2, y_train_2_exp2, y_valid_2_exp2 = split_exp2(
    train_group_2_exp2,
    "kpx_group_2",
)

X_train_3_exp2, X_valid_3_exp2, y_train_3_exp2, y_valid_3_exp2 = split_exp2(
    train_group_3_exp2,
    "kpx_group_3",
)

(
    lgbm_exp2_1,
    pred_exp2_1,
    rmse_exp2_1,
    mae_exp2_1,
    r2_exp2_1,
) = train_lgbm_baseline(
    X_train_1_exp2,
    y_train_1_exp2,
    X_valid_1_exp2,
    y_valid_1_exp2,
    capacity=21600,
)

(
    lgbm_exp2_2,
    pred_exp2_2,
    rmse_exp2_2,
    mae_exp2_2,
    r2_exp2_2,
) = train_lgbm_baseline(
    X_train_2_exp2,
    y_train_2_exp2,
    X_valid_2_exp2,
    y_valid_2_exp2,
    capacity=21600,
)

(
    lgbm_exp2_3,
    pred_exp2_3,
    rmse_exp2_3,
    mae_exp2_3,
    r2_exp2_3,
) = train_lgbm_baseline(
    X_train_3_exp2,
    y_train_3_exp2,
    X_valid_3_exp2,
    y_valid_3_exp2,
    capacity=21000,
)

exp2_result = pd.DataFrame({
    "group": [
        "Group 1",
        "Group 2",
        "Group 3",
    ],
    "exp1_rmse": [
        2728.115640,
        3008.105780,
        2786.029266,
    ],
    "exp2_rmse": [
        rmse_exp2_1,
        rmse_exp2_2,
        rmse_exp2_3,
    ],
    "exp2_mae": [
        mae_exp2_1,
        mae_exp2_2,
        mae_exp2_3,
    ],
    "exp2_r2": [
        r2_exp2_1,
        r2_exp2_2,
        r2_exp2_3,
    ],
})

exp2_result["rmse_change"] = (
    exp2_result["exp2_rmse"]
    - exp2_result["exp1_rmse"]
)

print(exp2_result)

print(
    "\nExp1 평균 RMSE:",
    exp2_result["exp1_rmse"].mean(),
)

print(
    "Exp2 평균 RMSE:",
    exp2_result["exp2_rmse"].mean(),
)

     group    exp1_rmse    exp2_rmse     exp2_mae   exp2_r2  rmse_change
0  Group 1  2728.115640  2722.557371  1962.911553  0.855986    -5.558269
1  Group 2  3008.105780  3008.302098  2163.315556  0.845340     0.196318
2  Group 3  2786.029266  2775.754215  1893.386487  0.812750   -10.275051

Exp1 평균 RMSE: 2840.7502286666663
Exp2 평균 RMSE: 2835.537894691774


In [24]:
for df in [
    ldaps_train_raw,
    ldaps_test_raw,
]:
    ldaps_direction_10m = np.arctan2(
        df["heightAboveGround_10_10v"],
        df["heightAboveGround_10_10u"],
    )

    df["wind_direction_10m_sin"] = np.sin(
        ldaps_direction_10m
    )

    df["wind_direction_10m_cos"] = np.cos(
        ldaps_direction_10m
    )


for df in [
    gfs_train_raw,
    gfs_test_raw,
]:
    gfs_direction_10m = np.arctan2(
        df["heightAboveGround_10_10v"],
        df["heightAboveGround_10_10u"],
    )

    gfs_direction_80m = np.arctan2(
        df["heightAboveGround_80_v"],
        df["heightAboveGround_80_u"],
    )

    gfs_direction_100m = np.arctan2(
        df["heightAboveGround_100_100v"],
        df["heightAboveGround_100_100u"],
    )

    df["wind_direction_10m_sin"] = np.sin(
        gfs_direction_10m
    )

    df["wind_direction_10m_cos"] = np.cos(
        gfs_direction_10m
    )

    df["wind_direction_80m_sin"] = np.sin(
        gfs_direction_80m
    )

    df["wind_direction_80m_cos"] = np.cos(
        gfs_direction_80m
    )

    df["wind_direction_100m_sin"] = np.sin(
        gfs_direction_100m
    )

    df["wind_direction_100m_cos"] = np.cos(
        gfs_direction_100m
    )

In [25]:
# 1. LDAPS 재집계
ldaps_train_hourly_exp3 = aggregate_weather(
    ldaps_train_raw,
    prefix="ldaps",
)

ldaps_test_hourly_exp3 = aggregate_weather(
    ldaps_test_raw,
    prefix="ldaps",
)

# 2. GFS 재집계
gfs_train_hourly_exp3 = aggregate_weather(
    gfs_train_raw,
    prefix="gfs",
)

gfs_test_hourly_exp3 = aggregate_weather(
    gfs_test_raw,
    prefix="gfs",
)

# 3. LDAPS Test 결측 보간
ldaps_test_feature_cols_exp3 = [
    col
    for col in ldaps_test_hourly_exp3.columns
    if col != "forecast_kst_dtm"
]

ldaps_test_hourly_exp3[ldaps_test_feature_cols_exp3] = (
    ldaps_test_hourly_exp3[ldaps_test_feature_cols_exp3]
    .interpolate(
        method="linear",
        limit_direction="both",
    )
)

# 4. 상수 컬럼 제거
ldaps_constant_cols_exp3 = [
    col
    for col in ldaps_train_hourly_exp3.columns
    if col != "forecast_kst_dtm"
    and ldaps_train_hourly_exp3[col].nunique(dropna=False) <= 1
]

gfs_constant_cols_exp3 = [
    col
    for col in gfs_train_hourly_exp3.columns
    if col != "forecast_kst_dtm"
    and gfs_train_hourly_exp3[col].nunique(dropna=False) <= 1
]

ldaps_train_hourly_exp3 = (
    ldaps_train_hourly_exp3
    .drop(columns=ldaps_constant_cols_exp3)
)

ldaps_test_hourly_exp3 = (
    ldaps_test_hourly_exp3
    .drop(columns=ldaps_constant_cols_exp3)
)

gfs_train_hourly_exp3 = (
    gfs_train_hourly_exp3
    .drop(columns=gfs_constant_cols_exp3)
)

gfs_test_hourly_exp3 = (
    gfs_test_hourly_exp3
    .drop(columns=gfs_constant_cols_exp3)
)

# 5. LDAPS + GFS 병합
weather_train_exp3 = pd.merge(
    ldaps_train_hourly_exp3,
    gfs_train_hourly_exp3,
    on="forecast_kst_dtm",
    how="inner",
    validate="one_to_one",
)

weather_test_exp3 = pd.merge(
    ldaps_test_hourly_exp3,
    gfs_test_hourly_exp3,
    on="forecast_kst_dtm",
    how="inner",
    validate="one_to_one",
)

print("Weather Train:", weather_train_exp3.shape)
print("Weather Test:", weather_test_exp3.shape)
print("Train 결측치:", weather_train_exp3.isna().sum().sum())
print("Test 결측치:", weather_test_exp3.isna().sum().sum())

Weather Train: (26304, 325)
Weather Test: (8760, 325)
Train 결측치: 0
Test 결측치: 0


In [26]:
# 6. Label 결합
train_exp3 = pd.merge(
    weather_train_exp3,
    labels,
    on="forecast_kst_dtm",
    how="left",
    validate="one_to_one",
)

test_exp3 = pd.merge(
    sample_submission[
        [
            "forecast_id",
            "forecast_kst_dtm",
        ]
    ],
    weather_test_exp3,
    on="forecast_kst_dtm",
    how="left",
    validate="one_to_one",
)

# 7. 시간 파생변수 추가
train_exp3 = add_time_features(train_exp3)
test_exp3 = add_time_features(test_exp3)

# 8. Feature 목록 구성
exclude_cols = [
    "forecast_id",
    "forecast_kst_dtm",
    "kpx_group_1",
    "kpx_group_2",
    "kpx_group_3",
]

feature_cols_exp3 = [
    col
    for col in train_exp3.columns
    if col not in exclude_cols
]

test_exp3 = test_exp3[
    [
        "forecast_id",
        "forecast_kst_dtm",
    ]
    + feature_cols_exp3
]

# 9. 그룹별 데이터 구성
train_group_1_exp3 = (
    train_exp3
    .dropna(subset=["kpx_group_1"])
    .reset_index(drop=True)
)

train_group_2_exp3 = (
    train_exp3
    .dropna(subset=["kpx_group_2"])
    .reset_index(drop=True)
)

train_group_3_exp3 = (
    train_exp3
    .dropna(subset=["kpx_group_3"])
    .reset_index(drop=True)
)

print("Exp3 feature 수:", len(feature_cols_exp3))

print(
    "Train 결측치:",
    train_exp3[feature_cols_exp3]
    .isna()
    .sum()
    .sum()
)

print(
    "Test 결측치:",
    test_exp3[feature_cols_exp3]
    .isna()
    .sum()
    .sum()
)

print(
    "Train 무한값:",
    np.isinf(
        train_exp3[feature_cols_exp3].to_numpy()
    ).sum()
)

print(
    "Test 무한값:",
    np.isinf(
        test_exp3[feature_cols_exp3].to_numpy()
    ).sum()
)

Exp3 feature 수: 337
Train 결측치: 0
Test 결측치: 0
Train 무한값: 0
Test 무한값: 0


In [27]:
# 10. 시계열 분할 함수
def split_exp3(
    group_df,
    target_col,
):
    train_mask = (
        group_df["forecast_kst_dtm"]
        < valid_start
    )

    valid_mask = (
        group_df["forecast_kst_dtm"]
        >= valid_start
    )

    return (
        group_df.loc[
            train_mask,
            feature_cols_exp3,
        ],
        group_df.loc[
            valid_mask,
            feature_cols_exp3,
        ],
        group_df.loc[
            train_mask,
            target_col,
        ],
        group_df.loc[
            valid_mask,
            target_col,
        ],
    )


X_train_1_exp3, X_valid_1_exp3, y_train_1_exp3, y_valid_1_exp3 = split_exp3(
    train_group_1_exp3,
    "kpx_group_1",
)

X_train_2_exp3, X_valid_2_exp3, y_train_2_exp3, y_valid_2_exp3 = split_exp3(
    train_group_2_exp3,
    "kpx_group_2",
)

X_train_3_exp3, X_valid_3_exp3, y_train_3_exp3, y_valid_3_exp3 = split_exp3(
    train_group_3_exp3,
    "kpx_group_3",
)

# 11. LightGBM 학습
(
    lgbm_exp3_1,
    pred_exp3_1,
    rmse_exp3_1,
    mae_exp3_1,
    r2_exp3_1,
) = train_lgbm_baseline(
    X_train_1_exp3,
    y_train_1_exp3,
    X_valid_1_exp3,
    y_valid_1_exp3,
    capacity=21600,
)

(
    lgbm_exp3_2,
    pred_exp3_2,
    rmse_exp3_2,
    mae_exp3_2,
    r2_exp3_2,
) = train_lgbm_baseline(
    X_train_2_exp3,
    y_train_2_exp3,
    X_valid_2_exp3,
    y_valid_2_exp3,
    capacity=21600,
)

(
    lgbm_exp3_3,
    pred_exp3_3,
    rmse_exp3_3,
    mae_exp3_3,
    r2_exp3_3,
) = train_lgbm_baseline(
    X_train_3_exp3,
    y_train_3_exp3,
    X_valid_3_exp3,
    y_valid_3_exp3,
    capacity=21000,
)

# 12. Exp2와 비교
exp3_result = pd.DataFrame({
    "group": [
        "Group 1",
        "Group 2",
        "Group 3",
    ],
    "exp2_rmse": [
        2722.557371,
        3008.302098,
        2775.754215,
    ],
    "exp3_rmse": [
        rmse_exp3_1,
        rmse_exp3_2,
        rmse_exp3_3,
    ],
    "exp3_mae": [
        mae_exp3_1,
        mae_exp3_2,
        mae_exp3_3,
    ],
    "exp3_r2": [
        r2_exp3_1,
        r2_exp3_2,
        r2_exp3_3,
    ],
})

exp3_result["rmse_change"] = (
    exp3_result["exp3_rmse"]
    - exp3_result["exp2_rmse"]
)

print(exp3_result)

print(
    "\nExp2 평균 RMSE:",
    exp3_result["exp2_rmse"].mean()
)

print(
    "Exp3 평균 RMSE:",
    exp3_result["exp3_rmse"].mean()
)

     group    exp2_rmse    exp3_rmse     exp3_mae   exp3_r2  rmse_change
0  Group 1  2722.557371  2739.553287  1970.896578  0.854183    16.995916
1  Group 2  3008.302098  3101.365754  2232.505210  0.835623    93.063656
2  Group 3  2775.754215  2763.956815  1880.153069  0.814338   -11.797400

Exp2 평균 RMSE: 2835.5378946666665
Exp3 평균 RMSE: 2868.291952174865


In [28]:
ldaps_direction_cols = [
    "wind_direction_10m_sin",
    "wind_direction_10m_cos",
]

gfs_direction_cols = [
    "wind_direction_10m_sin",
    "wind_direction_10m_cos",
    "wind_direction_80m_sin",
    "wind_direction_80m_cos",
    "wind_direction_100m_sin",
    "wind_direction_100m_cos",
]

for df in [
    ldaps_train_raw,
    ldaps_test_raw,
]:
    df.drop(
        columns=[
            col
            for col in ldaps_direction_cols
            if col in df.columns
        ],
        inplace=True,
    )

for df in [
    gfs_train_raw,
    gfs_test_raw,
]:
    df.drop(
        columns=[
            col
            for col in gfs_direction_cols
            if col in df.columns
        ],
        inplace=True,
    )

In [29]:
def q25(x):
    return x.quantile(0.25)

def q50(x):
    return x.quantile(0.50)

def q75(x):
    return x.quantile(0.75)

q25.__name__ = "q25"
q50.__name__ = "q50"
q75.__name__ = "q75"

In [30]:
def aggregate_weather_with_wind_quantiles(
    df,
    prefix,
):
    exclude_cols = {
        "forecast_kst_dtm",
        "data_available_kst_dtm",
        "grid_id",
        "latitude",
        "longitude",
    }

    feature_cols = [
        col
        for col in df.columns
        if col not in exclude_cols
    ]

    wind_cols = [
        col
        for col in feature_cols
        if (
            "wind_speed" in col
            or "wind_shear" in col
            or "wind_power_density" in col
        )
    ]

    normal_cols = [
        col
        for col in feature_cols
        if col not in wind_cols
    ]

    normal_agg = (
        df
        .groupby("forecast_kst_dtm")[normal_cols]
        .agg([
            "mean",
            "std",
            "min",
            "max",
        ])
    )

    wind_agg = (
        df
        .groupby("forecast_kst_dtm")[wind_cols]
        .agg([
            "mean",
            "std",
            "min",
            "max",
            q25,
            q50,
            q75,
        ])
    )

    aggregated = pd.concat(
        [
            normal_agg,
            wind_agg,
        ],
        axis=1,
    )

    aggregated.columns = [
        f"{prefix}_{col}_{stat}"
        for col, stat in aggregated.columns
    ]

    aggregated = (
        aggregated
        .reset_index()
        .sort_values("forecast_kst_dtm")
        .reset_index(drop=True)
    )

    return aggregated

In [31]:
ldaps_train_hourly_exp4 = aggregate_weather_with_wind_quantiles(
    ldaps_train_raw,
    prefix="ldaps",
)

ldaps_test_hourly_exp4 = aggregate_weather_with_wind_quantiles(
    ldaps_test_raw,
    prefix="ldaps",
)

gfs_train_hourly_exp4 = aggregate_weather_with_wind_quantiles(
    gfs_train_raw,
    prefix="gfs",
)

gfs_test_hourly_exp4 = aggregate_weather_with_wind_quantiles(
    gfs_test_raw,
    prefix="gfs",
)

In [32]:
ldaps_test_feature_cols_exp4 = [
    col
    for col in ldaps_test_hourly_exp4.columns
    if col != "forecast_kst_dtm"
]

ldaps_test_hourly_exp4[ldaps_test_feature_cols_exp4] = (
    ldaps_test_hourly_exp4[ldaps_test_feature_cols_exp4]
    .interpolate(
        method="linear",
        limit_direction="both",
    )
)

In [33]:
ldaps_test_feature_cols_exp4 = [
    col
    for col in ldaps_test_hourly_exp4.columns
    if col != "forecast_kst_dtm"
]

ldaps_test_hourly_exp4[ldaps_test_feature_cols_exp4] = (
    ldaps_test_hourly_exp4[ldaps_test_feature_cols_exp4]
    .interpolate(
        method="linear",
        limit_direction="both",
    )
)

In [34]:
ldaps_constant_cols_exp4 = [
    col
    for col in ldaps_train_hourly_exp4.columns
    if col != "forecast_kst_dtm"
    and ldaps_train_hourly_exp4[col].nunique(dropna=False) <= 1
]

gfs_constant_cols_exp4 = [
    col
    for col in gfs_train_hourly_exp4.columns
    if col != "forecast_kst_dtm"
    and gfs_train_hourly_exp4[col].nunique(dropna=False) <= 1
]

ldaps_train_hourly_exp4 = ldaps_train_hourly_exp4.drop(
    columns=ldaps_constant_cols_exp4
)

ldaps_test_hourly_exp4 = ldaps_test_hourly_exp4.drop(
    columns=ldaps_constant_cols_exp4
)

gfs_train_hourly_exp4 = gfs_train_hourly_exp4.drop(
    columns=gfs_constant_cols_exp4
)

gfs_test_hourly_exp4 = gfs_test_hourly_exp4.drop(
    columns=gfs_constant_cols_exp4
)

In [35]:
weather_train_exp4 = pd.merge(
    ldaps_train_hourly_exp4,
    gfs_train_hourly_exp4,
    on="forecast_kst_dtm",
    how="inner",
    validate="one_to_one",
)

weather_test_exp4 = pd.merge(
    ldaps_test_hourly_exp4,
    gfs_test_hourly_exp4,
    on="forecast_kst_dtm",
    how="inner",
    validate="one_to_one",
)

In [36]:
train_exp4 = pd.merge(
    weather_train_exp4,
    labels,
    on="forecast_kst_dtm",
    how="left",
    validate="one_to_one",
)

test_exp4 = pd.merge(
    sample_submission[
        [
            "forecast_id",
            "forecast_kst_dtm",
        ]
    ],
    weather_test_exp4,
    on="forecast_kst_dtm",
    how="left",
    validate="one_to_one",
)

train_exp4 = add_time_features(train_exp4)
test_exp4 = add_time_features(test_exp4)

exclude_cols = [
    "forecast_id",
    "forecast_kst_dtm",
    "kpx_group_1",
    "kpx_group_2",
    "kpx_group_3",
]

feature_cols_exp4 = [
    col
    for col in train_exp4.columns
    if col not in exclude_cols
]

test_exp4 = test_exp4[
    [
        "forecast_id",
        "forecast_kst_dtm",
    ]
    + feature_cols_exp4
]

train_group_1_exp4 = (
    train_exp4
    .dropna(subset=["kpx_group_1"])
    .reset_index(drop=True)
)

train_group_2_exp4 = (
    train_exp4
    .dropna(subset=["kpx_group_2"])
    .reset_index(drop=True)
)

train_group_3_exp4 = (
    train_exp4
    .dropna(subset=["kpx_group_3"])
    .reset_index(drop=True)
)

print("Exp4 feature 수:", len(feature_cols_exp4))
print(
    "Train 결측치:",
    train_exp4[feature_cols_exp4].isna().sum().sum(),
)
print(
    "Test 결측치:",
    test_exp4[feature_cols_exp4].isna().sum().sum(),
)

Exp4 feature 수: 332
Train 결측치: 0
Test 결측치: 0


In [37]:
def split_exp4(
    group_df,
    target_col,
):
    train_mask = (
        group_df["forecast_kst_dtm"]
        < valid_start
    )

    valid_mask = (
        group_df["forecast_kst_dtm"]
        >= valid_start
    )

    return (
        group_df.loc[
            train_mask,
            feature_cols_exp4,
        ],
        group_df.loc[
            valid_mask,
            feature_cols_exp4,
        ],
        group_df.loc[
            train_mask,
            target_col,
        ],
        group_df.loc[
            valid_mask,
            target_col,
        ],
    )


X_train_1_exp4, X_valid_1_exp4, y_train_1_exp4, y_valid_1_exp4 = split_exp4(
    train_group_1_exp4,
    "kpx_group_1",
)

X_train_2_exp4, X_valid_2_exp4, y_train_2_exp4, y_valid_2_exp4 = split_exp4(
    train_group_2_exp4,
    "kpx_group_2",
)

X_train_3_exp4, X_valid_3_exp4, y_train_3_exp4, y_valid_3_exp4 = split_exp4(
    train_group_3_exp4,
    "kpx_group_3",
)

(
    lgbm_exp4_1,
    pred_exp4_1,
    rmse_exp4_1,
    mae_exp4_1,
    r2_exp4_1,
) = train_lgbm_baseline(
    X_train_1_exp4,
    y_train_1_exp4,
    X_valid_1_exp4,
    y_valid_1_exp4,
    capacity=21600,
)

(
    lgbm_exp4_2,
    pred_exp4_2,
    rmse_exp4_2,
    mae_exp4_2,
    r2_exp4_2,
) = train_lgbm_baseline(
    X_train_2_exp4,
    y_train_2_exp4,
    X_valid_2_exp4,
    y_valid_2_exp4,
    capacity=21600,
)

(
    lgbm_exp4_3,
    pred_exp4_3,
    rmse_exp4_3,
    mae_exp4_3,
    r2_exp4_3,
) = train_lgbm_baseline(
    X_train_3_exp4,
    y_train_3_exp4,
    X_valid_3_exp4,
    y_valid_3_exp4,
    capacity=21000,
)

In [38]:
exp4_result = pd.DataFrame({
    "group": [
        "Group 1",
        "Group 2",
        "Group 3",
    ],
    "exp2_rmse": [
        2722.557371,
        3008.302098,
        2775.754215,
    ],
    "exp4_rmse": [
        rmse_exp4_1,
        rmse_exp4_2,
        rmse_exp4_3,
    ],
    "exp4_mae": [
        mae_exp4_1,
        mae_exp4_2,
        mae_exp4_3,
    ],
    "exp4_r2": [
        r2_exp4_1,
        r2_exp4_2,
        r2_exp4_3,
    ],
})

exp4_result["rmse_change"] = (
    exp4_result["exp4_rmse"]
    - exp4_result["exp2_rmse"]
)

print(exp4_result)

print(
    "\nExp2 평균 RMSE:",
    exp4_result["exp2_rmse"].mean(),
)

print(
    "Exp4 평균 RMSE:",
    exp4_result["exp4_rmse"].mean(),
)

     group    exp2_rmse    exp4_rmse     exp4_mae   exp4_r2  rmse_change
0  Group 1  2722.557371  2716.469788  1958.929212  0.856630    -6.087583
1  Group 2  3008.302098  3072.124941  2197.835353  0.838708    63.822843
2  Group 3  2775.754215  2772.816772  1893.882340  0.813146    -2.937443

Exp2 평균 RMSE: 2835.5378946666665
Exp4 평균 RMSE: 2853.803833833033


In [39]:
def get_feature_importance(model, feature_cols):
    importance = pd.DataFrame({
        "feature": feature_cols,
        "importance": model.feature_importances_,
    })

    importance["importance_ratio"] = (
        importance["importance"]
        / importance["importance"].sum()
    )

    return (
        importance
        .sort_values("importance", ascending=False)
        .reset_index(drop=True)
    )


importance_exp2_1 = get_feature_importance(
    lgbm_exp2_1,
    feature_cols_exp2,
)

importance_exp2_2 = get_feature_importance(
    lgbm_exp2_2,
    feature_cols_exp2,
)

importance_exp2_3 = get_feature_importance(
    lgbm_exp2_3,
    feature_cols_exp2,
)


physical_keywords = [
    "wind_speed_diff",
    "wind_shear",
    "wind_speed_cubed",
    "air_density",
    "wind_power_density",
]

for group_name, importance_df in [
    ("Group 1", importance_exp2_1),
    ("Group 2", importance_exp2_2),
    ("Group 3", importance_exp2_3),
]:
    physical_importance = importance_df[
        importance_df["feature"].str.contains(
            "|".join(physical_keywords),
            regex=True,
        )
    ]

    print(f"\n{group_name} 물리 파생변수 중요도")
    print(
        physical_importance[
            [
                "feature",
                "importance",
                "importance_ratio",
            ]
        ].head(30)
    )

    print(f"\n{group_name} 전체 상위 20개")
    print(
        importance_df[
            [
                "feature",
                "importance",
                "importance_ratio",
            ]
        ].head(20)
    )


Group 1 물리 파생변수 중요도
                                feature  importance  importance_ratio
17            gfs_air_density_proxy_std          57          0.009048
62            gfs_air_density_proxy_max          30          0.004762
71    gfs_wind_speed_diff_100m_80m_mean          27          0.004286
73     gfs_wind_speed_diff_100m_80m_max          27          0.004286
90   gfs_wind_shear_alpha_100m_10m_mean          24          0.003810
124           gfs_air_density_proxy_min          17          0.002698
125          gfs_air_density_proxy_mean          17          0.002698
138   gfs_wind_shear_alpha_100m_10m_min          15          0.002381
140   gfs_wind_shear_alpha_100m_10m_max          15          0.002381
144   gfs_wind_speed_diff_100m_10m_mean          14          0.002222
160    gfs_wind_speed_diff_100m_10m_std          13          0.002063
165    gfs_wind_speed_diff_100m_80m_std          13          0.002063
180    gfs_wind_speed_diff_100m_10m_max          11          0.001746

In [40]:
drop_physical_keywords = [
    "wind_speed_cubed_100m",
    "wind_power_density_100m",
]

feature_cols_exp5 = [
    col
    for col in feature_cols_exp2
    if not any(
        keyword in col
        for keyword in drop_physical_keywords
    )
]

removed_features = sorted(
    set(feature_cols_exp2)
    - set(feature_cols_exp5)
)

print("Exp2 feature 수:", len(feature_cols_exp2))
print("Exp5 feature 수:", len(feature_cols_exp5))
print("제거한 feature 수:", len(removed_features))

print("\n제거한 feature")
for col in removed_features:
    print(col)

Exp2 feature 수: 305
Exp5 feature 수: 297
제거한 feature 수: 8

제거한 feature
gfs_wind_power_density_100m_max
gfs_wind_power_density_100m_mean
gfs_wind_power_density_100m_min
gfs_wind_power_density_100m_std
gfs_wind_speed_cubed_100m_max
gfs_wind_speed_cubed_100m_mean
gfs_wind_speed_cubed_100m_min
gfs_wind_speed_cubed_100m_std


In [41]:
def split_exp5(group_df, target_col):
    train_mask = (
        group_df["forecast_kst_dtm"]
        < valid_start
    )

    valid_mask = (
        group_df["forecast_kst_dtm"]
        >= valid_start
    )

    return (
        group_df.loc[
            train_mask,
            feature_cols_exp5,
        ],
        group_df.loc[
            valid_mask,
            feature_cols_exp5,
        ],
        group_df.loc[
            train_mask,
            target_col,
        ],
        group_df.loc[
            valid_mask,
            target_col,
        ],
    )


X_train_1_exp5, X_valid_1_exp5, y_train_1_exp5, y_valid_1_exp5 = split_exp5(
    train_group_1_exp2,
    "kpx_group_1",
)

X_train_2_exp5, X_valid_2_exp5, y_train_2_exp5, y_valid_2_exp5 = split_exp5(
    train_group_2_exp2,
    "kpx_group_2",
)

X_train_3_exp5, X_valid_3_exp5, y_train_3_exp5, y_valid_3_exp5 = split_exp5(
    train_group_3_exp2,
    "kpx_group_3",
)

(
    lgbm_exp5_1,
    pred_exp5_1,
    rmse_exp5_1,
    mae_exp5_1,
    r2_exp5_1,
) = train_lgbm_baseline(
    X_train_1_exp5,
    y_train_1_exp5,
    X_valid_1_exp5,
    y_valid_1_exp5,
    capacity=21600,
)

(
    lgbm_exp5_2,
    pred_exp5_2,
    rmse_exp5_2,
    mae_exp5_2,
    r2_exp5_2,
) = train_lgbm_baseline(
    X_train_2_exp5,
    y_train_2_exp5,
    X_valid_2_exp5,
    y_valid_2_exp5,
    capacity=21600,
)

(
    lgbm_exp5_3,
    pred_exp5_3,
    rmse_exp5_3,
    mae_exp5_3,
    r2_exp5_3,
) = train_lgbm_baseline(
    X_train_3_exp5,
    y_train_3_exp5,
    X_valid_3_exp5,
    y_valid_3_exp5,
    capacity=21000,
)

In [42]:
exp5_result = pd.DataFrame({
    "group": [
        "Group 1",
        "Group 2",
        "Group 3",
    ],
    "exp2_rmse": [
        2722.557371,
        3008.302098,
        2775.754215,
    ],
    "exp5_rmse": [
        rmse_exp5_1,
        rmse_exp5_2,
        rmse_exp5_3,
    ],
    "exp5_mae": [
        mae_exp5_1,
        mae_exp5_2,
        mae_exp5_3,
    ],
    "exp5_r2": [
        r2_exp5_1,
        r2_exp5_2,
        r2_exp5_3,
    ],
})

exp5_result["rmse_change"] = (
    exp5_result["exp5_rmse"]
    - exp5_result["exp2_rmse"]
)

print(exp5_result)

print(
    "\nExp2 평균 RMSE:",
    exp5_result["exp2_rmse"].mean(),
)

print(
    "Exp5 평균 RMSE:",
    exp5_result["exp5_rmse"].mean(),
)

     group    exp2_rmse    exp5_rmse     exp5_mae   exp5_r2  rmse_change
0  Group 1  2722.557371  2734.213911  1961.274228  0.854751    11.656540
1  Group 2  3008.302098  3004.908695  2191.098151  0.845689    -3.393403
2  Group 3  2775.754215  2784.012511  1887.319171  0.811634     8.258296

Exp2 평균 RMSE: 2835.5378946666665
Exp5 평균 RMSE: 2841.0450387173064


In [43]:
importance_compare = pd.DataFrame({
    "feature": feature_cols_exp2,
    "group_1": lgbm_exp2_1.feature_importances_,
    "group_2": lgbm_exp2_2.feature_importances_,
    "group_3": lgbm_exp2_3.feature_importances_,
})

zero_importance_cols = importance_compare.loc[
    (importance_compare["group_1"] == 0)
    & (importance_compare["group_2"] == 0)
    & (importance_compare["group_3"] == 0),
    "feature",
].tolist()

print("Exp2 전체 feature 수:", len(feature_cols_exp2))
print("세 그룹 모두 중요도 0:", len(zero_importance_cols))

print("\n제거 후보")
for col in zero_importance_cols:
    print(col)

Exp2 전체 feature 수: 305
세 그룹 모두 중요도 0: 0

제거 후보


In [44]:
feature_cols_exp6 = [
    col
    for col in feature_cols_exp2
    if col not in zero_importance_cols
]

print("Exp6 feature 수:", len(feature_cols_exp6))
print(
    "제거된 feature 수:",
    len(feature_cols_exp2) - len(feature_cols_exp6),
)

Exp6 feature 수: 305
제거된 feature 수: 0


In [45]:
def split_exp6(group_df, target_col):
    train_mask = (
        group_df["forecast_kst_dtm"]
        < valid_start
    )

    valid_mask = (
        group_df["forecast_kst_dtm"]
        >= valid_start
    )

    return (
        group_df.loc[
            train_mask,
            feature_cols_exp6,
        ],
        group_df.loc[
            valid_mask,
            feature_cols_exp6,
        ],
        group_df.loc[
            train_mask,
            target_col,
        ],
        group_df.loc[
            valid_mask,
            target_col,
        ],
    )


X_train_1_exp6, X_valid_1_exp6, y_train_1_exp6, y_valid_1_exp6 = split_exp6(
    train_group_1_exp2,
    "kpx_group_1",
)

X_train_2_exp6, X_valid_2_exp6, y_train_2_exp6, y_valid_2_exp6 = split_exp6(
    train_group_2_exp2,
    "kpx_group_2",
)

X_train_3_exp6, X_valid_3_exp6, y_train_3_exp6, y_valid_3_exp6 = split_exp6(
    train_group_3_exp2,
    "kpx_group_3",
)

In [46]:
(
    lgbm_exp6_1,
    pred_exp6_1,
    rmse_exp6_1,
    mae_exp6_1,
    r2_exp6_1,
) = train_lgbm_baseline(
    X_train_1_exp6,
    y_train_1_exp6,
    X_valid_1_exp6,
    y_valid_1_exp6,
    capacity=21600,
)

(
    lgbm_exp6_2,
    pred_exp6_2,
    rmse_exp6_2,
    mae_exp6_2,
    r2_exp6_2,
) = train_lgbm_baseline(
    X_train_2_exp6,
    y_train_2_exp6,
    X_valid_2_exp6,
    y_valid_2_exp6,
    capacity=21600,
)

(
    lgbm_exp6_3,
    pred_exp6_3,
    rmse_exp6_3,
    mae_exp6_3,
    r2_exp6_3,
) = train_lgbm_baseline(
    X_train_3_exp6,
    y_train_3_exp6,
    X_valid_3_exp6,
    y_valid_3_exp6,
    capacity=21000,
)

In [47]:
exp6_result = pd.DataFrame({
    "group": [
        "Group 1",
        "Group 2",
        "Group 3",
    ],
    "exp2_rmse": [
        2722.557371,
        3008.302098,
        2775.754215,
    ],
    "exp6_rmse": [
        rmse_exp6_1,
        rmse_exp6_2,
        rmse_exp6_3,
    ],
    "exp6_mae": [
        mae_exp6_1,
        mae_exp6_2,
        mae_exp6_3,
    ],
    "exp6_r2": [
        r2_exp6_1,
        r2_exp6_2,
        r2_exp6_3,
    ],
})

exp6_result["rmse_change"] = (
    exp6_result["exp6_rmse"]
    - exp6_result["exp2_rmse"]
)

print(exp6_result)

print(
    "\nExp2 feature 수:",
    len(feature_cols_exp2),
)

print(
    "Exp6 feature 수:",
    len(feature_cols_exp6),
)

print(
    "\nExp2 평균 RMSE:",
    exp6_result["exp2_rmse"].mean(),
)

print(
    "Exp6 평균 RMSE:",
    exp6_result["exp6_rmse"].mean(),
)

     group    exp2_rmse    exp6_rmse     exp6_mae   exp6_r2   rmse_change
0  Group 1  2722.557371  2722.557371  1962.911553  0.855986 -2.573065e-07
1  Group 2  3008.302098  3008.302098  2163.315556  0.845340  3.880928e-07
2  Group 3  2775.754215  2775.754215  1893.386487  0.812750 -5.546326e-08

Exp2 feature 수: 305
Exp6 feature 수: 305

Exp2 평균 RMSE: 2835.5378946666665
Exp6 평균 RMSE: 2835.537894691774


In [48]:
cv_folds = [
    {
        "name": "fold_1",
        "train_end": pd.Timestamp("2024-04-01 01:00:00"),
        "valid_end": pd.Timestamp("2024-07-01 01:00:00"),
    },
    {
        "name": "fold_2",
        "train_end": pd.Timestamp("2024-07-01 01:00:00"),
        "valid_end": pd.Timestamp("2024-10-01 01:00:00"),
    },
    {
        "name": "fold_3",
        "train_end": pd.Timestamp("2024-10-01 01:00:00"),
        "valid_end": pd.Timestamp("2025-01-01 01:00:00"),
    },
]

In [49]:
def evaluate_lgbm_time_cv(
    group_df,
    target_col,
    feature_cols,
    capacity,
):
    fold_results = []

    for fold in cv_folds:
        train_mask = (
            group_df["forecast_kst_dtm"]
            < fold["train_end"]
        )

        valid_mask = (
            (group_df["forecast_kst_dtm"] >= fold["train_end"])
            & (group_df["forecast_kst_dtm"] < fold["valid_end"])
        )

        X_train = group_df.loc[
            train_mask,
            feature_cols,
        ]

        y_train = group_df.loc[
            train_mask,
            target_col,
        ]

        X_valid = group_df.loc[
            valid_mask,
            feature_cols,
        ]

        y_valid = group_df.loc[
            valid_mask,
            target_col,
        ]

        model, pred, rmse, mae, r2 = train_lgbm_baseline(
            X_train,
            y_train,
            X_valid,
            y_valid,
            capacity=capacity,
        )

        fold_results.append({
            "fold": fold["name"],
            "train_rows": len(X_train),
            "valid_rows": len(X_valid),
            "best_iteration": model.best_iteration_,
            "rmse": rmse,
            "mae": mae,
            "r2": r2,
        })

    return pd.DataFrame(fold_results)

In [50]:
cv_result_1 = evaluate_lgbm_time_cv(
    train_group_1_exp2,
    "kpx_group_1",
    feature_cols_exp2,
    capacity=21600,
)

cv_result_2 = evaluate_lgbm_time_cv(
    train_group_2_exp2,
    "kpx_group_2",
    feature_cols_exp2,
    capacity=21600,
)

cv_result_3 = evaluate_lgbm_time_cv(
    train_group_3_exp2,
    "kpx_group_3",
    feature_cols_exp2,
    capacity=21000,
)

In [51]:
cv_result_1["group"] = "Group 1"
cv_result_2["group"] = "Group 2"
cv_result_3["group"] = "Group 3"

cv_results = pd.concat(
    [
        cv_result_1,
        cv_result_2,
        cv_result_3,
    ],
    ignore_index=True,
)

print(cv_results)

cv_summary = (
    cv_results
    .groupby("group")
    .agg(
        mean_rmse=("rmse", "mean"),
        std_rmse=("rmse", "std"),
        mean_mae=("mae", "mean"),
        mean_r2=("r2", "mean"),
        mean_best_iteration=("best_iteration", "mean"),
    )
    .reset_index()
)

print("\n그룹별 CV 요약")
print(cv_summary)

print(
    "\n전체 평균 RMSE:",
    cv_summary["mean_rmse"].mean(),
)

     fold  train_rows  valid_rows  best_iteration         rmse          mae  \
0  fold_1       19606        2184             169  2721.752152  1918.550894   
1  fold_2       21790        2202             223  2670.862016  1717.357929   
2  fold_3       23992        2208             210  2722.557371  1962.911553   
3  fold_1       19607        2184             201  2603.120394  1823.058449   
4  fold_2       21791        2202             274  2434.972033  1549.268583   
5  fold_3       23993        2208             316  3008.302098  2163.315556   
6  fold_1       10944        2184             272  2763.663970  1843.283525   
7  fold_2       13128        2202             193  2750.497526  1717.813945   
8  fold_3       15330        2208             137  2775.754215  1893.386487   

         r2    group  
0  0.780813  Group 1  
1  0.798606  Group 1  
2  0.855986  Group 1  
3  0.796869  Group 2  
4  0.846524  Group 2  
5  0.845340  Group 2  
6  0.752726  Group 3  
7  0.808841  Group 3  
8 

In [52]:
baseline_cv_result_1 = evaluate_lgbm_time_cv(
    train_group_1,
    "kpx_group_1",
    model_feature_cols,
    capacity=21600,
)

baseline_cv_result_2 = evaluate_lgbm_time_cv(
    train_group_2,
    "kpx_group_2",
    model_feature_cols,
    capacity=21600,
)

baseline_cv_result_3 = evaluate_lgbm_time_cv(
    train_group_3,
    "kpx_group_3",
    model_feature_cols,
    capacity=21000,
)

baseline_cv_result_1["group"] = "Group 1"
baseline_cv_result_2["group"] = "Group 2"
baseline_cv_result_3["group"] = "Group 3"

baseline_cv_results = pd.concat(
    [
        baseline_cv_result_1,
        baseline_cv_result_2,
        baseline_cv_result_3,
    ],
    ignore_index=True,
)

baseline_cv_summary = (
    baseline_cv_results
    .groupby("group")
    .agg(
        baseline_mean_rmse=("rmse", "mean"),
        baseline_std_rmse=("rmse", "std"),
    )
    .reset_index()
)

exp2_cv_summary = (
    cv_results
    .groupby("group")
    .agg(
        exp2_mean_rmse=("rmse", "mean"),
        exp2_std_rmse=("rmse", "std"),
    )
    .reset_index()
)

cv_comparison = pd.merge(
    baseline_cv_summary,
    exp2_cv_summary,
    on="group",
)

cv_comparison["rmse_change"] = (
    cv_comparison["exp2_mean_rmse"]
    - cv_comparison["baseline_mean_rmse"]
)

print(cv_comparison)

print(
    "\nBaseline 전체 평균 RMSE:",
    cv_comparison["baseline_mean_rmse"].mean(),
)

print(
    "Exp2 전체 평균 RMSE:",
    cv_comparison["exp2_mean_rmse"].mean(),
)

     group  baseline_mean_rmse  baseline_std_rmse  exp2_mean_rmse  \
0  Group 1         2712.150314          28.588395     2705.057180   
1  Group 2         2691.787407         337.079377     2682.131508   
2  Group 3         2765.828659          14.347881     2763.305237   

   exp2_std_rmse  rmse_change  
0      29.616617    -7.093134  
1     294.718354    -9.655899  
2      12.632165    -2.523422  

Baseline 전체 평균 RMSE: 2723.255459893499
Exp2 전체 평균 RMSE: 2716.8313083514736


In [53]:
final_dir = Path("processed_final")
final_dir.mkdir(exist_ok=True)

train_exp2.to_parquet(
    final_dir / "train_preprocessed.parquet",
    index=False,
)

test_exp2.to_parquet(
    final_dir / "test_preprocessed.parquet",
    index=False,
)

train_group_1_exp2.to_parquet(
    final_dir / "train_group_1.parquet",
    index=False,
)

train_group_2_exp2.to_parquet(
    final_dir / "train_group_2.parquet",
    index=False,
)

train_group_3_exp2.to_parquet(
    final_dir / "train_group_3.parquet",
    index=False,
)

pd.Series(
    feature_cols_exp2,
    name="feature",
).to_csv(
    final_dir / "model_feature_cols.csv",
    index=False,
)

cv_results.to_csv(
    final_dir / "exp2_cv_results.csv",
    index=False,
)

cv_comparison.to_csv(
    final_dir / "baseline_vs_exp2_cv.csv",
    index=False,
)

print("최종 전처리 저장 완료")
print("최종 feature 수:", len(feature_cols_exp2))

최종 전처리 저장 완료
최종 feature 수: 305


In [54]:
capacities = {
    "kpx_group_1": 21600,
    "kpx_group_2": 21600,
    "kpx_group_3": 21000,
}


def calculate_group_metric(
    y_true,
    y_pred,
    capacity,
):
    y_true = np.asarray(
        y_true,
        dtype=float,
    )

    y_pred = np.asarray(
        y_pred,
        dtype=float,
    )

    valid_mask = (
        np.isfinite(y_true)
        & np.isfinite(y_pred)
        & (y_true >= capacity * 0.1)
    )

    y_true_eval = y_true[valid_mask]
    y_pred_eval = y_pred[valid_mask]

    if len(y_true_eval) == 0:
        return {
            "valid_count": 0,
            "nmae": np.nan,
            "one_minus_nmae": np.nan,
            "scr": np.nan,
            "ficr": np.nan,
        }

    hourly_nmae = (
        np.abs(y_pred_eval - y_true_eval)
        / capacity
    )

    group_nmae = hourly_nmae.mean()

    settlement_rate = np.select(
        [
            hourly_nmae <= 0.06,
            hourly_nmae <= 0.08,
        ],
        [
            4.0,
            3.0,
        ],
        default=0.0,
    )

    earned_settlement = np.sum(
        y_true_eval * settlement_rate
    )

    maximum_settlement = np.sum(
        y_true_eval * 4.0
    )

    scr = (
        earned_settlement
        / maximum_settlement
        if maximum_settlement > 0
        else np.nan
    )

    return {
        "valid_count": len(y_true_eval),
        "nmae": group_nmae,
        "one_minus_nmae": 1 - group_nmae,
        "scr": scr,
        "ficr": scr,
    }

In [55]:
def calculate_competition_score(
    y_true_list,
    y_pred_list,
    capacities_list,
):
    group_results = []

    for group_index, (
        y_true,
        y_pred,
        capacity,
    ) in enumerate(
        zip(
            y_true_list,
            y_pred_list,
            capacities_list,
        ),
        start=1,
    ):
        result = calculate_group_metric(
            y_true=y_true,
            y_pred=y_pred,
            capacity=capacity,
        )

        result["group"] = f"Group {group_index}"
        group_results.append(result)

    result_df = pd.DataFrame(group_results)

    mean_nmae = result_df["nmae"].mean()
    one_minus_nmae = 1 - mean_nmae
    ficr = result_df["scr"].mean()

    final_score = (
        0.5 * one_minus_nmae
        + 0.5 * ficr
    )

    summary = {
        "mean_nmae": mean_nmae,
        "one_minus_nmae": one_minus_nmae,
        "ficr": ficr,
        "score": final_score,
    }

    return result_df, summary


In [56]:
exp2_group_metrics, exp2_score = (
    calculate_competition_score(
        y_true_list=[
            y_valid_1_exp2,
            y_valid_2_exp2,
            y_valid_3_exp2,
        ],
        y_pred_list=[
            pred_exp2_1,
            pred_exp2_2,
            pred_exp2_3,
        ],
        capacities_list=[
            21600,
            21600,
            21000,
        ],
    )
)

print(exp2_group_metrics)

print("\nExp2 공식 지표")
print(
    "평균 NMAE:",
    exp2_score["mean_nmae"],
)
print(
    "1-NMAE:",
    exp2_score["one_minus_nmae"],
)
print(
    "FICR:",
    exp2_score["ficr"],
)
print(
    "최종 Score:",
    exp2_score["score"],
)

   valid_count      nmae  one_minus_nmae       scr      ficr    group
0         1371  0.113893        0.886107  0.365887  0.365887  Group 1
1         1329  0.132654        0.867346  0.272812  0.272812  Group 2
2         1196  0.131271        0.868729  0.341674  0.341674  Group 3

Exp2 공식 지표
평균 NMAE: 0.12593931638581587
1-NMAE: 0.8740606836141841
FICR: 0.326791095412142
최종 Score: 0.6004258895131631


In [57]:
def calculate_settlement_distribution(
    y_true,
    y_pred,
    capacity,
):
    y_true = np.asarray(
        y_true,
        dtype=float,
    )

    y_pred = np.asarray(
        y_pred,
        dtype=float,
    )

    valid_mask = (
        np.isfinite(y_true)
        & np.isfinite(y_pred)
        & (y_true >= capacity * 0.1)
    )

    y_true_eval = y_true[valid_mask]
    y_pred_eval = y_pred[valid_mask]

    hourly_nmae = (
        np.abs(y_pred_eval - y_true_eval)
        / capacity
    )

    count_4won = np.sum(
        hourly_nmae <= 0.06
    )

    count_3won = np.sum(
        (hourly_nmae > 0.06)
        & (hourly_nmae <= 0.08)
    )

    count_0won = np.sum(
        hourly_nmae > 0.08
    )

    total = len(hourly_nmae)

    return {
        "valid_count": total,
        "4won_count": count_4won,
        "3won_count": count_3won,
        "0won_count": count_0won,
        "4won_ratio": count_4won / total,
        "3won_ratio": count_3won / total,
        "0won_ratio": count_0won / total,
    }


settlement_results = []

for group_name, y_true, y_pred, capacity in [
    (
        "Group 1",
        y_valid_1_exp2,
        pred_exp2_1,
        21600,
    ),
    (
        "Group 2",
        y_valid_2_exp2,
        pred_exp2_2,
        21600,
    ),
    (
        "Group 3",
        y_valid_3_exp2,
        pred_exp2_3,
        21000,
    ),
]:
    result = calculate_settlement_distribution(
        y_true,
        y_pred,
        capacity,
    )

    result["group"] = group_name
    settlement_results.append(result)

settlement_df = pd.DataFrame(
    settlement_results
)

print(settlement_df)

   valid_count  4won_count  3won_count  0won_count  4won_ratio  3won_ratio  \
0         1371         441         157         773    0.321663    0.114515   
1         1329         374         103         852    0.281415    0.077502   
2         1196         349         131         716    0.291806    0.109532   

   0won_ratio    group  
0    0.563822  Group 1  
1    0.641084  Group 2  
2    0.598662  Group 3  


In [58]:
def train_xgb_baseline(
    X_train,
    y_train,
    X_valid,
    y_valid,
    capacity,
):
    model = XGBRegressor(
        objective="reg:squarederror",
        n_estimators=3000,
        learning_rate=0.03,
        max_depth=6,
        min_child_weight=5,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=1.0,
        random_state=42,
        n_jobs=-1,
        early_stopping_rounds=100,
    )

    model.fit(
        X_train,
        y_train,
        eval_set=[
            (X_valid, y_valid),
        ],
        verbose=False,
    )

    pred = model.predict(X_valid)
    pred = np.clip(pred, 0, capacity)

    return model, pred


def train_rf_baseline(
    X_train,
    y_train,
    X_valid,
    capacity,
):
    model = RandomForestRegressor(
        n_estimators=400,
        max_depth=None,
        min_samples_split=4,
        min_samples_leaf=2,
        max_features=0.7,
        random_state=42,
        n_jobs=-1,
    )

    model.fit(
        X_train,
        y_train,
    )

    pred = model.predict(X_valid)
    pred = np.clip(pred, 0, capacity)

    return model, pred

In [59]:
xgb_exp2_1, pred_xgb_exp2_1 = train_xgb_baseline(
    X_train_1_exp2,
    y_train_1_exp2,
    X_valid_1_exp2,
    y_valid_1_exp2,
    capacity=21600,
)

xgb_exp2_2, pred_xgb_exp2_2 = train_xgb_baseline(
    X_train_2_exp2,
    y_train_2_exp2,
    X_valid_2_exp2,
    y_valid_2_exp2,
    capacity=21600,
)

xgb_exp2_3, pred_xgb_exp2_3 = train_xgb_baseline(
    X_train_3_exp2,
    y_train_3_exp2,
    X_valid_3_exp2,
    y_valid_3_exp2,
    capacity=21000,
)


rf_exp2_1, pred_rf_exp2_1 = train_rf_baseline(
    X_train_1_exp2,
    y_train_1_exp2,
    X_valid_1_exp2,
    capacity=21600,
)

rf_exp2_2, pred_rf_exp2_2 = train_rf_baseline(
    X_train_2_exp2,
    y_train_2_exp2,
    X_valid_2_exp2,
    capacity=21600,
)

rf_exp2_3, pred_rf_exp2_3 = train_rf_baseline(
    X_train_3_exp2,
    y_train_3_exp2,
    X_valid_3_exp2,
    capacity=21000,
)

In [60]:
model_predictions = {
    "LightGBM": [
        pred_exp2_1,
        pred_exp2_2,
        pred_exp2_3,
    ],
    "XGBoost": [
        pred_xgb_exp2_1,
        pred_xgb_exp2_2,
        pred_xgb_exp2_3,
    ],
    "RandomForest": [
        pred_rf_exp2_1,
        pred_rf_exp2_2,
        pred_rf_exp2_3,
    ],
}

y_valid_list = [
    y_valid_1_exp2,
    y_valid_2_exp2,
    y_valid_3_exp2,
]

capacity_list = [
    21600,
    21600,
    21000,
]

single_model_results = []

for model_name, predictions in model_predictions.items():
    _, score_result = calculate_competition_score(
        y_true_list=y_valid_list,
        y_pred_list=predictions,
        capacities_list=capacity_list,
    )

    single_model_results.append({
        "model": model_name,
        "one_minus_nmae": score_result["one_minus_nmae"],
        "ficr": score_result["ficr"],
        "score": score_result["score"],
    })

single_model_results = pd.DataFrame(
    single_model_results
).sort_values(
    "score",
    ascending=False,
)

print(single_model_results)

          model  one_minus_nmae      ficr     score
0      LightGBM        0.874061  0.326791  0.600426
1       XGBoost        0.872560  0.319074  0.595817
2  RandomForest        0.872693  0.312864  0.592778


In [61]:
def optimize_group_ensemble(
    y_true,
    pred_rf,
    pred_xgb,
    pred_lgbm,
    capacity,
    step=0.05,
):
    best_result = None
    results = []

    weights = np.arange(
        0,
        1 + step / 2,
        step,
    )

    for weight_rf in weights:
        for weight_xgb in weights:
            weight_lgbm = (
                1
                - weight_rf
                - weight_xgb
            )

            if weight_lgbm < -1e-9:
                continue

            weight_lgbm = max(
                weight_lgbm,
                0,
            )

            ensemble_pred = (
                weight_rf * pred_rf
                + weight_xgb * pred_xgb
                + weight_lgbm * pred_lgbm
            )

            ensemble_pred = np.clip(
                ensemble_pred,
                0,
                capacity,
            )

            metric = calculate_group_metric(
                y_true,
                ensemble_pred,
                capacity,
            )

            group_score = (
                0.5 * metric["one_minus_nmae"]
                + 0.5 * metric["scr"]
            )

            result = {
                "weight_rf": weight_rf,
                "weight_xgb": weight_xgb,
                "weight_lgbm": weight_lgbm,
                "nmae": metric["nmae"],
                "one_minus_nmae": metric["one_minus_nmae"],
                "ficr": metric["scr"],
                "score": group_score,
            }

            results.append(result)

            if (
                best_result is None
                or group_score > best_result["score"]
            ):
                best_result = result

    return (
        best_result,
        pd.DataFrame(results).sort_values(
            "score",
            ascending=False,
        ),
    )

In [62]:
best_weight_1, ensemble_search_1 = optimize_group_ensemble(
    y_true=y_valid_1_exp2,
    pred_rf=pred_rf_exp2_1,
    pred_xgb=pred_xgb_exp2_1,
    pred_lgbm=pred_exp2_1,
    capacity=21600,
)

best_weight_2, ensemble_search_2 = optimize_group_ensemble(
    y_true=y_valid_2_exp2,
    pred_rf=pred_rf_exp2_2,
    pred_xgb=pred_xgb_exp2_2,
    pred_lgbm=pred_exp2_2,
    capacity=21600,
)

best_weight_3, ensemble_search_3 = optimize_group_ensemble(
    y_true=y_valid_3_exp2,
    pred_rf=pred_rf_exp2_3,
    pred_xgb=pred_xgb_exp2_3,
    pred_lgbm=pred_exp2_3,
    capacity=21000,
)

print("Group 1:", best_weight_1)
print("Group 2:", best_weight_2)
print("Group 3:", best_weight_3)

Group 1: {'weight_rf': np.float64(0.0), 'weight_xgb': np.float64(0.0), 'weight_lgbm': np.float64(1.0), 'nmae': np.float64(0.11389292232217045), 'one_minus_nmae': np.float64(0.8861070776778296), 'ficr': np.float64(0.36588730695721294), 'score': np.float64(0.6259971923175213)}
Group 2: {'weight_rf': np.float64(1.0), 'weight_xgb': np.float64(0.0), 'weight_lgbm': np.float64(0.0), 'nmae': np.float64(0.1299276688588444), 'one_minus_nmae': np.float64(0.8700723311411556), 'ficr': np.float64(0.2929811165246568), 'score': np.float64(0.5815267238329062)}
Group 3: {'weight_rf': np.float64(0.0), 'weight_xgb': np.float64(0.0), 'weight_lgbm': np.float64(1.0), 'nmae': np.float64(0.1312713544345602), 'one_minus_nmae': np.float64(0.8687286455654398), 'ficr': np.float64(0.3416741130820215), 'score': np.float64(0.6052013793237306)}


In [63]:
def make_weighted_prediction(
    pred_rf,
    pred_xgb,
    pred_lgbm,
    weights,
    capacity,
):
    pred = (
        weights["weight_rf"] * pred_rf
        + weights["weight_xgb"] * pred_xgb
        + weights["weight_lgbm"] * pred_lgbm
    )

    return np.clip(
        pred,
        0,
        capacity,
    )


pred_ensemble_1 = make_weighted_prediction(
    pred_rf_exp2_1,
    pred_xgb_exp2_1,
    pred_exp2_1,
    best_weight_1,
    capacity=21600,
)

pred_ensemble_2 = make_weighted_prediction(
    pred_rf_exp2_2,
    pred_xgb_exp2_2,
    pred_exp2_2,
    best_weight_2,
    capacity=21600,
)

pred_ensemble_3 = make_weighted_prediction(
    pred_rf_exp2_3,
    pred_xgb_exp2_3,
    pred_exp2_3,
    best_weight_3,
    capacity=21000,
)

ensemble_group_metrics, ensemble_score = (
    calculate_competition_score(
        y_true_list=[
            y_valid_1_exp2,
            y_valid_2_exp2,
            y_valid_3_exp2,
        ],
        y_pred_list=[
            pred_ensemble_1,
            pred_ensemble_2,
            pred_ensemble_3,
        ],
        capacities_list=[
            21600,
            21600,
            21000,
        ],
    )
)

print(ensemble_group_metrics)

print("\n앙상블 공식 지표")
print("1-NMAE:", ensemble_score["one_minus_nmae"])
print("FICR:", ensemble_score["ficr"])
print("Score:", ensemble_score["score"])

   valid_count      nmae  one_minus_nmae       scr      ficr    group
0         1371  0.113893        0.886107  0.365887  0.365887  Group 1
1         1329  0.129928        0.870072  0.292981  0.292981  Group 2
2         1196  0.131271        0.868729  0.341674  0.341674  Group 3

앙상블 공식 지표
1-NMAE: 0.874969351461475
FICR: 0.33351417885463047
Score: 0.6042417651580527


In [64]:
def generate_oof_predictions(
    group_df,
    target_col,
    feature_cols,
    capacity,
):
    oof_rows = []

    for fold in cv_folds:
        train_mask = (
            group_df["forecast_kst_dtm"]
            < fold["train_end"]
        )

        valid_mask = (
            (group_df["forecast_kst_dtm"] >= fold["train_end"])
            & (group_df["forecast_kst_dtm"] < fold["valid_end"])
        )

        X_train = group_df.loc[
            train_mask,
            feature_cols,
        ]

        y_train = group_df.loc[
            train_mask,
            target_col,
        ]

        X_valid = group_df.loc[
            valid_mask,
            feature_cols,
        ]

        y_valid = group_df.loc[
            valid_mask,
            target_col,
        ]

        lgbm_model, pred_lgbm, _, _, _ = train_lgbm_baseline(
            X_train,
            y_train,
            X_valid,
            y_valid,
            capacity,
        )

        xgb_model, pred_xgb = train_xgb_baseline(
            X_train,
            y_train,
            X_valid,
            y_valid,
            capacity,
        )

        rf_model, pred_rf = train_rf_baseline(
            X_train,
            y_train,
            X_valid,
            capacity,
        )

        fold_result = pd.DataFrame({
            "forecast_kst_dtm": group_df.loc[
                valid_mask,
                "forecast_kst_dtm",
            ].values,
            "y_true": y_valid.values,
            "pred_lgbm": pred_lgbm,
            "pred_xgb": pred_xgb,
            "pred_rf": pred_rf,
            "fold": fold["name"],
        })

        oof_rows.append(fold_result)

    return (
        pd.concat(
            oof_rows,
            ignore_index=True,
        )
        .sort_values("forecast_kst_dtm")
        .reset_index(drop=True)
    )

In [65]:
oof_group_1 = generate_oof_predictions(
    train_group_1_exp2,
    "kpx_group_1",
    feature_cols_exp2,
    capacity=21600,
)

oof_group_2 = generate_oof_predictions(
    train_group_2_exp2,
    "kpx_group_2",
    feature_cols_exp2,
    capacity=21600,
)

oof_group_3 = generate_oof_predictions(
    train_group_3_exp2,
    "kpx_group_3",
    feature_cols_exp2,
    capacity=21000,
)

print("Group 1 OOF:", oof_group_1.shape)
print("Group 2 OOF:", oof_group_2.shape)
print("Group 3 OOF:", oof_group_3.shape)

Group 1 OOF: (6594, 6)
Group 2 OOF: (6594, 6)
Group 3 OOF: (6594, 6)


In [66]:
def evaluate_oof_models(
    oof_df,
    capacity,
):
    results = []

    for model_name, pred_col in [
        ("LightGBM", "pred_lgbm"),
        ("XGBoost", "pred_xgb"),
        ("RandomForest", "pred_rf"),
    ]:
        metric = calculate_group_metric(
            oof_df["y_true"],
            oof_df[pred_col],
            capacity,
        )

        score = (
            0.5 * metric["one_minus_nmae"]
            + 0.5 * metric["scr"]
        )

        results.append({
            "model": model_name,
            "nmae": metric["nmae"],
            "one_minus_nmae": metric["one_minus_nmae"],
            "ficr": metric["scr"],
            "score": score,
        })

    return (
        pd.DataFrame(results)
        .sort_values("score", ascending=False)
        .reset_index(drop=True)
    )


print("Group 1")
print(
    evaluate_oof_models(
        oof_group_1,
        21600,
    )
)

print("\nGroup 2")
print(
    evaluate_oof_models(
        oof_group_2,
        21600,
    )
)

print("\nGroup 3")
print(
    evaluate_oof_models(
        oof_group_3,
        21000,
    )
)

Group 1
          model      nmae  one_minus_nmae      ficr     score
0       XGBoost  0.119522        0.880478  0.352041  0.616259
1      LightGBM  0.119796        0.880204  0.345543  0.612874
2  RandomForest  0.122507        0.877493  0.323672  0.600583

Group 2
          model      nmae  one_minus_nmae      ficr     score
0  RandomForest  0.117401        0.882599  0.362961  0.622780
1      LightGBM  0.119329        0.880671  0.351053  0.615862
2       XGBoost  0.119688        0.880312  0.345884  0.613098

Group 3
          model      nmae  one_minus_nmae      ficr     score
0      LightGBM  0.136562        0.863438  0.289357  0.576398
1       XGBoost  0.136026        0.863974  0.284202  0.574088
2  RandomForest  0.139660        0.860340  0.272729  0.566534


In [67]:
best_oof_weight_1, oof_search_1 = optimize_group_ensemble(
    y_true=oof_group_1["y_true"],
    pred_rf=oof_group_1["pred_rf"],
    pred_xgb=oof_group_1["pred_xgb"],
    pred_lgbm=oof_group_1["pred_lgbm"],
    capacity=21600,
    step=0.05,
)

best_oof_weight_2, oof_search_2 = optimize_group_ensemble(
    y_true=oof_group_2["y_true"],
    pred_rf=oof_group_2["pred_rf"],
    pred_xgb=oof_group_2["pred_xgb"],
    pred_lgbm=oof_group_2["pred_lgbm"],
    capacity=21600,
    step=0.05,
)

best_oof_weight_3, oof_search_3 = optimize_group_ensemble(
    y_true=oof_group_3["y_true"],
    pred_rf=oof_group_3["pred_rf"],
    pred_xgb=oof_group_3["pred_xgb"],
    pred_lgbm=oof_group_3["pred_lgbm"],
    capacity=21000,
    step=0.05,
)

print("Group 1:", best_oof_weight_1)
print("Group 2:", best_oof_weight_2)
print("Group 3:", best_oof_weight_3)

Group 1: {'weight_rf': np.float64(0.05), 'weight_xgb': np.float64(0.8), 'weight_lgbm': np.float64(0.1499999999999999), 'nmae': np.float64(0.1194136636031857), 'one_minus_nmae': np.float64(0.8805863363968143), 'ficr': np.float64(0.3553656683872135), 'score': np.float64(0.6179760023920139)}
Group 2: {'weight_rf': np.float64(1.0), 'weight_xgb': np.float64(0.0), 'weight_lgbm': np.float64(0.0), 'nmae': np.float64(0.11740101241877882), 'one_minus_nmae': np.float64(0.8825989875812212), 'ficr': np.float64(0.3629610533721663), 'score': np.float64(0.6227800204766938)}
Group 3: {'weight_rf': np.float64(0.0), 'weight_xgb': np.float64(0.0), 'weight_lgbm': np.float64(1.0), 'nmae': np.float64(0.13656151450073134), 'one_minus_nmae': np.float64(0.8634384854992687), 'ficr': np.float64(0.28935733704443856), 'score': np.float64(0.5763979112718536)}


In [68]:
oof_pred_1 = make_weighted_prediction(
    oof_group_1["pred_rf"],
    oof_group_1["pred_xgb"],
    oof_group_1["pred_lgbm"],
    best_oof_weight_1,
    capacity=21600,
)

oof_pred_2 = make_weighted_prediction(
    oof_group_2["pred_rf"],
    oof_group_2["pred_xgb"],
    oof_group_2["pred_lgbm"],
    best_oof_weight_2,
    capacity=21600,
)

oof_pred_3 = make_weighted_prediction(
    oof_group_3["pred_rf"],
    oof_group_3["pred_xgb"],
    oof_group_3["pred_lgbm"],
    best_oof_weight_3,
    capacity=21000,
)

oof_group_metrics, oof_score = calculate_competition_score(
    y_true_list=[
        oof_group_1["y_true"],
        oof_group_2["y_true"],
        oof_group_3["y_true"],
    ],
    y_pred_list=[
        oof_pred_1,
        oof_pred_2,
        oof_pred_3,
    ],
    capacities_list=[
        21600,
        21600,
        21000,
    ],
)

print(oof_group_metrics)

print("\nOOF 앙상블 공식 지표")
print("1-NMAE:", oof_score["one_minus_nmae"])
print("FICR:", oof_score["ficr"])
print("Score:", oof_score["score"])

   valid_count      nmae  one_minus_nmae       scr      ficr    group
0         3710  0.119414        0.880586  0.355366  0.355366  Group 1
1         3675  0.117401        0.882599  0.362961  0.362961  Group 2
2         3296  0.136562        0.863438  0.289357  0.289357  Group 3

OOF 앙상블 공식 지표
1-NMAE: 0.8755412698257681
FICR: 0.3358946862679395
Score: 0.6057179780468538


In [69]:
def optimize_group_postprocess(
    y_true,
    y_pred,
    capacity,
    scale_values=np.arange(0.90, 1.11, 0.01),
    bias_values=np.arange(-1000, 1001, 100),
):
    best_result = None
    rows = []

    for scale in scale_values:
        for bias in bias_values:
            pred_adj = np.clip(
                scale * np.asarray(y_pred) + bias,
                0,
                capacity,
            )

            metric = calculate_group_metric(
                y_true,
                pred_adj,
                capacity,
            )

            score = (
                0.5 * metric["one_minus_nmae"]
                + 0.5 * metric["scr"]
            )

            row = {
                "scale": scale,
                "bias": bias,
                "nmae": metric["nmae"],
                "one_minus_nmae": metric["one_minus_nmae"],
                "ficr": metric["scr"],
                "score": score,
            }

            rows.append(row)

            if (
                best_result is None
                or score > best_result["score"]
            ):
                best_result = row

    return (
        best_result,
        pd.DataFrame(rows).sort_values(
            "score",
            ascending=False,
        ),
    )

In [70]:
best_post_1, search_post_1 = optimize_group_postprocess(
    oof_group_1["y_true"],
    oof_pred_1,
    capacity=21600,
)

best_post_2, search_post_2 = optimize_group_postprocess(
    oof_group_2["y_true"],
    oof_pred_2,
    capacity=21600,
)

best_post_3, search_post_3 = optimize_group_postprocess(
    oof_group_3["y_true"],
    oof_pred_3,
    capacity=21000,
)

print("Group 1:", best_post_1)
print("Group 2:", best_post_2)
print("Group 3:", best_post_3)

Group 1: {'scale': np.float64(1.0500000000000003), 'bias': np.int64(900), 'nmae': np.float64(0.11316631053575965), 'one_minus_nmae': np.float64(0.8868336894642403), 'ficr': np.float64(0.44163689353439284), 'score': np.float64(0.6642352914993166)}
Group 2: {'scale': np.float64(1.0500000000000003), 'bias': np.int64(800), 'nmae': np.float64(0.11332929482618195), 'one_minus_nmae': np.float64(0.8866707051738181), 'ficr': np.float64(0.44491554548336315), 'score': np.float64(0.6657931253285906)}
Group 3: {'scale': np.float64(1.0300000000000002), 'bias': np.int64(1000), 'nmae': np.float64(0.13180141683433036), 'one_minus_nmae': np.float64(0.8681985831656697), 'ficr': np.float64(0.33453118701491785), 'score': np.float64(0.6013648850902937)}


In [71]:
oof_pred_post_1 = np.clip(
    best_post_1["scale"] * oof_pred_1
    + best_post_1["bias"],
    0,
    21600,
)

oof_pred_post_2 = np.clip(
    best_post_2["scale"] * oof_pred_2
    + best_post_2["bias"],
    0,
    21600,
)

oof_pred_post_3 = np.clip(
    best_post_3["scale"] * oof_pred_3
    + best_post_3["bias"],
    0,
    21000,
)

post_group_metrics, post_score = calculate_competition_score(
    y_true_list=[
        oof_group_1["y_true"],
        oof_group_2["y_true"],
        oof_group_3["y_true"],
    ],
    y_pred_list=[
        oof_pred_post_1,
        oof_pred_post_2,
        oof_pred_post_3,
    ],
    capacities_list=[
        21600,
        21600,
        21000,
    ],
)

print(post_group_metrics)

print("\n후처리 OOF 지표")
print("1-NMAE:", post_score["one_minus_nmae"])
print("FICR:", post_score["ficr"])
print("Score:", post_score["score"])

   valid_count      nmae  one_minus_nmae       scr      ficr    group
0         3710  0.113166        0.886834  0.441637  0.441637  Group 1
1         3675  0.113329        0.886671  0.444916  0.444916  Group 2
2         3296  0.131801        0.868199  0.334531  0.334531  Group 3

후처리 OOF 지표
1-NMAE: 0.8805676592679094
FICR: 0.40702787534422463
Score: 0.643797767306067


In [72]:
print("Group 1:", best_post_1)
print("Group 2:", best_post_2)
print("Group 3:", best_post_3)

Group 1: {'scale': np.float64(1.0500000000000003), 'bias': np.int64(900), 'nmae': np.float64(0.11316631053575965), 'one_minus_nmae': np.float64(0.8868336894642403), 'ficr': np.float64(0.44163689353439284), 'score': np.float64(0.6642352914993166)}
Group 2: {'scale': np.float64(1.0500000000000003), 'bias': np.int64(800), 'nmae': np.float64(0.11332929482618195), 'one_minus_nmae': np.float64(0.8866707051738181), 'ficr': np.float64(0.44491554548336315), 'score': np.float64(0.6657931253285906)}
Group 3: {'scale': np.float64(1.0300000000000002), 'bias': np.int64(1000), 'nmae': np.float64(0.13180141683433036), 'one_minus_nmae': np.float64(0.8681985831656697), 'ficr': np.float64(0.33453118701491785), 'score': np.float64(0.6013648850902937)}


In [73]:
def evaluate_postprocess_by_fold(
    oof_df,
    raw_pred,
    best_post,
    capacity,
):
    result = oof_df.copy()
    result["raw_pred"] = np.asarray(raw_pred)

    result["post_pred"] = np.clip(
        best_post["scale"] * result["raw_pred"]
        + best_post["bias"],
        0,
        capacity,
    )

    rows = []

    for fold_name, fold_df in result.groupby("fold"):
        raw_metric = calculate_group_metric(
            fold_df["y_true"],
            fold_df["raw_pred"],
            capacity,
        )

        post_metric = calculate_group_metric(
            fold_df["y_true"],
            fold_df["post_pred"],
            capacity,
        )

        raw_score = (
            0.5 * raw_metric["one_minus_nmae"]
            + 0.5 * raw_metric["scr"]
        )

        post_score = (
            0.5 * post_metric["one_minus_nmae"]
            + 0.5 * post_metric["scr"]
        )

        rows.append({
            "fold": fold_name,
            "raw_nmae": raw_metric["nmae"],
            "post_nmae": post_metric["nmae"],
            "raw_ficr": raw_metric["scr"],
            "post_ficr": post_metric["scr"],
            "raw_score": raw_score,
            "post_score": post_score,
            "score_change": post_score - raw_score,
        })

    return pd.DataFrame(rows)


post_fold_1 = evaluate_postprocess_by_fold(
    oof_group_1,
    oof_pred_1,
    best_post_1,
    capacity=21600,
)

post_fold_2 = evaluate_postprocess_by_fold(
    oof_group_2,
    oof_pred_2,
    best_post_2,
    capacity=21600,
)

post_fold_3 = evaluate_postprocess_by_fold(
    oof_group_3,
    oof_pred_3,
    best_post_3,
    capacity=21000,
)

print("Group 1")
print(post_fold_1)

print("\nGroup 2")
print(post_fold_2)

print("\nGroup 3")
print(post_fold_3)

Group 1
     fold  raw_nmae  post_nmae  raw_ficr  post_ficr  raw_score  post_score  \
0  fold_1  0.118303   0.117980  0.362229   0.413749   0.621963    0.647884   
1  fold_2  0.126378   0.115968  0.345025   0.416075   0.609323    0.650054   
2  fold_3  0.114995   0.106528  0.357158   0.477441   0.621081    0.685457   

   score_change  
0      0.025921  
1      0.040730  
2      0.064375  

Group 2
     fold  raw_nmae  post_nmae  raw_ficr  post_ficr  raw_score  post_score  \
0  fold_1  0.109472   0.113140  0.395053   0.429554   0.642790    0.658207   
1  fold_2  0.111301   0.114152  0.435883   0.454013   0.662291    0.669931   
2  fold_3  0.129928   0.112850  0.292981   0.449513   0.581527    0.668331   

   score_change  
0      0.015417  
1      0.007640  
2      0.086805  

Group 3
     fold  raw_nmae  post_nmae  raw_ficr  post_ficr  raw_score  post_score  \
0  fold_1  0.132039   0.126643  0.279614   0.315723   0.573787    0.594540   
1  fold_2  0.148471   0.136759  0.230552   0.325

In [74]:
def apply_postprocess(
    y_pred,
    post_params,
    capacity,
):
    return np.clip(
        post_params["scale"] * np.asarray(y_pred)
        + post_params["bias"],
        0,
        capacity,
    )


def walk_forward_postprocess_validation(
    oof_df,
    raw_pred,
    capacity,
):
    df = oof_df.copy()
    df["raw_pred"] = np.asarray(raw_pred)

    fold_order = [
        "fold_1",
        "fold_2",
        "fold_3",
    ]

    rows = []

    for valid_idx in range(1, len(fold_order)):
        calibration_folds = fold_order[:valid_idx]
        valid_fold = fold_order[valid_idx]

        calibration_mask = df["fold"].isin(
            calibration_folds
        )

        valid_mask = (
            df["fold"] == valid_fold
        )

        best_post, _ = optimize_group_postprocess(
            y_true=df.loc[
                calibration_mask,
                "y_true",
            ],
            y_pred=df.loc[
                calibration_mask,
                "raw_pred",
            ],
            capacity=capacity,
        )

        valid_raw_pred = df.loc[
            valid_mask,
            "raw_pred",
        ]

        valid_post_pred = apply_postprocess(
            valid_raw_pred,
            best_post,
            capacity,
        )

        raw_metric = calculate_group_metric(
            df.loc[valid_mask, "y_true"],
            valid_raw_pred,
            capacity,
        )

        post_metric = calculate_group_metric(
            df.loc[valid_mask, "y_true"],
            valid_post_pred,
            capacity,
        )

        raw_score = (
            0.5 * raw_metric["one_minus_nmae"]
            + 0.5 * raw_metric["scr"]
        )

        post_score = (
            0.5 * post_metric["one_minus_nmae"]
            + 0.5 * post_metric["scr"]
        )

        rows.append({
            "calibration_folds": "+".join(
                calibration_folds
            ),
            "valid_fold": valid_fold,
            "scale": best_post["scale"],
            "bias": best_post["bias"],
            "raw_nmae": raw_metric["nmae"],
            "post_nmae": post_metric["nmae"],
            "raw_ficr": raw_metric["scr"],
            "post_ficr": post_metric["scr"],
            "raw_score": raw_score,
            "post_score": post_score,
            "score_change": post_score - raw_score,
        })

    return pd.DataFrame(rows)

In [75]:
walk_post_1 = walk_forward_postprocess_validation(
    oof_group_1,
    oof_pred_1,
    capacity=21600,
)

walk_post_2 = walk_forward_postprocess_validation(
    oof_group_2,
    oof_pred_2,
    capacity=21600,
)

walk_post_3 = walk_forward_postprocess_validation(
    oof_group_3,
    oof_pred_3,
    capacity=21000,
)

print("Group 1")
print(walk_post_1)

print("\nGroup 2")
print(walk_post_2)

print("\nGroup 3")
print(walk_post_3)

Group 1
  calibration_folds valid_fold  scale  bias  raw_nmae  post_nmae  raw_ficr  \
0            fold_1     fold_2   1.05   700  0.126378   0.116078  0.345025   
1     fold_1+fold_2     fold_3   1.05   700  0.114995   0.106054  0.357158   

   post_ficr  raw_score  post_score  score_change  
0   0.411763   0.609323    0.647843      0.038519  
1   0.472179   0.621081    0.683063      0.061981  

Group 2
  calibration_folds valid_fold  scale  bias  raw_nmae  post_nmae  raw_ficr  \
0            fold_1     fold_2   0.98  1000  0.111301   0.107448  0.435883   
1     fold_1+fold_2     fold_3   0.98  1000  0.129928   0.118030  0.292981   

   post_ficr  raw_score  post_score  score_change  
0   0.471279   0.662291    0.681916      0.019625  
1   0.371267   0.581527    0.626619      0.045092  

Group 3
  calibration_folds valid_fold  scale  bias  raw_nmae  post_nmae  raw_ficr  \
0            fold_1     fold_2   1.11   900  0.148471   0.139203  0.230552   
1     fold_1+fold_2     fold_3   1.1

In [76]:
def apply_final_postprocess(
    pred_group_1,
    pred_group_2,
    pred_group_3,
):
    pred_group_1_post = np.clip(
        1.05 * np.asarray(pred_group_1) + 700,
        0,
        21600,
    )

    pred_group_2_post = np.clip(
        0.98 * np.asarray(pred_group_2) + 1000,
        0,
        21600,
    )

    pred_group_3_post = np.clip(
        np.asarray(pred_group_3),
        0,
        21000,
    )

    return (
        pred_group_1_post,
        pred_group_2_post,
        pred_group_3_post,
    )

In [77]:
oof_final_pred_1 = np.clip(
    1.05 * np.asarray(oof_pred_1) + 700,
    0,
    21600,
)

oof_final_pred_2 = np.clip(
    0.98 * np.asarray(oof_pred_2) + 1000,
    0,
    21600,
)

oof_final_pred_3 = np.clip(
    np.asarray(oof_pred_3),
    0,
    21000,
)

final_oof_group_metrics, final_oof_score = (
    calculate_competition_score(
        y_true_list=[
            oof_group_1["y_true"],
            oof_group_2["y_true"],
            oof_group_3["y_true"],
        ],
        y_pred_list=[
            oof_final_pred_1,
            oof_final_pred_2,
            oof_final_pred_3,
        ],
        capacities_list=[
            21600,
            21600,
            21000,
        ],
    )
)

print(final_oof_group_metrics)

print("\n최종 보수적 후처리 OOF")
print("1-NMAE:", final_oof_score["one_minus_nmae"])
print("FICR:", final_oof_score["ficr"])
print("Score:", final_oof_score["score"])

   valid_count      nmae  one_minus_nmae       scr      ficr    group
0         3710  0.112578        0.887422  0.439397  0.439397  Group 1
1         3675  0.111542        0.888458  0.421106  0.421106  Group 2
2         3296  0.136562        0.863438  0.289357  0.289357  Group 3

최종 보수적 후처리 OOF
1-NMAE: 0.8797727638696486
FICR: 0.3832870660808044
Score: 0.6315299149752265


In [80]:
weight_list = [best_oof_weight_1, best_oof_weight_2, best_oof_weight_3]

for weight in weight_list:
    print(weight)

{'weight_rf': np.float64(0.05), 'weight_xgb': np.float64(0.8), 'weight_lgbm': np.float64(0.1499999999999999), 'nmae': np.float64(0.1194136636031857), 'one_minus_nmae': np.float64(0.8805863363968143), 'ficr': np.float64(0.3553656683872135), 'score': np.float64(0.6179760023920139)}
{'weight_rf': np.float64(1.0), 'weight_xgb': np.float64(0.0), 'weight_lgbm': np.float64(0.0), 'nmae': np.float64(0.11740101241877882), 'one_minus_nmae': np.float64(0.8825989875812212), 'ficr': np.float64(0.3629610533721663), 'score': np.float64(0.6227800204766938)}
{'weight_rf': np.float64(0.0), 'weight_xgb': np.float64(0.0), 'weight_lgbm': np.float64(1.0), 'nmae': np.float64(0.13656151450073134), 'one_minus_nmae': np.float64(0.8634384854992687), 'ficr': np.float64(0.28935733704443856), 'score': np.float64(0.5763979112718536)}


In [81]:
lgbm_n_estimators_1 = int(
    round(
        cv_result_1["best_iteration"].mean()
    )
)

lgbm_n_estimators_3 = int(
    round(
        cv_result_3["best_iteration"].mean()
    )
)

xgb_n_estimators_1 = (
    int(xgb_exp2_1.best_iteration) + 1
    if getattr(xgb_exp2_1, "best_iteration", None) is not None
    else 300
)

print("Group 1 LGBM trees:", lgbm_n_estimators_1)
print("Group 1 XGB trees:", xgb_n_estimators_1)
print("Group 3 LGBM trees:", lgbm_n_estimators_3)

Group 1 LGBM trees: 201
Group 1 XGB trees: 247
Group 3 LGBM trees: 201


In [82]:
X_full_1 = train_group_1_exp2[
    feature_cols_exp2
]

y_full_1 = train_group_1_exp2[
    "kpx_group_1"
]

X_full_2 = train_group_2_exp2[
    feature_cols_exp2
]

y_full_2 = train_group_2_exp2[
    "kpx_group_2"
]

X_full_3 = train_group_3_exp2[
    feature_cols_exp2
]

y_full_3 = train_group_3_exp2[
    "kpx_group_3"
]

X_test_final = test_exp2[
    feature_cols_exp2
]

print("Group 1:", X_full_1.shape)
print("Group 2:", X_full_2.shape)
print("Group 3:", X_full_3.shape)
print("Test:", X_test_final.shape)

Group 1: (26200, 305)
Group 2: (26201, 305)
Group 3: (17538, 305)
Test: (8760, 305)


In [83]:
final_rf_1 = RandomForestRegressor(
    n_estimators=400,
    max_depth=None,
    min_samples_split=4,
    min_samples_leaf=2,
    max_features=0.7,
    random_state=42,
    n_jobs=-1,
)

final_xgb_1 = XGBRegressor(
    objective="reg:squarederror",
    n_estimators=xgb_n_estimators_1,
    learning_rate=0.03,
    max_depth=6,
    min_child_weight=5,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1,
)

final_lgbm_1 = LGBMRegressor(
    objective="regression",
    n_estimators=lgbm_n_estimators_1,
    learning_rate=0.03,
    num_leaves=31,
    max_depth=-1,
    min_child_samples=20,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1,
    verbosity=-1,
)

final_rf_1.fit(
    X_full_1,
    y_full_1,
)

final_xgb_1.fit(
    X_full_1,
    y_full_1,
    verbose=False,
)

final_lgbm_1.fit(
    X_full_1,
    y_full_1,
)

,learning_rate,0.03
,n_estimators,201
,objective,'regression'
,subsample,0.8
,colsample_bytree,0.8
,reg_alpha,0.1
,reg_lambda,1.0
,random_state,42
,n_jobs,-1
,verbosity,-1
,boosting_type,'gbdt'


In [84]:
final_rf_2 = RandomForestRegressor(
    n_estimators=400,
    max_depth=None,
    min_samples_split=4,
    min_samples_leaf=2,
    max_features=0.7,
    random_state=42,
    n_jobs=-1,
)

final_rf_2.fit(
    X_full_2,
    y_full_2,
)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",400
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",4
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",2
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",0.7
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel. :meth:`fit`, :meth:`predict`,:meth:`decision_path` and :meth:`apply` are all parallelized over thetrees. ``None`` means 1 unless in a :obj:`joblib.parallel_backend`context. ``-1`` means using all processors. See :term:`Glossary<n_jobs>` for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"criterion criterion: {""squared_error"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""absolute_error"" for the meanabsolute error, which minimizes the L1 loss using the median of each terminalnode, and ""poisson"" which uses reduction in Poisson deviance to find splits,also using the mean of each terminal node... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion... versionchanged:: 1.9 Criterion `""friedman_mse""` was deprecated.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"m

In [85]:
final_lgbm_3 = LGBMRegressor(
    objective="regression",
    n_estimators=lgbm_n_estimators_3,
    learning_rate=0.03,
    num_leaves=31,
    max_depth=-1,
    min_child_samples=20,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1,
    verbosity=-1,
)

final_lgbm_3.fit(
    X_full_3,
    y_full_3,
)

,learning_rate,0.03
,n_estimators,201
,objective,'regression'
,subsample,0.8
,colsample_bytree,0.8
,reg_alpha,0.1
,reg_lambda,1.0
,random_state,42
,n_jobs,-1
,verbosity,-1
,boosting_type,'gbdt'


In [86]:
pred_rf_test_1 = final_rf_1.predict(
    X_test_final
)

pred_xgb_test_1 = final_xgb_1.predict(
    X_test_final
)

pred_lgbm_test_1 = final_lgbm_1.predict(
    X_test_final
)

pred_test_1 = (
    0.05 * pred_rf_test_1
    + 0.80 * pred_xgb_test_1
    + 0.15 * pred_lgbm_test_1
)

pred_test_2 = final_rf_2.predict(
    X_test_final
)

pred_test_3 = final_lgbm_3.predict(
    X_test_final
)

pred_test_1 = np.clip(
    pred_test_1,
    0,
    21600,
)

pred_test_2 = np.clip(
    pred_test_2,
    0,
    21600,
)

pred_test_3 = np.clip(
    pred_test_3,
    0,
    21000,
)

In [87]:
pred_test_1_post = np.clip(
    1.05 * pred_test_1 + 700,
    0,
    21600,
)

pred_test_2_post = np.clip(
    0.98 * pred_test_2 + 1000,
    0,
    21600,
)

pred_test_3_post = pred_test_3.copy()

In [88]:
prediction_summary = pd.DataFrame({
    "group": [
        "Group 1",
        "Group 2",
        "Group 3",
    ],
    "min": [
        pred_test_1_post.min(),
        pred_test_2_post.min(),
        pred_test_3_post.min(),
    ],
    "mean": [
        pred_test_1_post.mean(),
        pred_test_2_post.mean(),
        pred_test_3_post.mean(),
    ],
    "max": [
        pred_test_1_post.max(),
        pred_test_2_post.max(),
        pred_test_3_post.max(),
    ],
    "capacity": [
        21600,
        21600,
        21000,
    ],
})

print(prediction_summary)

print(
    "\nNaN:",
    np.isnan(
        np.column_stack([
            pred_test_1_post,
            pred_test_2_post,
            pred_test_3_post,
        ])
    ).sum()
)

     group          min         mean           max  capacity
0  Group 1   700.000000  8549.268658  21600.000000     21600
1  Group 2  1002.983086  8746.074571  21106.955719     21600
2  Group 3     0.000000  6354.335184  19364.201264     21000

NaN: 0


In [89]:
submission = sample_submission.copy()

submission["kpx_group_1"] = (
    pred_test_1_post
)

submission["kpx_group_2"] = (
    pred_test_2_post
)

submission["kpx_group_3"] = (
    pred_test_3_post
)

submission["forecast_kst_dtm"] = (
    pd.to_datetime(
        submission["forecast_kst_dtm"]
    )
    .dt.strftime("%Y-%m-%d %H:%M:%S")
)

submission.to_csv(
    "submission_exp2_oof_postprocess.csv",
    index=False,
)

print("제출 파일:", submission.shape)
print(submission.head())

print("\n결측치")
print(submission.isna().sum())

제출 파일: (8760, 5)
     forecast_id     forecast_kst_dtm   kpx_group_1   kpx_group_2  \
0  forecast_0001  2025-01-01 01:00:00  20664.193207  18667.455405   
1  forecast_0002  2025-01-01 02:00:00  20450.548679  18533.986608   
2  forecast_0003  2025-01-01 03:00:00  20162.590527  18343.943228   
3  forecast_0004  2025-01-01 04:00:00  19581.556108  17989.286276   
4  forecast_0005  2025-01-01 05:00:00  19028.682311  18099.614914   

    kpx_group_3  
0  16966.179467  
1  16944.578948  
2  16829.110360  
3  15351.988762  
4  14817.391597  

결측치
forecast_id         0
forecast_kst_dtm    0
kpx_group_1         0
kpx_group_2         0
kpx_group_3         0
dtype: int64


In [94]:
def search_robust_postprocess(
    oof_df,
    raw_pred,
    capacity,
    scale_values,
    bias_values,
):
    df = oof_df.copy()
    df["raw_pred"] = np.asarray(raw_pred)

    raw_fold_scores = {}

    for fold_name, fold_df in df.groupby("fold"):
        raw_metric = calculate_group_metric(
            y_true=fold_df["y_true"],
            y_pred=fold_df["raw_pred"],
            capacity=capacity,
        )

        raw_fold_scores[fold_name] = (
            0.5 * raw_metric["one_minus_nmae"]
            + 0.5 * raw_metric["scr"]
        )

    result_rows = []

    for scale in scale_values:
        for bias in bias_values:
            fold_scores = []
            fold_changes = []
            fold_nmae = []
            fold_ficr = []

            for fold_name, fold_df in df.groupby("fold"):
                post_pred = np.clip(
                    scale * fold_df["raw_pred"].to_numpy()
                    + bias,
                    0,
                    capacity,
                )

                metric = calculate_group_metric(
                    y_true=fold_df["y_true"],
                    y_pred=post_pred,
                    capacity=capacity,
                )

                post_score = (
                    0.5 * metric["one_minus_nmae"]
                    + 0.5 * metric["scr"]
                )

                fold_scores.append(post_score)
                fold_changes.append(
                    post_score
                    - raw_fold_scores[fold_name]
                )
                fold_nmae.append(metric["nmae"])
                fold_ficr.append(metric["scr"])

            result_rows.append({
                "scale": scale,
                "bias": bias,
                "mean_score": np.mean(fold_scores),
                "std_score": np.std(fold_scores),
                "min_score": np.min(fold_scores),
                "mean_score_change": np.mean(fold_changes),
                "min_score_change": np.min(fold_changes),
                "mean_nmae": np.mean(fold_nmae),
                "mean_ficr": np.mean(fold_ficr),
            })

    results = pd.DataFrame(result_rows)

    robust_results = (
        results[
            results["min_score_change"] >= 0
        ]
        .sort_values(
            [
                "mean_score",
                "min_score",
                "std_score",
            ],
            ascending=[
                False,
                False,
                True,
            ],
        )
        .reset_index(drop=True)
    )

    all_results = (
        results
        .sort_values(
            "mean_score",
            ascending=False,
        )
        .reset_index(drop=True)
    )

    return robust_results, all_results

In [95]:
group1_scale_values = np.round(
    np.arange(
        1.030,
        1.0701,
        0.002,
    ),
    3,
)

group1_bias_values = np.arange(
    500,
    901,
    25,
)

group1_robust_search, group1_all_search = (
    search_robust_postprocess(
        oof_df=oof_group_1,
        raw_pred=oof_pred_1,
        capacity=21600,
        scale_values=group1_scale_values,
        bias_values=group1_bias_values,
    )
)

print("Group 1 안정적 후보 수:", len(group1_robust_search))
print(group1_robust_search.head(10))

Group 1 안정적 후보 수: 357
   scale  bias  mean_score  std_score  min_score  mean_score_change  \
0  1.050   875    0.661288   0.017394   0.648231           0.043832   
1  1.048   900    0.661232   0.017225   0.648184           0.043776   
2  1.056   825    0.661140   0.017714   0.647359           0.043684   
3  1.052   900    0.661134   0.017779   0.647341           0.043678   
4  1.050   900    0.661132   0.017223   0.647884           0.043676   
5  1.054   825    0.661092   0.017418   0.648155           0.043636   
6  1.052   850    0.661085   0.017411   0.648114           0.043629   
7  1.046   750    0.661029   0.015496   0.648587           0.043573   
8  1.054   875    0.661018   0.017897   0.647208           0.043562   
9  1.054   850    0.660999   0.017623   0.647669           0.043543   

   min_score_change  mean_nmae  mean_ficr  
0          0.026268   0.113388   0.435964  
1          0.026221   0.113358   0.435822  
2          0.025396   0.113591   0.435871  
3          0.025378 

In [96]:
group2_scale_values = np.round(
    np.arange(
        0.960,
        1.0001,
        0.002,
    ),
    3,
)

group2_bias_values = np.arange(
    800,
    1201,
    25,
)

group2_robust_search, group2_all_search = (
    search_robust_postprocess(
        oof_df=oof_group_2,
        raw_pred=oof_pred_2,
        capacity=21600,
        scale_values=group2_scale_values,
        bias_values=group2_bias_values,
    )
)

print("Group 2 안정적 후보 수:", len(group2_robust_search))
print(group2_robust_search.head(10))

Group 2 안정적 후보 수: 357
   scale  bias  mean_score  std_score  min_score  mean_score_change  \
0  1.000  1200    0.666449   0.011016   0.653594           0.037580   
1  0.998  1200    0.665634   0.011536   0.651197           0.036765   
2  1.000  1175    0.665323   0.010501   0.652171           0.036454   
3  0.996  1200    0.665229   0.011466   0.650638           0.036359   
4  0.992  1200    0.665186   0.012081   0.649293           0.036316   
5  0.994  1175    0.665180   0.011879   0.649680           0.036311   
6  0.998  1175    0.665053   0.011703   0.650310           0.036184   
7  1.000  1150    0.665004   0.011642   0.650389           0.036135   
8  1.000  1075    0.664880   0.011143   0.649864           0.036011   
9  0.994  1200    0.664854   0.011652   0.649960           0.035985   

   min_score_change  mean_nmae  mean_ficr  
0          0.018208   0.111835   0.444732  
1          0.017142   0.111740   0.443009  
2          0.015582   0.111737   0.442384  
3          0.016360 

In [97]:
best_fine_post_1 = group1_robust_search.iloc[0]
best_fine_post_2 = group2_robust_search.iloc[0]

print("Group 1 최적")
print(best_fine_post_1)

print("\nGroup 2 최적")
print(best_fine_post_2)

Group 1 최적
scale                  1.050000
bias                 875.000000
mean_score             0.661288
std_score              0.017394
min_score              0.648231
mean_score_change      0.043832
min_score_change       0.026268
mean_nmae              0.113388
mean_ficr              0.435964
Name: 0, dtype: float64

Group 2 최적
scale                   1.000000
bias                 1200.000000
mean_score              0.666449
std_score               0.011016
min_score               0.653594
mean_score_change       0.037580
min_score_change        0.018208
mean_nmae               0.111835
mean_ficr               0.444732
Name: 0, dtype: float64


In [98]:
def evaluate_fixed_postprocess(
    oof_df,
    raw_pred,
    capacity,
    scale,
    bias,
):
    post_pred = np.clip(
        scale * np.asarray(raw_pred) + bias,
        0,
        capacity,
    )

    metric = calculate_group_metric(
        y_true=oof_df["y_true"],
        y_pred=post_pred,
        capacity=capacity,
    )

    score = (
        0.5 * metric["one_minus_nmae"]
        + 0.5 * metric["scr"]
    )

    return {
        "scale": scale,
        "bias": bias,
        "nmae": metric["nmae"],
        "one_minus_nmae": metric["one_minus_nmae"],
        "ficr": metric["scr"],
        "score": score,
    }


postprocess_comparison = pd.DataFrame([
    {
        "group": "Group 1",
        "version": "current",
        **evaluate_fixed_postprocess(
            oof_group_1,
            oof_pred_1,
            capacity=21600,
            scale=1.05,
            bias=700,
        ),
    },
    {
        "group": "Group 1",
        "version": "fine_search",
        **evaluate_fixed_postprocess(
            oof_group_1,
            oof_pred_1,
            capacity=21600,
            scale=float(best_fine_post_1["scale"]),
            bias=float(best_fine_post_1["bias"]),
        ),
    },
    {
        "group": "Group 2",
        "version": "current",
        **evaluate_fixed_postprocess(
            oof_group_2,
            oof_pred_2,
            capacity=21600,
            scale=0.98,
            bias=1000,
        ),
    },
    {
        "group": "Group 2",
        "version": "fine_search",
        **evaluate_fixed_postprocess(
            oof_group_2,
            oof_pred_2,
            capacity=21600,
            scale=float(best_fine_post_2["scale"]),
            bias=float(best_fine_post_2["bias"]),
        ),
    },
])

print(postprocess_comparison)

     group      version  scale    bias      nmae  one_minus_nmae      ficr  \
0  Group 1      current   1.05   700.0  0.112578        0.887422  0.439397   
1  Group 1  fine_search   1.05   875.0  0.113059        0.886941  0.441934   
2  Group 2      current   0.98  1000.0  0.111542        0.888458  0.421106   
3  Group 2  fine_search   1.00  1200.0  0.111946        0.888054  0.441234   

      score  
0  0.663410  
1  0.664437  
2  0.654782  
3  0.664644  


In [99]:
fine_pred_1 = np.clip(
    float(best_fine_post_1["scale"])
    * np.asarray(oof_pred_1)
    + float(best_fine_post_1["bias"]),
    0,
    21600,
)

fine_pred_2 = np.clip(
    float(best_fine_post_2["scale"])
    * np.asarray(oof_pred_2)
    + float(best_fine_post_2["bias"]),
    0,
    21600,
)

fine_pred_3 = np.clip(
    np.asarray(oof_pred_3),
    0,
    21000,
)

fine_group_metrics, fine_total_score = (
    calculate_competition_score(
        y_true_list=[
            oof_group_1["y_true"],
            oof_group_2["y_true"],
            oof_group_3["y_true"],
        ],
        y_pred_list=[
            fine_pred_1,
            fine_pred_2,
            fine_pred_3,
        ],
        capacities_list=[
            21600,
            21600,
            21000,
        ],
    )
)

print(fine_group_metrics)

print("\n미세 탐색 후 OOF")
print("1-NMAE:", fine_total_score["one_minus_nmae"])
print("FICR:", fine_total_score["ficr"])
print("Score:", fine_total_score["score"])

print(
    "\n기존 보수적 후처리 대비:",
    fine_total_score["score"] - 0.6315299149752265,
)

   valid_count      nmae  one_minus_nmae       scr      ficr    group
0         3710  0.113059        0.886941  0.441934  0.441934  Group 1
1         3675  0.111946        0.888054  0.441234  0.441234  Group 2
2         3296  0.136562        0.863438  0.289357  0.289357  Group 3

미세 탐색 후 OOF
1-NMAE: 0.8794778677033671
FICR: 0.39084183111733156
Score: 0.6351598494103493

기존 보수적 후처리 대비: 0.003629934435122828


In [100]:
def walk_forward_fine_postprocess(
    oof_df,
    raw_pred,
    capacity,
    scale_values,
    bias_values,
):
    df = oof_df.copy()
    df["raw_pred"] = np.asarray(raw_pred)

    fold_order = [
        "fold_1",
        "fold_2",
        "fold_3",
    ]

    rows = []

    for valid_idx in range(1, len(fold_order)):
        calibration_folds = fold_order[:valid_idx]
        valid_fold = fold_order[valid_idx]

        calibration_df = df[
            df["fold"].isin(calibration_folds)
        ]

        valid_df = df[
            df["fold"] == valid_fold
        ]

        calibration_robust, calibration_all = (
            search_robust_postprocess(
                oof_df=calibration_df,
                raw_pred=calibration_df["raw_pred"],
                capacity=capacity,
                scale_values=scale_values,
                bias_values=bias_values,
            )
        )

        if len(calibration_robust) > 0:
            best_row = calibration_robust.iloc[0]
        else:
            best_row = calibration_all.iloc[0]

        scale = float(best_row["scale"])
        bias = float(best_row["bias"])

        raw_metric = calculate_group_metric(
            y_true=valid_df["y_true"],
            y_pred=valid_df["raw_pred"],
            capacity=capacity,
        )

        post_pred = np.clip(
            scale * valid_df["raw_pred"].to_numpy()
            + bias,
            0,
            capacity,
        )

        post_metric = calculate_group_metric(
            y_true=valid_df["y_true"],
            y_pred=post_pred,
            capacity=capacity,
        )

        raw_score = (
            0.5 * raw_metric["one_minus_nmae"]
            + 0.5 * raw_metric["scr"]
        )

        post_score = (
            0.5 * post_metric["one_minus_nmae"]
            + 0.5 * post_metric["scr"]
        )

        rows.append({
            "calibration_folds": "+".join(
                calibration_folds
            ),
            "valid_fold": valid_fold,
            "scale": scale,
            "bias": bias,
            "raw_nmae": raw_metric["nmae"],
            "post_nmae": post_metric["nmae"],
            "raw_ficr": raw_metric["scr"],
            "post_ficr": post_metric["scr"],
            "raw_score": raw_score,
            "post_score": post_score,
            "score_change": post_score - raw_score,
        })

    return pd.DataFrame(rows)

In [101]:
walk_fine_1 = walk_forward_fine_postprocess(
    oof_df=oof_group_1,
    raw_pred=oof_pred_1,
    capacity=21600,
    scale_values=group1_scale_values,
    bias_values=group1_bias_values,
)

print(walk_fine_1)

  calibration_folds valid_fold  scale   bias  raw_nmae  post_nmae  raw_ficr  \
0            fold_1     fold_2  1.046  750.0  0.126378   0.115941  0.345025   
1     fold_1+fold_2     fold_3  1.046  750.0  0.114995   0.105917  0.357158   

   post_ficr  raw_score  post_score  score_change  
0   0.413114   0.609323    0.648587      0.039263  
1   0.471662   0.621081    0.682873      0.061791  


In [102]:
group2_bias_values_extended = np.arange(
    800,
    1401,
    25,
)

walk_fine_2 = walk_forward_fine_postprocess(
    oof_df=oof_group_2,
    raw_pred=oof_pred_2,
    capacity=21600,
    scale_values=group2_scale_values,
    bias_values=group2_bias_values_extended,
)

print(walk_fine_2)

  calibration_folds valid_fold  scale    bias  raw_nmae  post_nmae  raw_ficr  \
0            fold_1     fold_2  0.972  1075.0  0.111301   0.107184  0.435883   
1     fold_1+fold_2     fold_3  0.960  1325.0  0.129928   0.117728  0.292981   

   post_ficr  raw_score  post_score  score_change  
0   0.469304   0.662291    0.681060      0.018770  
1   0.373212   0.581527    0.627742      0.046215  


In [103]:
def apply_final_postprocess_v2(
    pred_group_1,
    pred_group_2,
    pred_group_3,
):
    pred_group_1_post = np.clip(
        1.046 * np.asarray(pred_group_1) + 750,
        0,
        21600,
    )

    pred_group_2_post = np.clip(
        0.97 * np.asarray(pred_group_2) + 1200,
        0,
        21600,
    )

    pred_group_3_post = np.clip(
        np.asarray(pred_group_3),
        0,
        21000,
    )

    return (
        pred_group_1_post,
        pred_group_2_post,
        pred_group_3_post,
    )

In [104]:
oof_candidate_pred_1 = np.clip(
    1.046 * np.asarray(oof_pred_1) + 750,
    0,
    21600,
)

oof_candidate_pred_2 = np.clip(
    0.97 * np.asarray(oof_pred_2) + 1200,
    0,
    21600,
)

oof_candidate_pred_3 = np.clip(
    np.asarray(oof_pred_3),
    0,
    21000,
)

candidate_group_metrics, candidate_score = (
    calculate_competition_score(
        y_true_list=[
            oof_group_1["y_true"],
            oof_group_2["y_true"],
            oof_group_3["y_true"],
        ],
        y_pred_list=[
            oof_candidate_pred_1,
            oof_candidate_pred_2,
            oof_candidate_pred_3,
        ],
        capacities_list=[
            21600,
            21600,
            21000,
        ],
    )
)

print(candidate_group_metrics)

print("\nWalk-forward 보수적 조합")
print("1-NMAE:", candidate_score["one_minus_nmae"])
print("FICR:", candidate_score["ficr"])
print("Score:", candidate_score["score"])

   valid_count      nmae  one_minus_nmae       scr      ficr    group
0         3710  0.112477        0.887523  0.440198  0.440198  Group 1
1         3675  0.111492        0.888508  0.422324  0.422324  Group 2
2         3296  0.136562        0.863438  0.289357  0.289357  Group 3

Walk-forward 보수적 조합
1-NMAE: 0.8798230941812253
FICR: 0.3839597560900634
Score: 0.6318914251356443


In [105]:
from sklearn.ensemble import ExtraTreesRegressor


def generate_group3_extratrees_oof(
    group_df,
    feature_cols,
    capacity=21000,
):
    oof_rows = []

    for fold in cv_folds:
        train_mask = (
            group_df["forecast_kst_dtm"]
            < fold["train_end"]
        )

        valid_mask = (
            (group_df["forecast_kst_dtm"] >= fold["train_end"])
            & (group_df["forecast_kst_dtm"] < fold["valid_end"])
        )

        X_train = group_df.loc[
            train_mask,
            feature_cols,
        ]

        y_train = group_df.loc[
            train_mask,
            "kpx_group_3",
        ]

        X_valid = group_df.loc[
            valid_mask,
            feature_cols,
        ]

        y_valid = group_df.loc[
            valid_mask,
            "kpx_group_3",
        ]

        model = ExtraTreesRegressor(
            n_estimators=300,
            max_depth=None,
            min_samples_split=4,
            min_samples_leaf=2,
            max_features=0.7,
            bootstrap=False,
            random_state=42,
            n_jobs=-1,
        )

        model.fit(
            X_train,
            y_train,
        )

        pred = np.clip(
            model.predict(X_valid),
            0,
            capacity,
        )

        oof_rows.append(
            pd.DataFrame({
                "forecast_kst_dtm": group_df.loc[
                    valid_mask,
                    "forecast_kst_dtm",
                ].values,
                "fold": fold["name"],
                "y_true": y_valid.values,
                "pred_et": pred,
            })
        )

        print(
            fold["name"],
            "완료 | Train:",
            len(X_train),
            "| Valid:",
            len(X_valid),
        )

    return (
        pd.concat(
            oof_rows,
            ignore_index=True,
        )
        .sort_values("forecast_kst_dtm")
        .reset_index(drop=True)
    )


oof_et_group_3 = generate_group3_extratrees_oof(
    train_group_3_exp2,
    feature_cols_exp2,
    capacity=21000,
)

print("ExtraTrees OOF:", oof_et_group_3.shape)

fold_1 완료 | Train: 10944 | Valid: 2184
fold_2 완료 | Train: 13128 | Valid: 2202
fold_3 완료 | Train: 15330 | Valid: 2208
ExtraTrees OOF: (6594, 4)


In [106]:
group3_model_oof = pd.merge(
    oof_group_3[
        [
            "forecast_kst_dtm",
            "fold",
            "y_true",
            "pred_lgbm",
        ]
    ],
    oof_et_group_3[
        [
            "forecast_kst_dtm",
            "fold",
            "pred_et",
        ]
    ],
    on=[
        "forecast_kst_dtm",
        "fold",
    ],
    how="inner",
    validate="one_to_one",
)

print(group3_model_oof.shape)
print(group3_model_oof.isna().sum())

(6594, 5)
forecast_kst_dtm    0
fold                0
y_true              0
pred_lgbm           0
pred_et             0
dtype: int64


In [107]:
group3_single_results = []

for model_name, pred_col in [
    ("LightGBM", "pred_lgbm"),
    ("ExtraTrees", "pred_et"),
]:
    metric = calculate_group_metric(
        y_true=group3_model_oof["y_true"],
        y_pred=group3_model_oof[pred_col],
        capacity=21000,
    )

    score = (
        0.5 * metric["one_minus_nmae"]
        + 0.5 * metric["scr"]
    )

    group3_single_results.append({
        "model": model_name,
        "valid_count": metric["valid_count"],
        "nmae": metric["nmae"],
        "one_minus_nmae": metric["one_minus_nmae"],
        "ficr": metric["scr"],
        "score": score,
    })

group3_single_results = (
    pd.DataFrame(group3_single_results)
    .sort_values(
        "score",
        ascending=False,
    )
    .reset_index(drop=True)
)

print(group3_single_results)

        model  valid_count      nmae  one_minus_nmae      ficr     score
0    LightGBM         3296  0.136562        0.863438  0.289357  0.576398
1  ExtraTrees         3296  0.135278        0.864722  0.282289  0.573506


In [108]:
group3_ensemble_rows = []

for weight_et in np.arange(
    0,
    1.0001,
    0.05,
):
    weight_lgbm = 1 - weight_et

    pred_ensemble = np.clip(
        weight_lgbm
        * group3_model_oof["pred_lgbm"].to_numpy()
        + weight_et
        * group3_model_oof["pred_et"].to_numpy(),
        0,
        21000,
    )

    metric = calculate_group_metric(
        y_true=group3_model_oof["y_true"],
        y_pred=pred_ensemble,
        capacity=21000,
    )

    score = (
        0.5 * metric["one_minus_nmae"]
        + 0.5 * metric["scr"]
    )

    group3_ensemble_rows.append({
        "weight_lgbm": weight_lgbm,
        "weight_et": weight_et,
        "nmae": metric["nmae"],
        "one_minus_nmae": metric["one_minus_nmae"],
        "ficr": metric["scr"],
        "score": score,
    })

group3_ensemble_search = (
    pd.DataFrame(group3_ensemble_rows)
    .sort_values(
        "score",
        ascending=False,
    )
    .reset_index(drop=True)
)

print(group3_ensemble_search.head(10))

best_group3_ensemble = (
    group3_ensemble_search.iloc[0]
)

print("\nGroup 3 최적 가중치")
print(best_group3_ensemble)

   weight_lgbm  weight_et      nmae  one_minus_nmae      ficr     score
0         0.80       0.20  0.135660        0.864340  0.289762  0.577051
1         0.85       0.15  0.135858        0.864142  0.289228  0.576685
2         0.15       0.85  0.134894        0.865106  0.287934  0.576520
3         0.20       0.80  0.134814        0.865186  0.287658  0.576422
4         1.00       0.00  0.136562        0.863438  0.289357  0.576398
5         0.75       0.25  0.135484        0.864516  0.288023  0.576269
6         0.25       0.75  0.134757        0.865243  0.287195  0.576219
7         0.95       0.05  0.136308        0.863692  0.288457  0.576074
8         0.90       0.10  0.136073        0.863927  0.288099  0.576013
9         0.70       0.30  0.135327        0.864673  0.286941  0.575807

Group 3 최적 가중치
weight_lgbm       0.800000
weight_et         0.200000
nmae              0.135660
one_minus_nmae    0.864340
ficr              0.289762
score             0.577051
Name: 0, dtype: float64


In [109]:
group3_fold_results = []

best_weight_lgbm = float(
    best_group3_ensemble["weight_lgbm"]
)

best_weight_et = float(
    best_group3_ensemble["weight_et"]
)

for fold_name, fold_df in group3_model_oof.groupby(
    "fold"
):
    pred_lgbm = fold_df[
        "pred_lgbm"
    ].to_numpy()

    pred_ensemble = np.clip(
        best_weight_lgbm * pred_lgbm
        + best_weight_et
        * fold_df["pred_et"].to_numpy(),
        0,
        21000,
    )

    lgbm_metric = calculate_group_metric(
        fold_df["y_true"],
        pred_lgbm,
        21000,
    )

    ensemble_metric = calculate_group_metric(
        fold_df["y_true"],
        pred_ensemble,
        21000,
    )

    lgbm_score = (
        0.5 * lgbm_metric["one_minus_nmae"]
        + 0.5 * lgbm_metric["scr"]
    )

    ensemble_score = (
        0.5 * ensemble_metric["one_minus_nmae"]
        + 0.5 * ensemble_metric["scr"]
    )

    group3_fold_results.append({
        "fold": fold_name,
        "lgbm_nmae": lgbm_metric["nmae"],
        "ensemble_nmae": ensemble_metric["nmae"],
        "lgbm_ficr": lgbm_metric["scr"],
        "ensemble_ficr": ensemble_metric["scr"],
        "lgbm_score": lgbm_score,
        "ensemble_score": ensemble_score,
        "score_change": ensemble_score - lgbm_score,
    })

group3_fold_results = pd.DataFrame(
    group3_fold_results
)

print(group3_fold_results)

     fold  lgbm_nmae  ensemble_nmae  lgbm_ficr  ensemble_ficr  lgbm_score  \
0  fold_1   0.132039       0.131433   0.279614       0.284808    0.573787   
1  fold_2   0.148471       0.147687   0.230552       0.230481    0.541040   
2  fold_3   0.131271       0.129994   0.341674       0.338868    0.605201   

   ensemble_score  score_change  
0        0.576688      0.002900  
1        0.541397      0.000357  
2        0.604437     -0.000764  


In [110]:
# 현재 권장 후처리
final_oof_pred_1 = np.clip(
    1.046 * np.asarray(oof_pred_1) + 750,
    0,
    21600,
)

final_oof_pred_2 = np.clip(
    0.97 * np.asarray(oof_pred_2) + 1200,
    0,
    21600,
)

# 기존 Group 3: LightGBM 단독
final_oof_pred_3_lgbm = np.clip(
    group3_model_oof["pred_lgbm"].to_numpy(),
    0,
    21000,
)

# 후보 Group 3: LightGBM 80% + ExtraTrees 20%
final_oof_pred_3_blend = np.clip(
    0.8 * group3_model_oof["pred_lgbm"].to_numpy()
    + 0.2 * group3_model_oof["pred_et"].to_numpy(),
    0,
    21000,
)

# Group 3의 y_true 순서를 결합 데이터 기준으로 맞춤
group3_y_true_aligned = (
    group3_model_oof["y_true"].to_numpy()
)

baseline_metrics, baseline_total = (
    calculate_competition_score(
        y_true_list=[
            oof_group_1["y_true"],
            oof_group_2["y_true"],
            group3_y_true_aligned,
        ],
        y_pred_list=[
            final_oof_pred_1,
            final_oof_pred_2,
            final_oof_pred_3_lgbm,
        ],
        capacities_list=[
            21600,
            21600,
            21000,
        ],
    )
)

blend_metrics, blend_total = (
    calculate_competition_score(
        y_true_list=[
            oof_group_1["y_true"],
            oof_group_2["y_true"],
            group3_y_true_aligned,
        ],
        y_pred_list=[
            final_oof_pred_1,
            final_oof_pred_2,
            final_oof_pred_3_blend,
        ],
        capacities_list=[
            21600,
            21600,
            21000,
        ],
    )
)

comparison = pd.DataFrame([
    {
        "version": "Group3_LGBM",
        "one_minus_nmae": baseline_total["one_minus_nmae"],
        "ficr": baseline_total["ficr"],
        "score": baseline_total["score"],
    },
    {
        "version": "Group3_LGBM_80_ET_20",
        "one_minus_nmae": blend_total["one_minus_nmae"],
        "ficr": blend_total["ficr"],
        "score": blend_total["score"],
    },
])

comparison["score_change"] = (
    comparison["score"]
    - baseline_total["score"]
)

print(comparison)

print("\nGroup별 지표")
print(blend_metrics)

                version  one_minus_nmae      ficr     score  score_change
0           Group3_LGBM        0.879823  0.383960  0.631891      0.000000
1  Group3_LGBM_80_ET_20        0.880124  0.384095  0.632109      0.000218

Group별 지표
   valid_count      nmae  one_minus_nmae       scr      ficr    group
0         3710  0.112477        0.887523  0.440198  0.440198  Group 1
1         3675  0.111492        0.888508  0.422324  0.422324  Group 2
2         3296  0.135660        0.864340  0.289762  0.289762  Group 3


In [111]:
group3_error_df = group3_model_oof[
    [
        "forecast_kst_dtm",
        "fold",
        "y_true",
        "pred_lgbm",
    ]
].copy()

group3_error_df["pred_error"] = (
    group3_error_df["pred_lgbm"]
    - group3_error_df["y_true"]
)

group3_error_df["abs_error"] = np.abs(
    group3_error_df["pred_error"]
)

group3_error_df["error_ratio_capacity"] = (
    group3_error_df["abs_error"]
    / 21000
)

group3_error_df["signed_error_ratio_capacity"] = (
    group3_error_df["pred_error"]
    / 21000
)

group3_error_df["generation_ratio"] = (
    group3_error_df["y_true"]
    / 21000
)

In [112]:
generation_bins = [
    0.0,
    0.1,
    0.2,
    0.3,
    0.4,
    0.5,
    0.6,
    0.7,
    0.8,
    0.9,
    1.01,
]

generation_labels = [
    "0-10%",
    "10-20%",
    "20-30%",
    "30-40%",
    "40-50%",
    "50-60%",
    "60-70%",
    "70-80%",
    "80-90%",
    "90-100%",
]

group3_error_df["generation_bin"] = pd.cut(
    group3_error_df["generation_ratio"],
    bins=generation_bins,
    labels=generation_labels,
    right=False,
)

In [113]:
group3_bin_rows = []

for generation_bin, bin_df in group3_error_df.groupby(
    "generation_bin",
    observed=True,
):
    valid_df = bin_df[
        bin_df["y_true"] >= 2100
    ].copy()

    if len(valid_df) == 0:
        continue

    error_ratio = (
        np.abs(
            valid_df["pred_lgbm"]
            - valid_df["y_true"]
        )
        / 21000
    )

    rate_4_mask = error_ratio <= 0.06

    rate_3_mask = (
        (error_ratio > 0.06)
        & (error_ratio <= 0.08)
    )

    rate_0_mask = error_ratio > 0.08

    metric = calculate_group_metric(
        y_true=valid_df["y_true"],
        y_pred=valid_df["pred_lgbm"],
        capacity=21000,
    )

    score = (
        0.5 * metric["one_minus_nmae"]
        + 0.5 * metric["scr"]
    )

    group3_bin_rows.append({
        "generation_bin": generation_bin,
        "count": len(valid_df),
        "mean_actual": valid_df["y_true"].mean(),
        "mean_prediction": valid_df["pred_lgbm"].mean(),
        "mean_signed_error": valid_df["pred_error"].mean(),
        "mean_abs_error": valid_df["abs_error"].mean(),
        "nmae": metric["nmae"],
        "one_minus_nmae": metric["one_minus_nmae"],
        "ficr": metric["scr"],
        "score": score,
        "rate_4_ratio": rate_4_mask.mean(),
        "rate_3_ratio": rate_3_mask.mean(),
        "rate_0_ratio": rate_0_mask.mean(),
    })

group3_bin_metrics = pd.DataFrame(
    group3_bin_rows
)

print(group3_bin_metrics)

  generation_bin  count   mean_actual  mean_prediction  mean_signed_error  \
0         10-20%    705   3089.802383      3607.645303         517.842920   
1         20-30%    432   5256.889174      5792.858356         535.969183   
2         30-40%    386   7324.240598      7560.941470         236.700872   
3         40-50%    329   9387.908292      9256.902891        -131.005400   
4         50-60%    331  11611.818405     11070.180502        -541.637903   
5         60-70%    324  13656.524343     12008.123307       -1648.401036   
6         70-80%    360  15783.483508     13141.499071       -2641.984438   
7         80-90%    195  17906.724549     13737.439017       -4169.285531   
8        90-100%    234  20015.341692     14189.570557       -5825.771135   

   mean_abs_error      nmae  one_minus_nmae      ficr     score  rate_4_ratio  \
0     1888.913246  0.089948        0.910052  0.519929  0.714990      0.439716   
1     2504.126374  0.119244        0.880756  0.381751  0.631253    

In [114]:
group3_bias_by_bin = (
    group3_error_df[
        group3_error_df["y_true"] >= 2100
    ]
    .groupby(
        "generation_bin",
        observed=True,
    )
    .agg(
        count=("y_true", "size"),
        mean_actual=("y_true", "mean"),
        mean_prediction=("pred_lgbm", "mean"),
        mean_error=("pred_error", "mean"),
        median_error=("pred_error", "median"),
        mean_abs_error=("abs_error", "mean"),
    )
    .reset_index()
)

group3_bias_by_bin["bias_direction"] = np.where(
    group3_bias_by_bin["mean_error"] > 0,
    "과대예측",
    "과소예측",
)

print(group3_bias_by_bin)

  generation_bin  count   mean_actual  mean_prediction   mean_error  \
0         10-20%    705   3089.802383      3607.645303   517.842920   
1         20-30%    432   5256.889174      5792.858356   535.969183   
2         30-40%    386   7324.240598      7560.941470   236.700872   
3         40-50%    329   9387.908292      9256.902891  -131.005400   
4         50-60%    331  11611.818405     11070.180502  -541.637903   
5         60-70%    324  13656.524343     12008.123307 -1648.401036   
6         70-80%    360  15783.483508     13141.499071 -2641.984438   
7         80-90%    195  17906.724549     13737.439017 -4169.285531   
8        90-100%    234  20015.341692     14189.570557 -5825.771135   

   median_error  mean_abs_error bias_direction  
0    -94.757890     1888.913246           과대예측  
1     36.043596     2504.126374           과대예측  
2     -4.188270     2871.471958           과대예측  
3   -150.228352     2714.330126           과소예측  
4   -140.538592     2697.433389           과소

In [115]:
group3_error_df["hour"] = (
    group3_error_df["forecast_kst_dtm"].dt.hour
)

group3_hour_rows = []

for hour, hour_df in group3_error_df.groupby("hour"):
    valid_df = hour_df[
        hour_df["y_true"] >= 2100
    ]

    if len(valid_df) == 0:
        continue

    metric = calculate_group_metric(
        y_true=valid_df["y_true"],
        y_pred=valid_df["pred_lgbm"],
        capacity=21000,
    )

    score = (
        0.5 * metric["one_minus_nmae"]
        + 0.5 * metric["scr"]
    )

    group3_hour_rows.append({
        "hour": hour,
        "count": len(valid_df),
        "mean_actual": valid_df["y_true"].mean(),
        "mean_error": (
            valid_df["pred_lgbm"]
            - valid_df["y_true"]
        ).mean(),
        "nmae": metric["nmae"],
        "ficr": metric["scr"],
        "score": score,
    })

group3_hour_metrics = (
    pd.DataFrame(group3_hour_rows)
    .sort_values("score")
    .reset_index(drop=True)
)

print(group3_hour_metrics)

    hour  count   mean_actual   mean_error      nmae      ficr     score
0     22    152  10644.789046 -1567.209707  0.159445  0.231736  0.536145
1     20    128  10470.309367 -1115.609948  0.154478  0.242049  0.543785
2      5    161  10211.977683 -1354.990307  0.141743  0.237344  0.547800
3      7    148   9731.750784 -1065.154960  0.150172  0.247773  0.548801
4      6    158   9664.593367  -935.181626  0.140638  0.242633  0.550997
5      8    143   9422.393364 -1010.654915  0.138150  0.264786  0.563318
6     17    114   9407.649851  -414.433905  0.144958  0.274780  0.564911
7     19    123  10183.591764 -1006.064396  0.139530  0.270518  0.565494
8      0    153  10629.097680 -1201.184999  0.146482  0.289056  0.571287
9     21    141  10638.832184 -1424.051724  0.157369  0.300539  0.571585
10     1    157  10118.775357  -760.848062  0.129390  0.274533  0.572572
11    10    123  10022.145317 -1621.279854  0.133398  0.278862  0.572732
12     4    170   9969.454712 -1161.442765  0.13283

In [116]:
group3_error_df["month"] = (
    group3_error_df["forecast_kst_dtm"].dt.month
)

group3_month_rows = []

for month, month_df in group3_error_df.groupby("month"):
    valid_df = month_df[
        month_df["y_true"] >= 2100
    ]

    if len(valid_df) == 0:
        continue

    metric = calculate_group_metric(
        y_true=valid_df["y_true"],
        y_pred=valid_df["pred_lgbm"],
        capacity=21000,
    )

    score = (
        0.5 * metric["one_minus_nmae"]
        + 0.5 * metric["scr"]
    )

    group3_month_rows.append({
        "month": month,
        "count": len(valid_df),
        "mean_actual": valid_df["y_true"].mean(),
        "mean_error": (
            valid_df["pred_lgbm"]
            - valid_df["y_true"]
        ).mean(),
        "nmae": metric["nmae"],
        "ficr": metric["scr"],
        "score": score,
    })

group3_month_metrics = (
    pd.DataFrame(group3_month_rows)
    .sort_values("score")
    .reset_index(drop=True)
)

print(group3_month_metrics)

   month  count   mean_actual   mean_error      nmae      ficr     score
0      7    515  13503.225466 -1279.932148  0.167844  0.192440  0.512298
1     11    293  12213.766618 -1836.105115  0.160821  0.205274  0.522227
2     10    241   7987.624046 -1978.058839  0.152050  0.226786  0.537368
3      4    356   7580.360452 -1746.707797  0.130176  0.260258  0.565041
4      5    485   9728.307676  -787.866839  0.137299  0.274400  0.568550
5      8    199   8260.207472 -1316.311698  0.144006  0.305102  0.580548
6      6    295   7528.735529  -885.638195  0.125665  0.316062  0.595198
7      9    249   5298.801514 -1114.773831  0.111569  0.336563  0.612497
8     12    662  11201.825021   424.857541  0.110861  0.436269  0.662704
9      1      1  16328.905000  1495.817184  0.071229  0.750000  0.839385


In [117]:
def apply_group3_piecewise_bias(
    pred,
    low_bias=0,
    mid_bias=0,
    high_bias=0,
):
    pred = np.asarray(pred).copy()

    corrected = pred.copy()

    low_mask = pred < 7000

    mid_mask = (
        (pred >= 7000)
        & (pred < 14000)
    )

    high_mask = pred >= 14000

    corrected[low_mask] += low_bias
    corrected[mid_mask] += mid_bias
    corrected[high_mask] += high_bias

    return np.clip(
        corrected,
        0,
        21000,
    )

In [118]:
group3_piecewise_rows = []

bias_values = [
    -600,
    -300,
    0,
    300,
    600,
]

raw_group3_score = (
    0.5 * (
        1 - group3_single_results.loc[
            group3_single_results["model"] == "LightGBM",
            "nmae",
        ].iloc[0]
    )
    + 0.5 * group3_single_results.loc[
        group3_single_results["model"] == "LightGBM",
        "ficr",
    ].iloc[0]
)

for low_bias in bias_values:
    for mid_bias in bias_values:
        for high_bias in bias_values:
            pred_corrected = apply_group3_piecewise_bias(
                pred=group3_model_oof["pred_lgbm"],
                low_bias=low_bias,
                mid_bias=mid_bias,
                high_bias=high_bias,
            )

            metric = calculate_group_metric(
                y_true=group3_model_oof["y_true"],
                y_pred=pred_corrected,
                capacity=21000,
            )

            score = (
                0.5 * metric["one_minus_nmae"]
                + 0.5 * metric["scr"]
            )

            group3_piecewise_rows.append({
                "low_bias": low_bias,
                "mid_bias": mid_bias,
                "high_bias": high_bias,
                "nmae": metric["nmae"],
                "one_minus_nmae": metric[
                    "one_minus_nmae"
                ],
                "ficr": metric["scr"],
                "score": score,
                "score_change": (
                    score - raw_group3_score
                ),
            })

group3_piecewise_search = (
    pd.DataFrame(group3_piecewise_rows)
    .sort_values(
        "score",
        ascending=False,
    )
    .reset_index(drop=True)
)

print(group3_piecewise_search.head(15))

    low_bias  mid_bias  high_bias      nmae  one_minus_nmae      ficr  \
0        600       600        600  0.131517        0.868483  0.314788   
1        600       600        300  0.131658        0.868342  0.312040   
2        600       600          0  0.131956        0.868044  0.309177   
3        600       300        600  0.131587        0.868413  0.308564   
4        300       600        600  0.133328        0.866672  0.309115   
5        600       600       -300  0.132410        0.867590  0.307472   
6        600         0        600  0.131983        0.868017  0.306885   
7        600       300        300  0.131728        0.868272  0.305816   
8        300       600        300  0.133469        0.866531  0.306366   
9        600         0        300  0.132124        0.867876  0.304137   
10       600      -300        600  0.132707        0.867293  0.304402   
11       600       300          0  0.132026        0.867974  0.302954   
12       300       600          0  0.133767        

In [119]:
best_piecewise = (
    group3_piecewise_search.iloc[0]
)

group3_piecewise_fold_rows = []

for fold_name, fold_df in group3_model_oof.groupby(
    "fold"
):
    raw_pred = fold_df[
        "pred_lgbm"
    ].to_numpy()

    corrected_pred = apply_group3_piecewise_bias(
        pred=raw_pred,
        low_bias=float(
            best_piecewise["low_bias"]
        ),
        mid_bias=float(
            best_piecewise["mid_bias"]
        ),
        high_bias=float(
            best_piecewise["high_bias"]
        ),
    )

    raw_metric = calculate_group_metric(
        y_true=fold_df["y_true"],
        y_pred=raw_pred,
        capacity=21000,
    )

    corrected_metric = calculate_group_metric(
        y_true=fold_df["y_true"],
        y_pred=corrected_pred,
        capacity=21000,
    )

    raw_score = (
        0.5 * raw_metric["one_minus_nmae"]
        + 0.5 * raw_metric["scr"]
    )

    corrected_score = (
        0.5
        * corrected_metric["one_minus_nmae"]
        + 0.5
        * corrected_metric["scr"]
    )

    group3_piecewise_fold_rows.append({
        "fold": fold_name,
        "raw_nmae": raw_metric["nmae"],
        "corrected_nmae": corrected_metric["nmae"],
        "raw_ficr": raw_metric["scr"],
        "corrected_ficr": corrected_metric["scr"],
        "raw_score": raw_score,
        "corrected_score": corrected_score,
        "score_change": (
            corrected_score - raw_score
        ),
    })

group3_piecewise_fold_results = pd.DataFrame(
    group3_piecewise_fold_rows
)

print("최적 구간 보정")
print(best_piecewise)

print("\nFold별 결과")
print(group3_piecewise_fold_results)


최적 구간 보정
low_bias          600.000000
mid_bias          600.000000
high_bias         600.000000
nmae                0.131517
one_minus_nmae      0.868483
ficr                0.314788
score               0.591636
score_change        0.015238
Name: 0, dtype: float64

Fold별 결과
     fold  raw_nmae  corrected_nmae  raw_ficr  corrected_ficr  raw_score  \
0  fold_1  0.132039        0.126787  0.279614        0.306407   0.573787   
1  fold_2  0.148471        0.140340  0.230552        0.264591   0.541040   
2  fold_3  0.131271        0.128909  0.341674        0.359495   0.605201   

   corrected_score  score_change  
0         0.589810      0.016022  
1         0.562125      0.021085  
2         0.615293      0.010092  


In [120]:
final_oof_pred_1 = np.clip(
    1.046 * np.asarray(oof_pred_1) + 750,
    0,
    21600,
)

final_oof_pred_2 = np.clip(
    0.97 * np.asarray(oof_pred_2) + 1200,
    0,
    21600,
)

final_oof_pred_3 = np.clip(
    np.asarray(oof_pred_3) + 600,
    0,
    21000,
)

final_group_metrics, final_total_score = (
    calculate_competition_score(
        y_true_list=[
            oof_group_1["y_true"],
            oof_group_2["y_true"],
            oof_group_3["y_true"],
        ],
        y_pred_list=[
            final_oof_pred_1,
            final_oof_pred_2,
            final_oof_pred_3,
        ],
        capacities_list=[
            21600,
            21600,
            21000,
        ],
    )
)

print(final_group_metrics)

print("\n최종 후보 OOF")
print("1-NMAE:", final_total_score["one_minus_nmae"])
print("FICR:", final_total_score["ficr"])
print("Score:", final_total_score["score"])

print(
    "\n기존 제출용 OOF 대비:",
    final_total_score["score"] - 0.6315299149752265,
)

   valid_count      nmae  one_minus_nmae       scr      ficr    group
0         3710  0.112477        0.887523  0.440198  0.440198  Group 1
1         3675  0.111492        0.888508  0.422324  0.422324  Group 2
2         3296  0.131517        0.868483  0.314788  0.314788  Group 3

최종 후보 OOF
1-NMAE: 0.8815046518656742
FICR: 0.3924366530381173
Score: 0.6369706524518958

기존 제출용 OOF 대비: 0.005440737476669288


In [121]:
def apply_final_postprocess_v3(
    pred_group_1,
    pred_group_2,
    pred_group_3,
):
    pred_group_1_post = np.clip(
        1.046 * np.asarray(pred_group_1) + 750,
        0,
        21600,
    )

    pred_group_2_post = np.clip(
        0.97 * np.asarray(pred_group_2) + 1200,
        0,
        21600,
    )

    pred_group_3_post = np.clip(
        np.asarray(pred_group_3) + 600,
        0,
        21000,
    )

    return (
        pred_group_1_post,
        pred_group_2_post,
        pred_group_3_post,
    )

In [122]:
group3_global_bias_rows = []

for bias in np.arange(
    400,
    1001,
    50,
):
    pred_corrected = np.clip(
        group3_model_oof["pred_lgbm"].to_numpy() + bias,
        0,
        21000,
    )

    metric = calculate_group_metric(
        y_true=group3_model_oof["y_true"],
        y_pred=pred_corrected,
        capacity=21000,
    )

    score = (
        0.5 * metric["one_minus_nmae"]
        + 0.5 * metric["scr"]
    )

    fold_changes = []

    for fold_name, fold_df in group3_model_oof.groupby("fold"):
        raw_pred = fold_df["pred_lgbm"].to_numpy()

        post_pred = np.clip(
            raw_pred + bias,
            0,
            21000,
        )

        raw_metric = calculate_group_metric(
            fold_df["y_true"],
            raw_pred,
            21000,
        )

        post_metric = calculate_group_metric(
            fold_df["y_true"],
            post_pred,
            21000,
        )

        raw_score = (
            0.5 * raw_metric["one_minus_nmae"]
            + 0.5 * raw_metric["scr"]
        )

        post_score = (
            0.5 * post_metric["one_minus_nmae"]
            + 0.5 * post_metric["scr"]
        )

        fold_changes.append(
            post_score - raw_score
        )

    group3_global_bias_rows.append({
        "bias": bias,
        "nmae": metric["nmae"],
        "one_minus_nmae": metric["one_minus_nmae"],
        "ficr": metric["scr"],
        "score": score,
        "mean_fold_change": np.mean(fold_changes),
        "min_fold_change": np.min(fold_changes),
    })

group3_global_bias_search = (
    pd.DataFrame(group3_global_bias_rows)
    .sort_values(
        [
            "score",
            "min_fold_change",
        ],
        ascending=[
            False,
            False,
        ],
    )
    .reset_index(drop=True)
)

print(group3_global_bias_search)

    bias      nmae  one_minus_nmae      ficr     score  mean_fold_change  \
0   1000  0.130588        0.869412  0.330171  0.599791          0.024393   
1    950  0.130590        0.869410  0.329343  0.599377          0.023809   
2    900  0.130619        0.869381  0.328437  0.598909          0.023209   
3    850  0.130686        0.869314  0.327064  0.598189          0.022538   
4    800  0.130787        0.869213  0.325329  0.597271          0.021534   
5    750  0.130920        0.869080  0.322472  0.595776          0.020029   
6    700  0.131086        0.868914  0.319217  0.594065          0.018301   
7    650  0.131286        0.868714  0.316370  0.592542          0.016757   
8    600  0.131517        0.868483  0.314788  0.591636          0.015733   
9    550  0.131783        0.868217  0.312023  0.590120          0.014048   
10   500  0.132079        0.867921  0.309164  0.588542          0.012505   
11   450  0.132405        0.867595  0.306929  0.587262          0.011178   
12   400  0.

In [123]:
group3_extended_bias_rows = []

for bias in np.arange(
    800,
    1601,
    50,
):
    pred_corrected = np.clip(
        group3_model_oof["pred_lgbm"].to_numpy() + bias,
        0,
        21000,
    )

    metric = calculate_group_metric(
        y_true=group3_model_oof["y_true"],
        y_pred=pred_corrected,
        capacity=21000,
    )

    score = (
        0.5 * metric["one_minus_nmae"]
        + 0.5 * metric["scr"]
    )

    fold_rows = []

    for fold_name, fold_df in group3_model_oof.groupby("fold"):
        raw_pred = fold_df["pred_lgbm"].to_numpy()

        post_pred = np.clip(
            raw_pred + bias,
            0,
            21000,
        )

        raw_metric = calculate_group_metric(
            y_true=fold_df["y_true"],
            y_pred=raw_pred,
            capacity=21000,
        )

        post_metric = calculate_group_metric(
            y_true=fold_df["y_true"],
            y_pred=post_pred,
            capacity=21000,
        )

        raw_score = (
            0.5 * raw_metric["one_minus_nmae"]
            + 0.5 * raw_metric["scr"]
        )

        post_score = (
            0.5 * post_metric["one_minus_nmae"]
            + 0.5 * post_metric["scr"]
        )

        fold_rows.append({
            "fold": fold_name,
            "score_change": post_score - raw_score,
        })

    fold_result = pd.DataFrame(fold_rows)

    group3_extended_bias_rows.append({
        "bias": bias,
        "nmae": metric["nmae"],
        "one_minus_nmae": metric["one_minus_nmae"],
        "ficr": metric["scr"],
        "score": score,
        "mean_fold_change": fold_result["score_change"].mean(),
        "min_fold_change": fold_result["score_change"].min(),
        "fold_1_change": fold_result.loc[
            fold_result["fold"] == "fold_1",
            "score_change",
        ].iloc[0],
        "fold_2_change": fold_result.loc[
            fold_result["fold"] == "fold_2",
            "score_change",
        ].iloc[0],
        "fold_3_change": fold_result.loc[
            fold_result["fold"] == "fold_3",
            "score_change",
        ].iloc[0],
    })

group3_extended_bias_search = (
    pd.DataFrame(group3_extended_bias_rows)
    .sort_values(
        [
            "score",
            "min_fold_change",
        ],
        ascending=[
            False,
            False,
        ],
    )
    .reset_index(drop=True)
)

print(group3_extended_bias_search)

    bias      nmae  one_minus_nmae      ficr     score  mean_fold_change  \
0   1300  0.131169        0.868831  0.334706  0.601769          0.026898   
1   1350  0.131371        0.868629  0.333936  0.601283          0.026339   
2   1450  0.131856        0.868144  0.334081  0.601113          0.026509   
3   1150  0.130747        0.869253  0.332685  0.600969          0.025768   
4   1500  0.132153        0.867847  0.333345  0.600596          0.025834   
5   1200  0.130857        0.869143  0.331936  0.600539          0.025516   
6   1400  0.131600        0.868400  0.332483  0.600441          0.025657   
7   1100  0.130667        0.869333  0.331436  0.600385          0.025124   
8   1600  0.132855        0.867145  0.333545  0.600345          0.025943   
9   1250  0.130994        0.869006  0.331574  0.600290          0.025344   
10  1550  0.132488        0.867512  0.332771  0.600141          0.025561   
11  1000  0.130588        0.869412  0.330171  0.599791          0.024393   
12  1050  0.

In [124]:
def walk_forward_group3_bias(
    oof_df,
    bias_values,
    capacity=21000,
):
    fold_order = [
        "fold_1",
        "fold_2",
        "fold_3",
    ]

    rows = []

    for valid_idx in range(1, len(fold_order)):
        calibration_folds = fold_order[:valid_idx]
        valid_fold = fold_order[valid_idx]

        calibration_df = oof_df[
            oof_df["fold"].isin(calibration_folds)
        ].copy()

        valid_df = oof_df[
            oof_df["fold"] == valid_fold
        ].copy()

        calibration_rows = []

        for bias in bias_values:
            calibration_pred = np.clip(
                calibration_df["pred_lgbm"].to_numpy()
                + bias,
                0,
                capacity,
            )

            metric = calculate_group_metric(
                y_true=calibration_df["y_true"],
                y_pred=calibration_pred,
                capacity=capacity,
            )

            score = (
                0.5 * metric["one_minus_nmae"]
                + 0.5 * metric["scr"]
            )

            calibration_rows.append({
                "bias": bias,
                "score": score,
                "nmae": metric["nmae"],
                "ficr": metric["scr"],
            })

        calibration_results = (
            pd.DataFrame(calibration_rows)
            .sort_values(
                [
                    "score",
                    "nmae",
                ],
                ascending=[
                    False,
                    True,
                ],
            )
            .reset_index(drop=True)
        )

        best_bias = float(
            calibration_results.iloc[0]["bias"]
        )

        raw_pred = valid_df[
            "pred_lgbm"
        ].to_numpy()

        post_pred = np.clip(
            raw_pred + best_bias,
            0,
            capacity,
        )

        raw_metric = calculate_group_metric(
            y_true=valid_df["y_true"],
            y_pred=raw_pred,
            capacity=capacity,
        )

        post_metric = calculate_group_metric(
            y_true=valid_df["y_true"],
            y_pred=post_pred,
            capacity=capacity,
        )

        raw_score = (
            0.5 * raw_metric["one_minus_nmae"]
            + 0.5 * raw_metric["scr"]
        )

        post_score = (
            0.5 * post_metric["one_minus_nmae"]
            + 0.5 * post_metric["scr"]
        )

        rows.append({
            "calibration_folds": "+".join(
                calibration_folds
            ),
            "valid_fold": valid_fold,
            "selected_bias": best_bias,
            "raw_nmae": raw_metric["nmae"],
            "post_nmae": post_metric["nmae"],
            "raw_ficr": raw_metric["scr"],
            "post_ficr": post_metric["scr"],
            "raw_score": raw_score,
            "post_score": post_score,
            "score_change": post_score - raw_score,
        })

    return pd.DataFrame(rows)

In [125]:
group3_bias_values_extended = np.arange(
    600,
    1601,
    50,
)

group3_bias_walk_forward = (
    walk_forward_group3_bias(
        oof_df=group3_model_oof,
        bias_values=group3_bias_values_extended,
        capacity=21000,
    )
)

print(group3_bias_walk_forward)

  calibration_folds valid_fold  selected_bias  raw_nmae  post_nmae  raw_ficr  \
0            fold_1     fold_2         1000.0  0.148471   0.137177  0.230552   
1     fold_1+fold_2     fold_3         1600.0  0.131271   0.134338  0.341674   

   post_ficr  raw_score  post_score  score_change  
0   0.294478   0.541040    0.578650      0.037610  
1   0.351789   0.605201    0.608726      0.003524  


In [126]:
group3_candidate_biases = [
    800,
    900,
    1000,
    1100,
    1200,
]

group3_candidate_rows = []

for bias in group3_candidate_biases:
    fold_scores = []

    for fold_name, fold_df in group3_model_oof.groupby("fold"):
        pred = np.clip(
            fold_df["pred_lgbm"].to_numpy()
            + bias,
            0,
            21000,
        )

        metric = calculate_group_metric(
            y_true=fold_df["y_true"],
            y_pred=pred,
            capacity=21000,
        )

        score = (
            0.5 * metric["one_minus_nmae"]
            + 0.5 * metric["scr"]
        )

        fold_scores.append(score)

    full_pred = np.clip(
        group3_model_oof["pred_lgbm"].to_numpy()
        + bias,
        0,
        21000,
    )

    full_metric = calculate_group_metric(
        y_true=group3_model_oof["y_true"],
        y_pred=full_pred,
        capacity=21000,
    )

    full_score = (
        0.5 * full_metric["one_minus_nmae"]
        + 0.5 * full_metric["scr"]
    )

    group3_candidate_rows.append({
        "bias": bias,
        "full_score": full_score,
        "nmae": full_metric["nmae"],
        "ficr": full_metric["scr"],
        "mean_fold_score": np.mean(fold_scores),
        "std_fold_score": np.std(fold_scores),
        "min_fold_score": np.min(fold_scores),
    })

group3_candidate_comparison = (
    pd.DataFrame(group3_candidate_rows)
    .sort_values(
        [
            "full_score",
            "min_fold_score",
        ],
        ascending=[
            False,
            False,
        ],
    )
    .reset_index(drop=True)
)

print(group3_candidate_comparison)

   bias  full_score      nmae      ficr  mean_fold_score  std_fold_score  \
0  1200    0.600539  0.130857  0.331936         0.598859        0.011724   
1  1100    0.600385  0.130667  0.331436         0.598467        0.014375   
2  1000    0.599791  0.130588  0.330171         0.597736        0.016186   
3   900    0.598909  0.130619  0.328437         0.596552        0.018334   
4   800    0.597271  0.130787  0.325329         0.594877        0.018694   

   min_fold_score  
0        0.587465  
1        0.582472  
2        0.578650  
3        0.575524  
4        0.573386  


In [127]:
final_oof_pred_1 = np.clip(
    1.046 * np.asarray(oof_pred_1) + 750,
    0,
    21600,
)

final_oof_pred_2 = np.clip(
    0.97 * np.asarray(oof_pred_2) + 1200,
    0,
    21600,
)

final_oof_pred_3 = np.clip(
    np.asarray(oof_pred_3) + 1200,
    0,
    21000,
)

final_group_metrics_v4, final_total_score_v4 = (
    calculate_competition_score(
        y_true_list=[
            oof_group_1["y_true"],
            oof_group_2["y_true"],
            oof_group_3["y_true"],
        ],
        y_pred_list=[
            final_oof_pred_1,
            final_oof_pred_2,
            final_oof_pred_3,
        ],
        capacities_list=[
            21600,
            21600,
            21000,
        ],
    )
)

print(final_group_metrics_v4)

print("\n최종 후보 OOF")
print(
    "1-NMAE:",
    final_total_score_v4["one_minus_nmae"],
)
print(
    "FICR:",
    final_total_score_v4["ficr"],
)
print(
    "Score:",
    final_total_score_v4["score"],
)

print(
    "\n기존 제출용 OOF 대비:",
    final_total_score_v4["score"]
    - 0.6315299149752265,
)

print(
    "직전 보수안 대비:",
    final_total_score_v4["score"]
    - 0.6318914251356443,
)

   valid_count      nmae  one_minus_nmae       scr      ficr    group
0         3710  0.112477        0.887523  0.440198  0.440198  Group 1
1         3675  0.111492        0.888508  0.422324  0.422324  Group 2
2         3296  0.130857        0.869143  0.331936  0.331936  Group 3

최종 후보 OOF
1-NMAE: 0.8817244957230943
FICR: 0.39815267419312644
Score: 0.6399385849581104

기존 제출용 OOF 대비: 0.00840866998288392
직전 보수안 대비: 0.008047159822466066


In [128]:
def apply_final_postprocess_v4(
    pred_group_1,
    pred_group_2,
    pred_group_3,
):
    pred_group_1_post = np.clip(
        1.046 * np.asarray(pred_group_1) + 750,
        0,
        21600,
    )

    pred_group_2_post = np.clip(
        0.97 * np.asarray(pred_group_2) + 1200,
        0,
        21600,
    )

    pred_group_3_post = np.clip(
        np.asarray(pred_group_3) + 1200,
        0,
        21000,
    )

    return (
        pred_group_1_post,
        pred_group_2_post,
        pred_group_3_post,
    )

In [132]:
candidate_variables = []

for name, value in globals().items():
    name_lower = name.lower()

    if any(
        keyword in name_lower
        for keyword in [
            "pred",
            "submission",
            "test",
        ]
    ):
        if isinstance(
            value,
            (
                np.ndarray,
                pd.Series,
                pd.DataFrame,
                list,
                dict,
            ),
        ):
            try:
                shape = np.shape(value)
            except Exception:
                shape = "확인 불가"

            candidate_variables.append({
                "variable": name,
                "type": type(value).__name__,
                "shape": shape,
            })

candidate_variables_df = pd.DataFrame(
    candidate_variables
).sort_values("variable")

print(candidate_variables_df.to_string(index=False))

RuntimeError: dictionary changed size during iteration

In [133]:
print(final_test_predictions.keys())

NameError: name 'final_test_predictions' is not defined

In [134]:
candidate_variables = []

for name, value in list(globals().items()):
    name_lower = name.lower()

    if any(
        keyword in name_lower
        for keyword in [
            "pred",
            "submission",
            "test",
        ]
    ):
        if isinstance(
            value,
            (
                np.ndarray,
                pd.Series,
                pd.DataFrame,
                list,
                dict,
            ),
        ):
            try:
                shape = np.shape(value)
            except Exception:
                shape = "확인 불가"

            candidate_variables.append({
                "variable": name,
                "type": type(value).__name__,
                "shape": shape,
            })

candidate_variables_df = (
    pd.DataFrame(candidate_variables)
    .sort_values("variable")
    .reset_index(drop=True)
)

print(candidate_variables_df.to_string(index=False))

                    variable      type        shape
                X_test_final DataFrame  (8760, 305)
              corrected_pred   ndarray      (2208,)
            final_oof_pred_1   ndarray      (6594,)
            final_oof_pred_2   ndarray      (6594,)
            final_oof_pred_3   ndarray      (6594,)
      final_oof_pred_3_blend   ndarray      (6594,)
       final_oof_pred_3_lgbm   ndarray      (6594,)
                 fine_pred_1   ndarray      (6594,)
                 fine_pred_2   ndarray      (6594,)
                 fine_pred_3   ndarray      (6594,)
                   full_pred   ndarray      (6594,)
             gfs_test_hourly DataFrame  (8760, 153)
        gfs_test_hourly_exp1 DataFrame  (8760, 165)
        gfs_test_hourly_exp2 DataFrame  (8760, 177)
        gfs_test_hourly_exp3 DataFrame  (8760, 201)
        gfs_test_hourly_exp4 DataFrame  (8760, 201)
                gfs_test_raw DataFrame  (78840, 49)
            ldaps_test_check DataFrame  (8760, 125)
     ldaps_t

In [135]:
prediction_candidates = []

for name, value in list(globals().items()):
    if isinstance(
        value,
        (
            np.ndarray,
            pd.Series,
            list,
        ),
    ):
        try:
            arr = np.asarray(value)

            if arr.ndim == 1 and len(arr) == 8760:
                prediction_candidates.append({
                    "variable": name,
                    "type": type(value).__name__,
                    "shape": arr.shape,
                    "min": np.nanmin(arr),
                    "max": np.nanmax(arr),
                    "mean": np.nanmean(arr),
                })
        except Exception:
            pass

prediction_candidates_df = (
    pd.DataFrame(prediction_candidates)
    .sort_values("variable")
    .reset_index(drop=True)
)

print(prediction_candidates_df.to_string(index=False))

        variable    type   shape         min          max        mean
 missing_by_time  Series (8760,)    0.000000    76.000000    0.021461
pred_lgbm_test_1 ndarray (8760,) -337.007556 20332.284797 7453.941069
  pred_rf_test_1 ndarray (8760,)   10.646831 20533.974766 7452.344915
     pred_test_1 ndarray (8760,)    0.000000 20860.742850 7476.304467
pred_test_1_post ndarray (8760,)  700.000000 21600.000000 8549.268658
     pred_test_2 ndarray (8760,)    3.043965 20517.301755 7904.157726
pred_test_2_post ndarray (8760,) 1002.983086 21106.955719 8746.074571
     pred_test_3 ndarray (8760,)    0.000000 19364.201264 6354.335184
pred_test_3_post ndarray (8760,)    0.000000 19364.201264 6354.335184
 pred_xgb_test_1 ndarray (8760,) -238.346512 20999.199219 7481.820801


In [136]:
submission_pred_1, submission_pred_2, submission_pred_3 = (
    apply_final_postprocess_v4(
        pred_group_1=pred_test_1,
        pred_group_2=pred_test_2,
        pred_group_3=pred_test_3,
    )
)

print(
    submission_pred_1.min(),
    submission_pred_1.max(),
    submission_pred_1.mean(),
)

print(
    submission_pred_2.min(),
    submission_pred_2.max(),
    submission_pred_2.mean(),
)

print(
    submission_pred_3.min(),
    submission_pred_3.max(),
    submission_pred_3.mean(),
)

750.0 21600.0 8569.456892174827
1202.9526464195237 21101.782701949694 8867.032993927132
1200.0 20564.201264498857 7554.335184473451


In [137]:
submission_v4 = sample_submission.copy()

submission_v4["kpx_group_1"] = submission_pred_1
submission_v4["kpx_group_2"] = submission_pred_2
submission_v4["kpx_group_3"] = submission_pred_3

submission_path_v4 = (
    "submission_exp2_postprocess_v4_g3_bias1200.csv"
)

submission_v4.to_csv(
    submission_path_v4,
    index=False,
)

print("저장 완료:", submission_path_v4)
print(submission_v4.head())
print(submission_v4.shape)
print(submission_v4.isna().sum())

저장 완료: submission_exp2_postprocess_v4_g3_bias1200.csv
     forecast_id    forecast_kst_dtm   kpx_group_1   kpx_group_2   kpx_group_3
0  forecast_0001 2025-01-01 01:00:00  20638.139138  18687.175247  18166.179467
1  forecast_0002 2025-01-01 02:00:00  20425.308494  18555.068377  18144.578948
2  forecast_0003 2025-01-01 03:00:00  20138.447325  18366.964216  18029.110360
3  forecast_0004 2025-01-01 04:00:00  19559.626371  18015.926212  16551.988762
4  forecast_0005 2025-01-01 05:00:00  19008.858760  18125.129048  16017.391597
(8760, 5)
forecast_id         0
forecast_kst_dtm    0
kpx_group_1         0
kpx_group_2         0
kpx_group_3         0
dtype: int64


In [138]:
assert len(submission_v4) == 8760
assert submission_v4.isna().sum().sum() == 0

assert submission_v4[
    "kpx_group_1"
].between(
    0,
    21600,
).all()

assert submission_v4[
    "kpx_group_2"
].between(
    0,
    21600,
).all()

assert submission_v4[
    "kpx_group_3"
].between(
    0,
    21000,
).all()

print("제출 파일 검증 완료")

제출 파일 검증 완료


In [139]:
comparison_test_post = pd.DataFrame({
    "group": [
        "Group 1",
        "Group 2",
        "Group 3",
    ],
    "old_mean": [
        pred_test_1_post.mean(),
        pred_test_2_post.mean(),
        pred_test_3_post.mean(),
    ],
    "new_mean": [
        submission_pred_1.mean(),
        submission_pred_2.mean(),
        submission_pred_3.mean(),
    ],
    "mean_change": [
        (
            submission_pred_1
            - pred_test_1_post
        ).mean(),
        (
            submission_pred_2
            - pred_test_2_post
        ).mean(),
        (
            submission_pred_3
            - pred_test_3_post
        ).mean(),
    ],
})

print(comparison_test_post)

     group     old_mean     new_mean  mean_change
0  Group 1  8549.268658  8569.456892    20.188234
1  Group 2  8746.074571  8867.032994   120.958423
2  Group 3  6354.335184  7554.335184  1200.000000


In [140]:
submission_g3_bias400 = sample_submission.copy()

# Group 1과 Group 2는 기존 최고 제출값을 그대로 유지
submission_g3_bias400["kpx_group_1"] = (
    pred_test_1_post
)

submission_g3_bias400["kpx_group_2"] = (
    pred_test_2_post
)

# Group 3만 변경
submission_g3_bias400["kpx_group_3"] = np.clip(
    pred_test_3 + 400,
    0,
    21000,
)

submission_g3_bias400_path = (
    "submission_ablation_g3_bias400.csv"
)

submission_g3_bias400.to_csv(
    submission_g3_bias400_path,
    index=False,
)

print("저장 완료:", submission_g3_bias400_path)
print(submission_g3_bias400.head())

저장 완료: submission_ablation_g3_bias400.csv
     forecast_id    forecast_kst_dtm   kpx_group_1   kpx_group_2   kpx_group_3
0  forecast_0001 2025-01-01 01:00:00  20664.193207  18667.455405  17366.179467
1  forecast_0002 2025-01-01 02:00:00  20450.548679  18533.986608  17344.578948
2  forecast_0003 2025-01-01 03:00:00  20162.590527  18343.943228  17229.110360
3  forecast_0004 2025-01-01 04:00:00  19581.556108  17989.286276  15751.988762
4  forecast_0005 2025-01-01 05:00:00  19028.682311  18099.614914  15217.391597


In [141]:
ablation_check = pd.DataFrame({
    "group": [
        "Group 1",
        "Group 2",
        "Group 3",
    ],
    "old_mean": [
        pred_test_1_post.mean(),
        pred_test_2_post.mean(),
        pred_test_3_post.mean(),
    ],
    "new_mean": [
        submission_g3_bias400[
            "kpx_group_1"
        ].mean(),
        submission_g3_bias400[
            "kpx_group_2"
        ].mean(),
        submission_g3_bias400[
            "kpx_group_3"
        ].mean(),
    ],
})

ablation_check["mean_change"] = (
    ablation_check["new_mean"]
    - ablation_check["old_mean"]
)

print(ablation_check)

     group     old_mean     new_mean  mean_change
0  Group 1  8549.268658  8549.268658          0.0
1  Group 2  8746.074571  8746.074571          0.0
2  Group 3  6354.335184  6754.335184        400.0


In [142]:
assert len(submission_g3_bias400) == 8760
assert submission_g3_bias400.isna().sum().sum() == 0

assert np.allclose(
    submission_g3_bias400["kpx_group_1"],
    pred_test_1_post,
)

assert np.allclose(
    submission_g3_bias400["kpx_group_2"],
    pred_test_2_post,
)

assert submission_g3_bias400[
    "kpx_group_3"
].between(
    0,
    21000,
).all()

print("Ablation 제출 파일 검증 완료")

Ablation 제출 파일 검증 완료


In [143]:
pred_test_1_final = np.clip(
    1.05 * pred_test_1 + 700,
    0,
    21600,
)

pred_test_2_final = np.clip(
    0.98 * pred_test_2 + 1000,
    0,
    21600,
)

pred_test_3_final = np.clip(
    pred_test_3,
    0,
    21000,
)

In [144]:
def make_ficr_sample_weight(
    y,
    capacity,
    method="linear",
    low_generation_weight=0.2,
):
    y = np.asarray(y, dtype=float)

    generation_ratio = np.clip(
        y / capacity,
        0,
        1,
    )

    if method == "uniform":
        weight = np.ones_like(
            generation_ratio
        )

    elif method == "sqrt":
        weight = np.sqrt(
            generation_ratio
        )

    elif method == "linear":
        weight = generation_ratio

    elif method == "power_1.5":
        weight = generation_ratio ** 1.5

    else:
        raise ValueError(
            f"지원하지 않는 method: {method}"
        )

    # 평가 제외 구간도 완전히 버리지는 않음
    weight = np.where(
        generation_ratio < 0.1,
        low_generation_weight,
        weight,
    )

    # 평균 가중치를 1로 정규화
    weight = weight / weight.mean()

    return weight

In [145]:
def generate_weighted_lgbm_oof(
    group_df,
    feature_cols,
    target_col,
    capacity,
    lgbm_params,
    weight_method,
):
    oof_rows = []

    for fold in cv_folds:
        train_mask = (
            group_df["forecast_kst_dtm"]
            < fold["train_end"]
        )

        valid_mask = (
            (
                group_df["forecast_kst_dtm"]
                >= fold["train_end"]
            )
            & (
                group_df["forecast_kst_dtm"]
                < fold["valid_end"]
            )
        )

        X_train = group_df.loc[
            train_mask,
            feature_cols,
        ]

        y_train = group_df.loc[
            train_mask,
            target_col,
        ]

        X_valid = group_df.loc[
            valid_mask,
            feature_cols,
        ]

        y_valid = group_df.loc[
            valid_mask,
            target_col,
        ]

        sample_weight = make_ficr_sample_weight(
            y=y_train,
            capacity=capacity,
            method=weight_method,
        )

        model = LGBMRegressor(
            **lgbm_params,
        )

        model.fit(
            X_train,
            y_train,
            sample_weight=sample_weight,
        )

        pred = np.clip(
            model.predict(X_valid),
            0,
            capacity,
        )

        metric = calculate_group_metric(
            y_true=y_valid,
            y_pred=pred,
            capacity=capacity,
        )

        score = (
            0.5
            * metric["one_minus_nmae"]
            + 0.5
            * metric["scr"]
        )

        oof_rows.append(
            pd.DataFrame({
                "forecast_kst_dtm": group_df.loc[
                    valid_mask,
                    "forecast_kst_dtm",
                ].to_numpy(),
                "fold": fold["name"],
                "y_true": y_valid.to_numpy(),
                "pred": pred,
                "nmae": metric["nmae"],
                "ficr": metric["scr"],
                "score": score,
            })
        )

        print(
            fold["name"],
            "| method:",
            weight_method,
            "| NMAE:",
            round(metric["nmae"], 6),
            "| FICR:",
            round(metric["scr"], 6),
            "| Score:",
            round(score, 6),
        )

    return (
        pd.concat(
            oof_rows,
            ignore_index=True,
        )
        .sort_values(
            "forecast_kst_dtm"
        )
        .reset_index(drop=True)
    )

In [146]:
group3_lgbm_params = {
    "n_estimators": 1000,
    "learning_rate": 0.03,
    "num_leaves": 31,
    "max_depth": -1,
    "min_child_samples": 20,
    "subsample": 0.9,
    "colsample_bytree": 0.9,
    "reg_alpha": 0.0,
    "reg_lambda": 0.0,
    "random_state": 42,
    "n_jobs": -1,
    "verbosity": -1,
}

In [147]:
group3_weight_methods = [
    "uniform",
    "sqrt",
    "linear",
    "power_1.5",
]

group3_weighted_oof = {}
group3_weighted_results = []

for method in group3_weight_methods:
    print(
        f"\nGroup 3 LightGBM: {method}"
    )

    oof_weighted = generate_weighted_lgbm_oof(
        group_df=train_group_3_exp2,
        feature_cols=feature_cols_exp2,
        target_col="kpx_group_3",
        capacity=21000,
        lgbm_params=group3_lgbm_params,
        weight_method=method,
    )

    metric = calculate_group_metric(
        y_true=oof_weighted["y_true"],
        y_pred=oof_weighted["pred"],
        capacity=21000,
    )

    score = (
        0.5
        * metric["one_minus_nmae"]
        + 0.5
        * metric["scr"]
    )

    group3_weighted_oof[method] = (
        oof_weighted
    )

    group3_weighted_results.append({
        "method": method,
        "nmae": metric["nmae"],
        "one_minus_nmae": metric[
            "one_minus_nmae"
        ],
        "ficr": metric["scr"],
        "score": score,
    })

group3_weighted_results = (
    pd.DataFrame(
        group3_weighted_results
    )
    .sort_values(
        "score",
        ascending=False,
    )
    .reset_index(drop=True)
)

print(group3_weighted_results)


Group 3 LightGBM: uniform
fold_1 | method: uniform | NMAE: 0.132083 | FICR: 0.276742 | Score: 0.572329
fold_2 | method: uniform | NMAE: 0.144829 | FICR: 0.238732 | Score: 0.546951
fold_3 | method: uniform | NMAE: 0.133908 | FICR: 0.367973 | Score: 0.617032

Group 3 LightGBM: sqrt
fold_1 | method: sqrt | NMAE: 0.132283 | FICR: 0.28677 | Score: 0.577243
fold_2 | method: sqrt | NMAE: 0.147154 | FICR: 0.224012 | Score: 0.538429
fold_3 | method: sqrt | NMAE: 0.132878 | FICR: 0.365928 | Score: 0.616525

Group 3 LightGBM: linear
fold_1 | method: linear | NMAE: 0.133929 | FICR: 0.282818 | Score: 0.574445
fold_2 | method: linear | NMAE: 0.145697 | FICR: 0.247346 | Score: 0.550825
fold_3 | method: linear | NMAE: 0.133997 | FICR: 0.360446 | Score: 0.613224

Group 3 LightGBM: power_1.5
fold_1 | method: power_1.5 | NMAE: 0.135854 | FICR: 0.272291 | Score: 0.568219
fold_2 | method: power_1.5 | NMAE: 0.151099 | FICR: 0.24086 | Score: 0.544881
fold_3 | method: power_1.5 | NMAE: 0.136506 | FICR: 0.346

In [148]:
group3_weight_methods = [
    "uniform",
    "sqrt",
    "linear",
    "power_1.5",
]

group3_weighted_oof = {}
group3_weighted_results = []

for method in group3_weight_methods:
    print(
        f"\nGroup 3 LightGBM: {method}"
    )

    oof_weighted = generate_weighted_lgbm_oof(
        group_df=train_group_3_exp2,
        feature_cols=feature_cols_exp2,
        target_col="kpx_group_3",
        capacity=21000,
        lgbm_params=group3_lgbm_params,
        weight_method=method,
    )

    metric = calculate_group_metric(
        y_true=oof_weighted["y_true"],
        y_pred=oof_weighted["pred"],
        capacity=21000,
    )

    score = (
        0.5
        * metric["one_minus_nmae"]
        + 0.5
        * metric["scr"]
    )

    group3_weighted_oof[method] = (
        oof_weighted
    )

    group3_weighted_results.append({
        "method": method,
        "nmae": metric["nmae"],
        "one_minus_nmae": metric[
            "one_minus_nmae"
        ],
        "ficr": metric["scr"],
        "score": score,
    })

group3_weighted_results = (
    pd.DataFrame(
        group3_weighted_results
    )
    .sort_values(
        "score",
        ascending=False,
    )
    .reset_index(drop=True)
)

print(group3_weighted_results)


Group 3 LightGBM: uniform
fold_1 | method: uniform | NMAE: 0.132083 | FICR: 0.276742 | Score: 0.572329
fold_2 | method: uniform | NMAE: 0.144829 | FICR: 0.238732 | Score: 0.546951
fold_3 | method: uniform | NMAE: 0.133908 | FICR: 0.367973 | Score: 0.617032

Group 3 LightGBM: sqrt
fold_1 | method: sqrt | NMAE: 0.132283 | FICR: 0.28677 | Score: 0.577243
fold_2 | method: sqrt | NMAE: 0.147154 | FICR: 0.224012 | Score: 0.538429
fold_3 | method: sqrt | NMAE: 0.132878 | FICR: 0.365928 | Score: 0.616525

Group 3 LightGBM: linear
fold_1 | method: linear | NMAE: 0.133929 | FICR: 0.282818 | Score: 0.574445
fold_2 | method: linear | NMAE: 0.145697 | FICR: 0.247346 | Score: 0.550825
fold_3 | method: linear | NMAE: 0.133997 | FICR: 0.360446 | Score: 0.613224

Group 3 LightGBM: power_1.5
fold_1 | method: power_1.5 | NMAE: 0.135854 | FICR: 0.272291 | Score: 0.568219
fold_2 | method: power_1.5 | NMAE: 0.151099 | FICR: 0.24086 | Score: 0.544881
fold_3 | method: power_1.5 | NMAE: 0.136506 | FICR: 0.346

In [149]:
group3_weighted_fold_rows = []

for method, oof_df in group3_weighted_oof.items():
    for fold_name, fold_df in oof_df.groupby(
        "fold"
    ):
        metric = calculate_group_metric(
            y_true=fold_df["y_true"],
            y_pred=fold_df["pred"],
            capacity=21000,
        )

        score = (
            0.5
            * metric["one_minus_nmae"]
            + 0.5
            * metric["scr"]
        )

        group3_weighted_fold_rows.append({
            "method": method,
            "fold": fold_name,
            "nmae": metric["nmae"],
            "ficr": metric["scr"],
            "score": score,
        })

group3_weighted_fold_results = (
    pd.DataFrame(
        group3_weighted_fold_rows
    )
)

print(
    group3_weighted_fold_results
    .sort_values(
        [
            "method",
            "fold",
        ]
    )
)

       method    fold      nmae      ficr     score
6      linear  fold_1  0.133929  0.282818  0.574445
7      linear  fold_2  0.145697  0.247346  0.550825
8      linear  fold_3  0.133997  0.360446  0.613224
9   power_1.5  fold_1  0.135854  0.272291  0.568219
10  power_1.5  fold_2  0.151099  0.240860  0.544881
11  power_1.5  fold_3  0.136506  0.346567  0.605031
3        sqrt  fold_1  0.132283  0.286770  0.577243
4        sqrt  fold_2  0.147154  0.224012  0.538429
5        sqrt  fold_3  0.132878  0.365928  0.616525
0     uniform  fold_1  0.132083  0.276742  0.572329
1     uniform  fold_2  0.144829  0.238732  0.546951
2     uniform  fold_3  0.133908  0.367973  0.617032


In [150]:
group3_uniform_oof = (
    group3_weighted_oof["uniform"][
        [
            "forecast_kst_dtm",
            "fold",
            "y_true",
            "pred",
        ]
    ]
    .rename(
        columns={
            "pred": "pred_uniform",
        }
    )
)

group3_linear_oof = (
    group3_weighted_oof["linear"][
        [
            "forecast_kst_dtm",
            "fold",
            "pred",
        ]
    ]
    .rename(
        columns={
            "pred": "pred_linear",
        }
    )
)

group3_weight_blend_oof = pd.merge(
    group3_uniform_oof,
    group3_linear_oof,
    on=[
        "forecast_kst_dtm",
        "fold",
    ],
    how="inner",
    validate="one_to_one",
)

print(group3_weight_blend_oof.shape)
print(group3_weight_blend_oof.isna().sum())

(6594, 5)
forecast_kst_dtm    0
fold                0
y_true              0
pred_uniform        0
pred_linear         0
dtype: int64


In [151]:
uniform_fold_scores = {}

for fold_name, fold_df in group3_weight_blend_oof.groupby(
    "fold"
):
    metric = calculate_group_metric(
        y_true=fold_df["y_true"],
        y_pred=fold_df["pred_uniform"],
        capacity=21000,
    )

    uniform_fold_scores[fold_name] = (
        0.5 * metric["one_minus_nmae"]
        + 0.5 * metric["scr"]
    )

In [152]:
group3_linear_blend_rows = []

for weight_linear in np.arange(
    0,
    1.0001,
    0.05,
):
    weight_uniform = 1 - weight_linear

    blend_pred = np.clip(
        weight_uniform
        * group3_weight_blend_oof[
            "pred_uniform"
        ].to_numpy()
        + weight_linear
        * group3_weight_blend_oof[
            "pred_linear"
        ].to_numpy(),
        0,
        21000,
    )

    full_metric = calculate_group_metric(
        y_true=group3_weight_blend_oof["y_true"],
        y_pred=blend_pred,
        capacity=21000,
    )

    full_score = (
        0.5 * full_metric["one_minus_nmae"]
        + 0.5 * full_metric["scr"]
    )

    fold_changes = []
    fold_scores = []
    fold_ficr_changes = []

    for fold_name, fold_df in (
        group3_weight_blend_oof.groupby("fold")
    ):
        fold_blend_pred = np.clip(
            weight_uniform
            * fold_df["pred_uniform"].to_numpy()
            + weight_linear
            * fold_df["pred_linear"].to_numpy(),
            0,
            21000,
        )

        fold_metric = calculate_group_metric(
            y_true=fold_df["y_true"],
            y_pred=fold_blend_pred,
            capacity=21000,
        )

        fold_score = (
            0.5
            * fold_metric["one_minus_nmae"]
            + 0.5
            * fold_metric["scr"]
        )

        uniform_metric = calculate_group_metric(
            y_true=fold_df["y_true"],
            y_pred=fold_df["pred_uniform"],
            capacity=21000,
        )

        fold_scores.append(fold_score)

        fold_changes.append(
            fold_score
            - uniform_fold_scores[fold_name]
        )

        fold_ficr_changes.append(
            fold_metric["scr"]
            - uniform_metric["scr"]
        )

    group3_linear_blend_rows.append({
        "weight_uniform": weight_uniform,
        "weight_linear": weight_linear,
        "nmae": full_metric["nmae"],
        "one_minus_nmae": full_metric[
            "one_minus_nmae"
        ],
        "ficr": full_metric["scr"],
        "score": full_score,
        "mean_fold_score": np.mean(fold_scores),
        "std_fold_score": np.std(fold_scores),
        "mean_score_change": np.mean(
            fold_changes
        ),
        "min_score_change": np.min(
            fold_changes
        ),
        "mean_ficr_change": np.mean(
            fold_ficr_changes
        ),
        "min_ficr_change": np.min(
            fold_ficr_changes
        ),
    })

group3_linear_blend_search = (
    pd.DataFrame(group3_linear_blend_rows)
    .sort_values(
        [
            "score",
            "min_score_change",
        ],
        ascending=[
            False,
            False,
        ],
    )
    .reset_index(drop=True)
)

print(
    group3_linear_blend_search.head(15)
)

    weight_uniform  weight_linear      nmae  one_minus_nmae      ficr  \
0             0.60           0.40  0.135627        0.864373  0.306123   
1             0.55           0.45  0.135630        0.864370  0.304675   
2             0.70           0.30  0.135711        0.864289  0.304676   
3             0.50           0.50  0.135651        0.864349  0.304182   
4             0.65           0.35  0.135655        0.864345  0.304076   
5             0.80           0.20  0.135885        0.864115  0.303788   
6             0.85           0.15  0.135998        0.864002  0.303883   
7             0.20           0.80  0.136430        0.863570  0.304123   
8             0.90           0.10  0.136131        0.863869  0.303778   
9             0.25           0.75  0.136242        0.863758  0.303744   
10            0.45           0.55  0.135707        0.864293  0.303118   
11            0.75           0.25  0.135786        0.864214  0.302662   
12            0.30           0.70  0.136077        

In [153]:
best_linear_blend = (
    group3_linear_blend_search.iloc[0]
)

best_weight_linear = float(
    best_linear_blend["weight_linear"]
)

best_weight_uniform = (
    1 - best_weight_linear
)

group3_linear_blend_fold_rows = []

for fold_name, fold_df in (
    group3_weight_blend_oof.groupby("fold")
):
    pred_uniform = fold_df[
        "pred_uniform"
    ].to_numpy()

    pred_blend = np.clip(
        best_weight_uniform * pred_uniform
        + best_weight_linear
        * fold_df["pred_linear"].to_numpy(),
        0,
        21000,
    )

    uniform_metric = calculate_group_metric(
        y_true=fold_df["y_true"],
        y_pred=pred_uniform,
        capacity=21000,
    )

    blend_metric = calculate_group_metric(
        y_true=fold_df["y_true"],
        y_pred=pred_blend,
        capacity=21000,
    )

    uniform_score = (
        0.5 * uniform_metric["one_minus_nmae"]
        + 0.5 * uniform_metric["scr"]
    )

    blend_score = (
        0.5 * blend_metric["one_minus_nmae"]
        + 0.5 * blend_metric["scr"]
    )

    group3_linear_blend_fold_rows.append({
        "fold": fold_name,
        "uniform_nmae": uniform_metric["nmae"],
        "blend_nmae": blend_metric["nmae"],
        "uniform_ficr": uniform_metric["scr"],
        "blend_ficr": blend_metric["scr"],
        "uniform_score": uniform_score,
        "blend_score": blend_score,
        "score_change": (
            blend_score - uniform_score
        ),
        "ficr_change": (
            blend_metric["scr"]
            - uniform_metric["scr"]
        ),
    })

group3_linear_blend_fold_results = (
    pd.DataFrame(
        group3_linear_blend_fold_rows
    )
)

print("최적 가중치")
print(best_linear_blend)

print("\nFold별 결과")
print(group3_linear_blend_fold_results)

최적 가중치
weight_uniform       0.600000
weight_linear        0.400000
nmae                 0.135627
one_minus_nmae       0.864373
ficr                 0.306123
score                0.585248
mean_fold_score      0.581374
std_fold_score       0.030298
mean_score_change    0.002603
min_score_change     0.000804
mean_ficr_change     0.004377
min_ficr_change      0.001025
Name: 0, dtype: float64

Fold별 결과
     fold  uniform_nmae  blend_nmae  uniform_ficr  blend_ficr  uniform_score  \
0  fold_1      0.132083    0.131089      0.276742    0.281444       0.572329   
1  fold_2      0.144829    0.144246      0.238732    0.239757       0.546951   
2  fold_3      0.133908    0.133001      0.367973    0.375378       0.617032   

   blend_score  score_change  ficr_change  
0     0.575177      0.002848     0.004702  
1     0.547755      0.000804     0.001025  
2     0.621189      0.004157     0.007406  


In [154]:
group3_linear_blend_robust = (
    group3_linear_blend_search[
        group3_linear_blend_search[
            "min_score_change"
        ] >= -0.001
    ]
    .sort_values(
        [
            "score",
            "mean_ficr_change",
        ],
        ascending=[
            False,
            False,
        ],
    )
    .reset_index(drop=True)
)

print(group3_linear_blend_robust.head(10))

   weight_uniform  weight_linear      nmae  one_minus_nmae      ficr  \
0            0.60           0.40  0.135627        0.864373  0.306123   
1            0.55           0.45  0.135630        0.864370  0.304675   
2            0.70           0.30  0.135711        0.864289  0.304676   
3            0.50           0.50  0.135651        0.864349  0.304182   
4            0.65           0.35  0.135655        0.864345  0.304076   
5            0.80           0.20  0.135885        0.864115  0.303788   
6            0.85           0.15  0.135998        0.864002  0.303883   
7            0.20           0.80  0.136430        0.863570  0.304123   
8            0.90           0.10  0.136131        0.863869  0.303778   
9            0.25           0.75  0.136242        0.863758  0.303744   

      score  mean_fold_score  std_fold_score  mean_score_change  \
0  0.585248         0.581374        0.030298           0.002603   
1  0.584522         0.580727        0.029788           0.001956   
2  0.5

In [155]:
def train_group3_weighted_full_models(
    train_df,
    test_df,
    feature_cols,
    target_col,
    capacity,
    lgbm_params,
):
    X_train = train_df[feature_cols]
    y_train = train_df[target_col]
    X_test = test_df[feature_cols]

    uniform_model = LGBMRegressor(
        **lgbm_params,
    )

    uniform_model.fit(
        X_train,
        y_train,
    )

    linear_weight = make_ficr_sample_weight(
        y=y_train,
        capacity=capacity,
        method="linear",
    )

    linear_model = LGBMRegressor(
        **lgbm_params,
    )

    linear_model.fit(
        X_train,
        y_train,
        sample_weight=linear_weight,
    )

    pred_uniform = np.clip(
        uniform_model.predict(X_test),
        0,
        capacity,
    )

    pred_linear = np.clip(
        linear_model.predict(X_test),
        0,
        capacity,
    )

    pred_blend = np.clip(
        0.60 * pred_uniform
        + 0.40 * pred_linear,
        0,
        capacity,
    )

    return {
        "uniform_model": uniform_model,
        "linear_model": linear_model,
        "pred_uniform": pred_uniform,
        "pred_linear": pred_linear,
        "pred_blend": pred_blend,
    }


group3_weighted_full = (
    train_group3_weighted_full_models(
        train_df=train_group_3_exp2,
        test_df=test_group_3_exp2,
        feature_cols=feature_cols_exp2,
        target_col="kpx_group_3",
        capacity=21000,
        lgbm_params=group3_lgbm_params,
    )
)

pred_test_3_uniform_weighted = (
    group3_weighted_full["pred_uniform"]
)

pred_test_3_linear_weighted = (
    group3_weighted_full["pred_linear"]
)

pred_test_3_ficr_blend = (
    group3_weighted_full["pred_blend"]
)

print("Uniform 예측:", pred_test_3_uniform_weighted.shape)
print("Linear 예측:", pred_test_3_linear_weighted.shape)
print("Blend 예측:", pred_test_3_ficr_blend.shape)

NameError: name 'test_group_3_exp2' is not defined

In [156]:
test_dataframe_candidates = []

for name, value in list(globals().items()):
    if isinstance(value, pd.DataFrame):
        if len(value) == 8760:
            feature_match_count = sum(
                col in value.columns
                for col in feature_cols_exp2
            )

            test_dataframe_candidates.append({
                "variable": name,
                "shape": value.shape,
                "feature_match_count": feature_match_count,
                "has_target_1": "kpx_group_1" in value.columns,
                "has_target_2": "kpx_group_2" in value.columns,
                "has_target_3": "kpx_group_3" in value.columns,
            })

test_dataframe_candidates_df = (
    pd.DataFrame(test_dataframe_candidates)
    .sort_values(
        "feature_match_count",
        ascending=False,
    )
    .reset_index(drop=True)
)

print(test_dataframe_candidates_df.to_string(index=False))

              variable       shape  feature_match_count  has_target_1  has_target_2  has_target_3
             test_exp2 (8760, 307)                  305         False         False         False
             test_exp3 (8760, 339)                  305         False         False         False
             test_exp4 (8760, 334)                  305         False         False         False
          X_test_final (8760, 305)                  305         False         False         False
             test_exp1 (8760, 295)                  293         False         False         False
     weather_test_exp2 (8760, 293)                  292         False         False         False
     weather_test_exp4 (8760, 320)                  292         False         False         False
     weather_test_exp3 (8760, 325)                  292         False         False         False
                  test (8760, 283)                  281         False         False         False
              old_te

In [157]:
test_group_3_exp2 = test_exp2

print("Train:", train_group_3_exp2.shape)
print("Test:", test_group_3_exp2.shape)
print(
    "누락 피처:",
    len([
        col
        for col in feature_cols_exp2
        if col not in test_group_3_exp2.columns
    ]),
)

Train: (17538, 309)
Test: (8760, 307)
누락 피처: 0


In [158]:
group3_weighted_full = train_group3_weighted_full_models(
    train_df=train_group_3_exp2,
    test_df=test_group_3_exp2,
    feature_cols=feature_cols_exp2,
    target_col="kpx_group_3",
    capacity=21000,
    lgbm_params=group3_lgbm_params,
)

pred_test_3_uniform_weighted = (
    group3_weighted_full["pred_uniform"]
)

pred_test_3_linear_weighted = (
    group3_weighted_full["pred_linear"]
)

pred_test_3_ficr_blend = (
    group3_weighted_full["pred_blend"]
)

print(
    "Uniform:",
    pred_test_3_uniform_weighted.shape,
)

print(
    "Linear:",
    pred_test_3_linear_weighted.shape,
)

print(
    "Blend:",
    pred_test_3_ficr_blend.shape,
)

Uniform: (8760,)
Linear: (8760,)
Blend: (8760,)


In [159]:
uniform_difference = (
    pred_test_3_uniform_weighted
    - np.asarray(pred_test_3)
)

print(
    "평균 절대 차이:",
    np.mean(np.abs(uniform_difference)),
)

print(
    "최대 절대 차이:",
    np.max(np.abs(uniform_difference)),
)

print(
    "상관계수:",
    np.corrcoef(
        pred_test_3_uniform_weighted,
        pred_test_3,
    )[0, 1],
)

print(
    "기존 평균:",
    np.mean(pred_test_3),
)

print(
    "새 Uniform 평균:",
    np.mean(pred_test_3_uniform_weighted),
)

평균 절대 차이: 426.8798584712361
최대 절대 차이: 2644.4173869103142
상관계수: 0.994681750033439
기존 평균: 6354.335184473451
새 Uniform 평균: 6322.745887334688


In [160]:
submission_g3_ficr_blend = sample_submission.copy()

submission_g3_ficr_blend[
    "kpx_group_1"
] = pred_test_1_post

submission_g3_ficr_blend[
    "kpx_group_2"
] = pred_test_2_post

submission_g3_ficr_blend[
    "kpx_group_3"
] = pred_test_3_ficr_blend

submission_g3_ficr_blend_path = (
    "submission_ablation_g3_uniform60_linear40.csv"
)

submission_g3_ficr_blend.to_csv(
    submission_g3_ficr_blend_path,
    index=False,
)

print(
    "저장 완료:",
    submission_g3_ficr_blend_path,
)

저장 완료: submission_ablation_g3_uniform60_linear40.csv


In [161]:
assert len(submission_g3_ficr_blend) == 8760
assert submission_g3_ficr_blend.isna().sum().sum() == 0

assert np.allclose(
    submission_g3_ficr_blend["kpx_group_1"],
    pred_test_1_post,
)

assert np.allclose(
    submission_g3_ficr_blend["kpx_group_2"],
    pred_test_2_post,
)

assert submission_g3_ficr_blend[
    "kpx_group_3"
].between(
    0,
    21000,
).all()

print("제출 파일 검증 완료")

제출 파일 검증 완료


In [162]:
uniform_diff = (
    pred_test_3_uniform_weighted
    - np.asarray(pred_test_3)
)

uniform_check = {
    "mae_difference": np.mean(
        np.abs(uniform_diff)
    ),
    "rmse_difference": np.sqrt(
        np.mean(uniform_diff ** 2)
    ),
    "max_difference": np.max(
        np.abs(uniform_diff)
    ),
    "correlation": np.corrcoef(
        pred_test_3_uniform_weighted,
        pred_test_3,
    )[0, 1],
    "old_mean": np.mean(pred_test_3),
    "new_uniform_mean": np.mean(
        pred_test_3_uniform_weighted
    ),
}

print(uniform_check)

{'mae_difference': np.float64(426.8798584712361), 'rmse_difference': np.float64(575.8528280218345), 'max_difference': np.float64(2644.4173869103142), 'correlation': np.float64(0.994681750033439), 'old_mean': np.float64(6354.335184473451), 'new_uniform_mean': np.float64(6322.745887334688)}


In [163]:
lgbm_model_candidates = []

for name, value in list(globals().items()):
    if isinstance(value, LGBMRegressor):
        params = value.get_params()

        lgbm_model_candidates.append({
            "variable": name,
            "n_estimators": params.get(
                "n_estimators"
            ),
            "learning_rate": params.get(
                "learning_rate"
            ),
            "num_leaves": params.get(
                "num_leaves"
            ),
            "max_depth": params.get(
                "max_depth"
            ),
            "min_child_samples": params.get(
                "min_child_samples"
            ),
            "subsample": params.get(
                "subsample"
            ),
            "colsample_bytree": params.get(
                "colsample_bytree"
            ),
            "reg_alpha": params.get(
                "reg_alpha"
            ),
            "reg_lambda": params.get(
                "reg_lambda"
            ),
        })

lgbm_model_candidates_df = pd.DataFrame(
    lgbm_model_candidates
)

print(
    lgbm_model_candidates_df.to_string(
        index=False
    )
)

    variable  n_estimators  learning_rate  num_leaves  max_depth  min_child_samples  subsample  colsample_bytree  reg_alpha  reg_lambda
  lgbm_new_1          3000           0.03          31         -1                 20        0.8               0.8        0.1         1.0
  lgbm_new_2          3000           0.03          31         -1                 20        0.8               0.8        0.1         1.0
  lgbm_new_3          3000           0.03          31         -1                 20        0.8               0.8        0.1         1.0
 lgbm_exp1_1          3000           0.03          31         -1                 20        0.8               0.8        0.1         1.0
 lgbm_exp1_2          3000           0.03          31         -1                 20        0.8               0.8        0.1         1.0
 lgbm_exp1_3          3000           0.03          31         -1                 20        0.8               0.8        0.1         1.0
 lgbm_exp2_1          3000           0.03       

In [164]:
parameter_candidates = []

for name, value in list(globals().items()):
    if isinstance(value, dict):
        keys = set(value.keys())

        lgbm_keys = {
            "n_estimators",
            "learning_rate",
            "num_leaves",
            "max_depth",
        }

        if len(keys & lgbm_keys) >= 2:
            parameter_candidates.append({
                "variable": name,
                "keys": sorted(keys),
                "value": value,
            })

for candidate in parameter_candidates:
    print("\n", candidate["variable"])
    print(candidate["value"])


 rf_param_space
{'n_estimators': [250, 350, 450, 600], 'max_depth': [None, 12, 18, 24, 32], 'min_samples_split': [2, 4, 6, 10], 'min_samples_leaf': [1, 2, 3, 5], 'max_features': [0.35, 0.5, 0.7, 1.0], 'bootstrap': [True, False]}

 baseline_rf_params
{'n_estimators': 400, 'max_depth': None, 'min_samples_split': 4, 'min_samples_leaf': 2, 'max_features': 0.7, 'bootstrap': True, 'max_samples': None}

 params
{'boosting_type': 'gbdt', 'class_weight': None, 'colsample_bytree': 0.8, 'importance_type': 'split', 'learning_rate': 0.03, 'max_depth': -1, 'min_child_samples': 20, 'min_child_weight': 0.001, 'min_split_gain': 0.0, 'n_estimators': 201, 'n_jobs': -1, 'num_leaves': 31, 'objective': 'regression', 'random_state': 42, 'reg_alpha': 0.1, 'reg_lambda': 1.0, 'subsample': 0.8, 'subsample_for_bin': 200000, 'subsample_freq': 0, 'verbosity': -1}

 group3_lgbm_params
{'n_estimators': 1000, 'learning_rate': 0.03, 'num_leaves': 31, 'max_depth': -1, 'min_child_samples': 20, 'subsample': 0.9, 'colsamp

In [165]:
group3_lgbm_params_exact = {
    "boosting_type": "gbdt",
    "objective": "regression",
    "n_estimators": 201,
    "learning_rate": 0.03,
    "num_leaves": 31,
    "max_depth": -1,
    "min_child_samples": 20,
    "min_child_weight": 0.001,
    "min_split_gain": 0.0,
    "subsample": 0.8,
    "subsample_freq": 0,
    "colsample_bytree": 0.8,
    "reg_alpha": 0.1,
    "reg_lambda": 1.0,
    "random_state": 42,
    "n_jobs": -1,
    "verbosity": -1,
}

In [166]:
group3_weight_methods_exact = [
    "uniform",
    "linear",
]

group3_weighted_oof_exact = {}
group3_weighted_results_exact = []

for method in group3_weight_methods_exact:
    print(f"\nGroup 3 exact LightGBM: {method}")

    oof_exact = generate_weighted_lgbm_oof(
        group_df=train_group_3_exp2,
        feature_cols=feature_cols_exp2,
        target_col="kpx_group_3",
        capacity=21000,
        lgbm_params=group3_lgbm_params_exact,
        weight_method=method,
    )

    metric = calculate_group_metric(
        y_true=oof_exact["y_true"],
        y_pred=oof_exact["pred"],
        capacity=21000,
    )

    score = (
        0.5 * metric["one_minus_nmae"]
        + 0.5 * metric["scr"]
    )

    group3_weighted_oof_exact[method] = oof_exact

    group3_weighted_results_exact.append({
        "method": method,
        "nmae": metric["nmae"],
        "one_minus_nmae": metric["one_minus_nmae"],
        "ficr": metric["scr"],
        "score": score,
    })

group3_weighted_results_exact = (
    pd.DataFrame(group3_weighted_results_exact)
    .sort_values("score", ascending=False)
    .reset_index(drop=True)
)

print(group3_weighted_results_exact)


Group 3 exact LightGBM: uniform
fold_1 | method: uniform | NMAE: 0.129805 | FICR: 0.282059 | Score: 0.576127
fold_2 | method: uniform | NMAE: 0.141843 | FICR: 0.24145 | Score: 0.549804
fold_3 | method: uniform | NMAE: 0.129454 | FICR: 0.37796 | Score: 0.624253

Group 3 exact LightGBM: linear
fold_1 | method: linear | NMAE: 0.131852 | FICR: 0.304778 | Score: 0.586463
fold_2 | method: linear | NMAE: 0.141588 | FICR: 0.259968 | Score: 0.55919
fold_3 | method: linear | NMAE: 0.133548 | FICR: 0.355344 | Score: 0.610898
    method      nmae  one_minus_nmae      ficr     score
0   linear  0.135312        0.864688  0.311243  0.587965
1  uniform  0.133195        0.866805  0.307850  0.587327


In [167]:
group3_weighted_fold_exact_rows = []

for method, oof_df in group3_weighted_oof_exact.items():
    for fold_name, fold_df in oof_df.groupby("fold"):
        metric = calculate_group_metric(
            y_true=fold_df["y_true"],
            y_pred=fold_df["pred"],
            capacity=21000,
        )

        score = (
            0.5 * metric["one_minus_nmae"]
            + 0.5 * metric["scr"]
        )

        group3_weighted_fold_exact_rows.append({
            "method": method,
            "fold": fold_name,
            "nmae": metric["nmae"],
            "ficr": metric["scr"],
            "score": score,
        })

group3_weighted_fold_exact = pd.DataFrame(
    group3_weighted_fold_exact_rows
)

print(
    group3_weighted_fold_exact
    .sort_values(["method", "fold"])
)

    method    fold      nmae      ficr     score
3   linear  fold_1  0.131852  0.304778  0.586463
4   linear  fold_2  0.141588  0.259968  0.559190
5   linear  fold_3  0.133548  0.355344  0.610898
0  uniform  fold_1  0.129805  0.282059  0.576127
1  uniform  fold_2  0.141843  0.241450  0.549804
2  uniform  fold_3  0.129454  0.377960  0.624253


In [168]:
uniform_exact_oof = (
    group3_weighted_oof_exact["uniform"][
        [
            "forecast_kst_dtm",
            "fold",
            "y_true",
            "pred",
        ]
    ]
    .rename(columns={"pred": "pred_uniform"})
)

linear_exact_oof = (
    group3_weighted_oof_exact["linear"][
        [
            "forecast_kst_dtm",
            "fold",
            "pred",
        ]
    ]
    .rename(columns={"pred": "pred_linear"})
)

group3_exact_blend_oof = pd.merge(
    uniform_exact_oof,
    linear_exact_oof,
    on=[
        "forecast_kst_dtm",
        "fold",
    ],
    how="inner",
    validate="one_to_one",
)

exact_blend_rows = []

for weight_linear in np.arange(
    0,
    0.5001,
    0.05,
):
    weight_uniform = 1 - weight_linear

    full_pred = np.clip(
        weight_uniform
        * group3_exact_blend_oof["pred_uniform"].to_numpy()
        + weight_linear
        * group3_exact_blend_oof["pred_linear"].to_numpy(),
        0,
        21000,
    )

    full_metric = calculate_group_metric(
        y_true=group3_exact_blend_oof["y_true"],
        y_pred=full_pred,
        capacity=21000,
    )

    full_score = (
        0.5 * full_metric["one_minus_nmae"]
        + 0.5 * full_metric["scr"]
    )

    fold_score_changes = []
    fold_ficr_changes = []

    for fold_name, fold_df in group3_exact_blend_oof.groupby(
        "fold"
    ):
        uniform_pred = fold_df[
            "pred_uniform"
        ].to_numpy()

        blend_pred = np.clip(
            weight_uniform * uniform_pred
            + weight_linear
            * fold_df["pred_linear"].to_numpy(),
            0,
            21000,
        )

        uniform_metric = calculate_group_metric(
            y_true=fold_df["y_true"],
            y_pred=uniform_pred,
            capacity=21000,
        )

        blend_metric = calculate_group_metric(
            y_true=fold_df["y_true"],
            y_pred=blend_pred,
            capacity=21000,
        )

        uniform_score = (
            0.5 * uniform_metric["one_minus_nmae"]
            + 0.5 * uniform_metric["scr"]
        )

        blend_score = (
            0.5 * blend_metric["one_minus_nmae"]
            + 0.5 * blend_metric["scr"]
        )

        fold_score_changes.append(
            blend_score - uniform_score
        )

        fold_ficr_changes.append(
            blend_metric["scr"]
            - uniform_metric["scr"]
        )

    exact_blend_rows.append({
        "weight_uniform": weight_uniform,
        "weight_linear": weight_linear,
        "nmae": full_metric["nmae"],
        "ficr": full_metric["scr"],
        "score": full_score,
        "mean_score_change": np.mean(
            fold_score_changes
        ),
        "min_score_change": np.min(
            fold_score_changes
        ),
        "mean_ficr_change": np.mean(
            fold_ficr_changes
        ),
        "min_ficr_change": np.min(
            fold_ficr_changes
        ),
    })

group3_exact_blend_search = (
    pd.DataFrame(exact_blend_rows)
    .sort_values(
        [
            "score",
            "min_score_change",
        ],
        ascending=[
            False,
            False,
        ],
    )
    .reset_index(drop=True)
)

print(group3_exact_blend_search)

    weight_uniform  weight_linear      nmae      ficr     score  \
0             0.50           0.50  0.132819  0.315849  0.591515   
1             0.55           0.45  0.132719  0.315338  0.591310   
2             0.60           0.40  0.132646  0.314927  0.591141   
3             0.70           0.30  0.132592  0.313433  0.590420   
4             0.65           0.35  0.132605  0.313436  0.590415   
5             0.75           0.25  0.132614  0.311328  0.589357   
6             0.80           0.20  0.132665  0.311236  0.589285   
7             0.85           0.15  0.132746  0.310248  0.588751   
8             0.90           0.10  0.132863  0.308648  0.587892   
9             1.00           0.00  0.133195  0.307850  0.587327   
10            0.95           0.05  0.133016  0.306802  0.586893   

    mean_score_change  min_score_change  mean_ficr_change  min_ficr_change  
0            0.004687         -0.000666          0.008913        -0.000456  
1            0.004634         -0.002186  

In [169]:
group3_weighted_full_exact = (
    train_group3_weighted_full_models(
        train_df=train_group_3_exp2,
        test_df=test_exp2,
        feature_cols=feature_cols_exp2,
        target_col="kpx_group_3",
        capacity=21000,
        lgbm_params=group3_lgbm_params_exact,
    )
)

pred_test_3_uniform_exact = (
    group3_weighted_full_exact["pred_uniform"]
)

pred_test_3_linear_exact = (
    group3_weighted_full_exact["pred_linear"]
)

In [170]:
exact_uniform_difference = (
    pred_test_3_uniform_exact
    - np.asarray(pred_test_3)
)

print(
    "평균 절대 차이:",
    np.mean(
        np.abs(exact_uniform_difference)
    ),
)

print(
    "최대 절대 차이:",
    np.max(
        np.abs(exact_uniform_difference)
    ),
)

print(
    "상관계수:",
    np.corrcoef(
        pred_test_3_uniform_exact,
        pred_test_3,
    )[0, 1],
)

print(
    "기존 평균:",
    np.mean(pred_test_3),
)

print(
    "재학습 Uniform 평균:",
    np.mean(pred_test_3_uniform_exact),
)

평균 절대 차이: 0.0
최대 절대 차이: 0.0
상관계수: 0.9999999999999999
기존 평균: 6354.335184473451
재학습 Uniform 평균: 6354.335184473451


In [171]:
print(
    group3_exact_blend_search.head(10)
)

   weight_uniform  weight_linear      nmae      ficr     score  \
0            0.50           0.50  0.132819  0.315849  0.591515   
1            0.55           0.45  0.132719  0.315338  0.591310   
2            0.60           0.40  0.132646  0.314927  0.591141   
3            0.70           0.30  0.132592  0.313433  0.590420   
4            0.65           0.35  0.132605  0.313436  0.590415   
5            0.75           0.25  0.132614  0.311328  0.589357   
6            0.80           0.20  0.132665  0.311236  0.589285   
7            0.85           0.15  0.132746  0.310248  0.588751   
8            0.90           0.10  0.132863  0.308648  0.587892   
9            1.00           0.00  0.133195  0.307850  0.587327   

   mean_score_change  min_score_change  mean_ficr_change  min_ficr_change  
0           0.004687         -0.000666          0.008913        -0.000456  
1           0.004634         -0.002186          0.008715        -0.003693  
2           0.004364         -0.001464       

In [172]:
group3_exact_blend_robust = (
    group3_exact_blend_search[
        (
            group3_exact_blend_search[
                "min_score_change"
            ] >= 0
        )
        & (
            group3_exact_blend_search[
                "min_ficr_change"
            ] >= 0
        )
    ]
    .sort_values(
        [
            "score",
            "mean_ficr_change",
        ],
        ascending=[
            False,
            False,
        ],
    )
    .reset_index(drop=True)
)

print(group3_exact_blend_robust)

   weight_uniform  weight_linear      nmae     ficr     score  \
0             1.0            0.0  0.133195  0.30785  0.587327   

   mean_score_change  min_score_change  mean_ficr_change  min_ficr_change  
0                0.0               0.0               0.0              0.0  


In [173]:
best_exact_blend = (
    group3_exact_blend_robust.iloc[0]
)

selected_weight_uniform = float(
    best_exact_blend["weight_uniform"]
)

selected_weight_linear = float(
    best_exact_blend["weight_linear"]
)

print(
    "Uniform 가중치:",
    selected_weight_uniform,
)

print(
    "Linear 가중치:",
    selected_weight_linear,
)

print(best_exact_blend)

Uniform 가중치: 1.0
Linear 가중치: 0.0
weight_uniform       1.000000
weight_linear        0.000000
nmae                 0.133195
ficr                 0.307850
score                0.587327
mean_score_change    0.000000
min_score_change     0.000000
mean_ficr_change     0.000000
min_ficr_change      0.000000
Name: 0, dtype: float64


In [174]:
pred_test_3_exact_blend = np.clip(
    selected_weight_uniform
    * np.asarray(pred_test_3)
    + selected_weight_linear
    * np.asarray(pred_test_3_linear_exact),
    0,
    21000,
)

In [175]:
group3_exact_blend_difference = (
    pred_test_3_exact_blend
    - np.asarray(pred_test_3)
)

print(
    "평균 절대 변화:",
    np.mean(
        np.abs(
            group3_exact_blend_difference
        )
    ),
)

print(
    "평균 변화:",
    np.mean(
        group3_exact_blend_difference
    ),
)

print(
    "최대 절대 변화:",
    np.max(
        np.abs(
            group3_exact_blend_difference
        )
    ),
)

print(
    "상관계수:",
    np.corrcoef(
        pred_test_3,
        pred_test_3_exact_blend,
    )[0, 1],
)

평균 절대 변화: 0.0
평균 변화: 0.0
최대 절대 변화: 0.0
상관계수: 0.9999999999999999


In [176]:
submission_g3_exact_weighted = (
    sample_submission.copy()
)

submission_g3_exact_weighted[
    "kpx_group_1"
] = pred_test_1_post

submission_g3_exact_weighted[
    "kpx_group_2"
] = pred_test_2_post

submission_g3_exact_weighted[
    "kpx_group_3"
] = pred_test_3_exact_blend

submission_g3_exact_weighted_path = (
    "submission_ablation_g3_exact_weighted_blend.csv"
)

submission_g3_exact_weighted.to_csv(
    submission_g3_exact_weighted_path,
    index=False,
)

print(
    "저장 완료:",
    submission_g3_exact_weighted_path,
)

저장 완료: submission_ablation_g3_exact_weighted_blend.csv


In [177]:
assert len(
    submission_g3_exact_weighted
) == 8760

assert (
    submission_g3_exact_weighted
    .isna()
    .sum()
    .sum()
    == 0
)

assert np.allclose(
    submission_g3_exact_weighted[
        "kpx_group_1"
    ],
    pred_test_1_post,
)

assert np.allclose(
    submission_g3_exact_weighted[
        "kpx_group_2"
    ],
    pred_test_2_post,
)

assert submission_g3_exact_weighted[
    "kpx_group_3"
].between(
    0,
    21000,
).all()

print("제출 파일 검증 완료")

제출 파일 검증 완료
